In [ ]:
# ============================================================
# PHASE 0 — ENVIRONMENT SETUP
# ECG Attention CNN-Transformer Research
# ============================================================

!pip install -q wfdb

print("Environment setup complete.")

In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import wfdb

print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("SciPy  :", scipy.__version__)
print("Sklearn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("WFDB   :", wfdb.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n✅ Environment is ready.")

In [8]:
# ============================================================
# STEP 2 — FIX ENVIRONMENT VERSION
# ============================================================

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--force-reinstall",
    "pandas==2.2.2"
])

print("✅ Pandas 2.2.2 installed.")
print("⚠️ Restart the Colab runtime after this cell.")

KeyboardInterrupt: 

In [ ]:
# ============================================================
# STEP 2B — VERIFY CLEAN ENVIRONMENT
# ============================================================

import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import wfdb

print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("SciPy  :", scipy.__version__)
print("Sklearn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("WFDB   :", wfdb.__version__)

print("\nCUDA available:", torch.cuda.is_available())

print("\n✅ CLEAN ENVIRONMENT VERIFIED")

In [ ]:
# ============================================================
# STEP 3 — PTB-XL DIRECTORY + METADATA
# ============================================================

import os
import urllib.request

PTBXL_ROOT = "/content/ptb-xl"
PTBXL_URL = "https://physionet.org/files/ptb-xl/1.0.3/"

os.makedirs(PTBXL_ROOT, exist_ok=True)

files = [
    "ptbxl_database.csv",
    "scp_statements.csv"
]

for filename in files:

    url = PTBXL_URL + filename
    destination = os.path.join(PTBXL_ROOT, filename)

    if os.path.exists(destination):
        print(f"Already exists: {filename}")
    else:
        print(f"Downloading: {filename}")
        urllib.request.urlretrieve(url, destination)
        print(f"Downloaded: {filename}")

print("\n✅ PTB-XL metadata download completed.")

print("\nFiles in PTB-XL directory:")
print(os.listdir(PTBXL_ROOT))

In [ ]:
# ============================================================
# STEP 4 — LOAD PTB-XL METADATA
# ============================================================

import os
import pandas as pd

PTBXL_ROOT = "/content/ptb-xl"

METADATA_PATH = os.path.join(
    PTBXL_ROOT,
    "ptbxl_database.csv"
)

SCP_PATH = os.path.join(
    PTBXL_ROOT,
    "scp_statements.csv"
)

# Load main metadata
df = pd.read_csv(
    METADATA_PATH,
    index_col="ecg_id"
)

# Load SCP diagnostic statements
scp_df = pd.read_csv(
    SCP_PATH,
    index_col=0
)

print("=" * 60)
print("PTB-XL METADATA LOADED")
print("=" * 60)

print("\nECG records       :", len(df))
print("Metadata columns  :", len(df.columns))
print("SCP statements    :", len(scp_df))

print("\nMetadata columns:")
print(df.columns.tolist())

print("\nFirst 5 ECG records:")
display(df.head())

print("\nSCP statement columns:")
print(scp_df.columns.tolist())

print("\nFirst 10 SCP statements:")
display(scp_df.head(10))

In [ ]:
# ============================================================
# STEP 5 — PTB-XL DIAGNOSTIC LABELS
# ============================================================

import ast
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Verify required columns
# ------------------------------------------------------------

required_columns = ["scp_codes"]

for col in required_columns:
    if col not in df.columns:
        raise KeyError(
            f"Required column '{col}' is missing from PTB-XL metadata."
        )

print("Required metadata columns verified.")

# ------------------------------------------------------------
# Parse SCP codes
# ------------------------------------------------------------

def parse_scp_codes(value):
    """
    Convert PTB-XL scp_codes representation into a Python dictionary.
    """
    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return {}


df["scp_codes_parsed"] = df["scp_codes"].apply(parse_scp_codes)

# ------------------------------------------------------------
# Verify SCP mapping
# ------------------------------------------------------------

if "diagnostic_class" not in scp_df.columns:
    raise KeyError(
        "diagnostic_class column not found in scp_statements.csv"
    )

print("SCP diagnostic mapping verified.")

# ------------------------------------------------------------
# Five diagnostic superclasses
# ------------------------------------------------------------

DIAGNOSTIC_CLASSES = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP"
]

print("\nTarget classes:")
for i, label in enumerate(DIAGNOSTIC_CLASSES):
    print(f"{i}: {label}")

# ------------------------------------------------------------
# Map SCP codes -> diagnostic superclasses
# ------------------------------------------------------------

def get_diagnostic_classes(scp_codes):
    """
    Return all diagnostic superclasses associated
    with the SCP codes of an ECG.
    """

    classes = set()

    for code in scp_codes.keys():

        if code not in scp_df.index:
            continue

        diagnostic_class = scp_df.loc[
            code, "diagnostic_class"
        ]

        if pd.isna(diagnostic_class):
            continue

        # Some entries may contain multiple classes
        if isinstance(diagnostic_class, str):
            for label in diagnostic_class.split(","):
                label = label.strip()

                if label in DIAGNOSTIC_CLASSES:
                    classes.add(label)

    return sorted(classes)


df["diagnostic_classes"] = df[
    "scp_codes_parsed"
].apply(get_diagnostic_classes)

# ------------------------------------------------------------
# Create multi-label binary targets
# ------------------------------------------------------------

for label in DIAGNOSTIC_CLASSES:

    df[label] = df["diagnostic_classes"].apply(
        lambda classes: int(label in classes)
    )

# ------------------------------------------------------------
# Inspect results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DIAGNOSTIC LABEL SUMMARY")
print("=" * 60)

for label in DIAGNOSTIC_CLASSES:
    count = int(df[label].sum())
    percentage = 100 * count / len(df)

    print(
        f"{label:5s} : {count:6d} ECGs "
        f"({percentage:.2f}%)"
    )

# ------------------------------------------------------------
# Multi-label statistics
# ------------------------------------------------------------

label_counts_per_ecg = df[
    DIAGNOSTIC_CLASSES
].sum(axis=1)

print("\n" + "=" * 60)
print("MULTI-LABEL STATISTICS")
print("=" * 60)

print(
    "ECGs with no target class :",
    int((label_counts_per_ecg == 0).sum())
)

print(
    "ECGs with exactly 1 class :",
    int((label_counts_per_ecg == 1).sum())
)

print(
    "ECGs with 2+ classes      :",
    int((label_counts_per_ecg >= 2).sum())
)

print(
    "Maximum classes / ECG     :",
    int(label_counts_per_ecg.max())
)

print("\nExample labels:")
display(
    df[
        ["scp_codes", "diagnostic_classes"]
        + DIAGNOSTIC_CLASSES
    ].head(10)
)

print("\n✅ Multi-label diagnostic targets created successfully.")

In [ ]:
# ============================================================
# STEP 3 — DOWNLOAD PTB-XL 100 Hz ECG WAVEFORMS
# ============================================================

import os
import subprocess

PTBXL_PATH = "/content/ptbxl"

os.makedirs(PTBXL_PATH, exist_ok=True)

print("Downloading PTB-XL 100 Hz waveforms...")
print("This may take some time. DO NOT INTERRUPT the cell.")

url = "https://physionet.org/files/ptb-xl/1.0.3/records100/"

command = [
    "wget",
    "-r",
    "-N",
    "-c",
    "-np",
    "-nH",
    "--cut-dirs=3",
    "--accept=*.hea,*.dat",
    "-P", PTBXL_PATH,
    url
]

result = subprocess.run(command)

if result.returncode != 0:
    raise RuntimeError("Download did not complete successfully.")

print("\n✅ PTB-XL 100 Hz waveform download completed.")

In [ ]:
# ============================================================
# STEP 4: VERIFY PTB-XL WAVEFORM DOWNLOAD
# ============================================================

import os

PTBXL_PATH = "/content/ptbxl"

print("Checking PTB-XL waveform files...\n")

dat_files = []
hea_files = []

for root, dirs, files in os.walk(PTBXL_PATH):
    for file in files:
        if file.endswith(".dat"):
            dat_files.append(os.path.join(root, file))
        elif file.endswith(".hea"):
            hea_files.append(os.path.join(root, file))

print(f"ECG waveform (.dat) files found : {len(dat_files)}")
print(f"Header (.hea) files found        : {len(hea_files)}")

print("\nFirst 10 waveform files:\n")

for path in dat_files[:10]:
    print(path)

if len(dat_files) > 0 and len(hea_files) > 0:
    print("\n✅ PTB-XL waveform data is ready!")
else:
    print("\n❌ Waveform files were not found.")

In [ ]:
# ============================================================
# TEST PTB-XL 100 Hz DOWNLOAD
# ============================================================

import os
import urllib.request

TEST_DIR = "/content/ptbxl_test"
os.makedirs(TEST_DIR, exist_ok=True)

# One known PTB-XL 100 Hz record
header_url = (
    "https://physionet.org/files/ptb-xl/1.0.3/"
    "records100/00000/00001_lr.hea"
)

signal_url = (
    "https://physionet.org/files/ptb-xl/1.0.3/"
    "records100/00000/00001_lr.dat"
)

header_path = os.path.join(TEST_DIR, "00001_lr.hea")
signal_path = os.path.join(TEST_DIR, "00001_lr.dat")

print("Downloading test header...")

urllib.request.urlretrieve(
    header_url,
    header_path
)

print("Header downloaded.")

print("Downloading test ECG signal...")

urllib.request.urlretrieve(
    signal_url,
    signal_path
)

print("Signal downloaded.")

print("\nFiles:")
print(os.listdir(TEST_DIR))

print("\nHeader size:", os.path.getsize(header_path), "bytes")
print("Signal size:", os.path.getsize(signal_path), "bytes")

print("\n✅ TEST DOWNLOAD SUCCESSFUL")

In [ ]:
# ============================================================
# STEP 6 — VERIFY ONE REAL PTB-XL ECG
# ============================================================

import os
import wfdb
import numpy as np

TEST_DIR = "/content/ptbxl_test"

record_path = os.path.join(
    TEST_DIR,
    "00001_lr"
)

signal, fields = wfdb.rdsamp(record_path)

print("=" * 60)
print("PTB-XL ECG VERIFICATION")
print("=" * 60)

print("Signal shape       :", signal.shape)
print("Sampling frequency :", fields["fs"], "Hz")
print("Number of leads    :", fields["n_sig"])
print("Signal length      :", signal.shape[0], "samples")
print("Lead names         :", fields["sig_name"])

print("\nData type          :", signal.dtype)
print("Minimum value     :", signal.min())
print("Maximum value     :", signal.max())
print("NaN values        :", np.isnan(signal).sum())
print("Inf values        :", np.isinf(signal).sum())

In [ ]:
# ============================================================
# STEP 7 — VISUALIZE 12-LEAD ECG
# ============================================================

import matplotlib.pyplot as plt

time = np.arange(signal.shape[0]) / fields["fs"]

fig, axes = plt.subplots(
    12,
    1,
    figsize=(14, 18),
    sharex=True
)

for i, ax in enumerate(axes):

    ax.plot(time, signal[:, i], linewidth=0.8)

    ax.set_ylabel(
        fields["sig_name"][i],
        rotation=0,
        labelpad=25
    )

    ax.grid(alpha=0.2)

axes[-1].set_xlabel("Time (seconds)")

fig.suptitle(
    "PTB-XL 100 Hz — Raw 12-Lead ECG",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
import os

# Actual PTB-XL location
PTBXL_PATH = "/content/physionet.org/files/ptb-xl/1.0.3"

METADATA_PATH = os.path.join(
    PTBXL_PATH,
    "ptbxl_database.csv"
)

print("PTB-XL directory:")
print(PTBXL_PATH)

print("\nChecking files...")

print("Directory exists:", os.path.exists(PTBXL_PATH))
print("Metadata exists:", os.path.exists(METADATA_PATH))

if os.path.exists(PTBXL_PATH):
    print("\nFiles currently available:")
    print(os.listdir(PTBXL_PATH)[:20])

In [ ]:
import os

print("=== /content ===")
for item in os.listdir("/content"):
    print(item)

print("\n=== Searching for PTB-XL metadata ===")

matches = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file in ["ptbxl_database.csv", "scp_statements.csv"]:
            matches.append(os.path.join(root, file))

if matches:
    print("\n✅ Found:")
    for path in matches:
        print(path)
else:
    print("\n❌ ptbxl_database.csv was not found anywhere under /content")

In [ ]:
import os
import pandas as pd

# ============================================================
# PTB-XL PATH CONFIGURATION
# ============================================================

PTBXL_PATH = "/content/ptb-xl"

METADATA_PATH = os.path.join(
    PTBXL_PATH,
    "ptbxl_database.csv"
)

SCP_PATH = os.path.join(
    PTBXL_PATH,
    "scp_statements.csv"
)

print("PTB-XL path:", PTBXL_PATH)
print("Metadata:", METADATA_PATH)
print("SCP statements:", SCP_PATH)

assert os.path.exists(PTBXL_PATH), "❌ PTB-XL directory missing"
assert os.path.exists(METADATA_PATH), "❌ ptbxl_database.csv missing"
assert os.path.exists(SCP_PATH), "❌ scp_statements.csv missing"

print("\n✅ PTB-XL metadata files verified")

In [ ]:
# ============================================================
# LOAD PTB-XL METADATA
# ============================================================

df = pd.read_csv(
    METADATA_PATH,
    index_col="ecg_id"
)

scp_df = pd.read_csv(SCP_PATH)

print("✅ Metadata loaded successfully")
print("Number of ECG records:", len(df))
print("Metadata shape:", df.shape)
print("SCP statements shape:", scp_df.shape)

print("\nFirst 5 records:")
display(df.head())

In [ ]:
# ============================================================
# CHECK WAVEFORM DATA
# ============================================================

print("Contents of PTB-XL directory:\n")

for item in os.listdir(PTBXL_PATH):
    full_path = os.path.join(PTBXL_PATH, item)

    if os.path.isdir(full_path):
        print("📁", item)
    else:
        print("📄", item)

In [ ]:
# ============================================================
# CHECK 100 Hz AND 500 Hz DATA
# ============================================================

for folder in ["records100", "records500"]:

    folder_path = os.path.join(PTBXL_PATH, folder)

    print(f"\n{folder}: ", end="")

    if not os.path.exists(folder_path):
        print("❌ NOT FOUND")
        continue

    count = 0

    for root, dirs, files in os.walk(folder_path):
        count += len(files)

    print(f"✅ Found — {count} files")

In [ ]:
# ============================================================
# STEP 4A — SELECT A DEVELOPMENT SUBSET
# ============================================================

import os
import pandas as pd

PTBXL_PATH = "/content/ptb-xl"
RECORDS100_PATH = os.path.join(PTBXL_PATH, "records100")

os.makedirs(RECORDS100_PATH, exist_ok=True)

# Make sure metadata is loaded
METADATA_PATH = os.path.join(
    PTBXL_PATH,
    "ptbxl_database.csv"
)

df = pd.read_csv(
    METADATA_PATH,
    index_col="ecg_id"
)

# Select first 500 records for development
dev_df = df.head(500).copy()

print("Total PTB-XL records:", len(df))
print("Development records :", len(dev_df))

print("\nExample waveform paths:")
display(
    dev_df[
        ["filename_lr", "patient_id", "scp_codes"]
    ].head()
)

In [ ]:
# ============================================================
# STEP 4B — DOWNLOAD 500 DEVELOPMENT ECGs
# ============================================================

import urllib.request
import os
from tqdm.auto import tqdm

BASE_URL = "https://physionet.org/files/ptb-xl/1.0.3/"

success = 0
failed = []

for _, row in tqdm(
    dev_df.iterrows(),
    total=len(dev_df),
    desc="Downloading ECGs"
):

    relative_path = row["filename_lr"]

    for extension in [".hea", ".dat"]:

        url = BASE_URL + relative_path + extension

        local_path = os.path.join(
            PTBXL_PATH,
            relative_path + extension
        )

        os.makedirs(
            os.path.dirname(local_path),
            exist_ok=True
        )

        # Don't download if already present
        if os.path.exists(local_path):
            continue

        try:
            urllib.request.urlretrieve(
                url,
                local_path
            )

        except Exception as e:
            failed.append(
                (relative_path, str(e))
            )

    if (
        os.path.exists(
            os.path.join(
                PTBXL_PATH,
                relative_path + ".hea"
            )
        )
        and
        os.path.exists(
            os.path.join(
                PTBXL_PATH,
                relative_path + ".dat"
            )
        )
    ):
        success += 1

print("\n" + "=" * 60)
print("DOWNLOAD SUMMARY")
print("=" * 60)

print("Successfully available:", success)
print("Failed:", len(failed))

if failed:
    print("\nFirst failures:")
    for item in failed[:10]:
        print(item)


In [ ]:
# ============================================================
# STEP 5 — VERIFY 500 PTB-XL ECG RECORDS
# ============================================================

import os
import wfdb
import numpy as np
from tqdm.auto import tqdm

valid_records = []
failed_records = []

for _, row in tqdm(
    dev_df.iterrows(),
    total=len(dev_df),
    desc="Validating ECGs"
):

    record_name = row["filename_lr"]

    try:
        record_path = os.path.join(
            PTBXL_PATH,
            record_name
        )

        signal, fields = wfdb.rdsamp(record_path)

        # -----------------------------
        # Basic structural validation
        # -----------------------------

        if signal.shape != (1000, 12):
            raise ValueError(
                f"Unexpected shape: {signal.shape}"
            )

        if fields["fs"] != 100:
            raise ValueError(
                f"Unexpected sampling rate: {fields['fs']}"
            )

        if np.isnan(signal).any():
            raise ValueError("NaN values detected")

        if np.isinf(signal).any():
            raise ValueError("Inf values detected")

        valid_records.append(record_name)

    except Exception as e:

        failed_records.append(
            {
                "record": record_name,
                "error": str(e)
            }
        )

print("\n" + "=" * 60)
print("ECG VALIDATION RESULTS")
print("=" * 60)

print("Total records :", len(dev_df))
print("Valid records :", len(valid_records))
print("Failed records:", len(failed_records))

if failed_records:
    print("\nFirst failures:")
    for item in failed_records[:10]:
        print(item)

else:
    print("\n✅ ALL 500 ECG RECORDS PASSED VALIDATION")

In [ ]:
# ============================================================
# STEP 6 — BUILD PTB-XL DIAGNOSTIC SUPERCLASS LABELS
# ============================================================

import ast
import pandas as pd
import numpy as np

SCP_PATH = os.path.join(
    PTBXL_PATH,
    "scp_statements.csv"
)

scp_df = pd.read_csv(
    SCP_PATH,
    index_col=0
)

print("SCP statements loaded:", scp_df.shape)

# ------------------------------------------------------------
# Parse scp_codes
# ------------------------------------------------------------

def parse_scp_codes(value):
    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    try:
        return ast.literal_eval(value)
    except Exception:
        return {}

df["scp_codes_parsed"] = df["scp_codes"].apply(
    parse_scp_codes
)

# ------------------------------------------------------------
# Five diagnostic superclasses
# ------------------------------------------------------------

TARGET_CLASSES = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP"
]

# ------------------------------------------------------------
# Map SCP codes → diagnostic superclass
# ------------------------------------------------------------

def get_diagnostic_classes(scp_codes):

    classes = set()

    for code in scp_codes.keys():

        if code not in scp_df.index:
            continue

        diagnostic_class = scp_df.loc[
            code,
            "diagnostic_class"
        ]

        if pd.isna(diagnostic_class):
            continue

        # PTB-XL may contain multiple superclass labels
        for label in str(diagnostic_class).split(","):

            label = label.strip()

            if label in TARGET_CLASSES:
                classes.add(label)

    return sorted(classes)


df["diagnostic_classes"] = df[
    "scp_codes_parsed"
].apply(get_diagnostic_classes)

# ------------------------------------------------------------
# Create binary multi-label targets
# ------------------------------------------------------------

for label in TARGET_CLASSES:

    df[label] = df["diagnostic_classes"].apply(
        lambda x: int(label in x)
    )

# ------------------------------------------------------------
# Development subset
# ------------------------------------------------------------

dev_df = df.loc[dev_df.index].copy()

# ------------------------------------------------------------
# Display class distribution
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DIAGNOSTIC CLASS DISTRIBUTION — 500 ECG DEVELOPMENT SET")
print("=" * 60)

for label in TARGET_CLASSES:

    count = int(dev_df[label].sum())

    percentage = (
        count / len(dev_df) * 100
    )

    print(
        f"{label:5s} : "
        f"{count:4d} "
        f"({percentage:.2f}%)"
    )

# ------------------------------------------------------------
# Multi-label statistics
# ------------------------------------------------------------

label_count = dev_df[
    TARGET_CLASSES
].sum(axis=1)

print("\n" + "=" * 60)
print("MULTI-LABEL STATISTICS")
print("=" * 60)

print(
    "No diagnostic superclass :",
    int((label_count == 0).sum())
)

print(
    "Exactly one class        :",
    int((label_count == 1).sum())
)

print(
    "Two or more classes      :",
    int((label_count >= 2).sum())
)

print(
    "Maximum classes / ECG    :",
    int(label_count.max())
)

print("\nExample:")
display(
    dev_df[
        [
            "patient_id",
            "scp_codes",
            "diagnostic_classes"
        ] + TARGET_CLASSES
    ].head(10)
)

print("\n✅ Diagnostic labels created.")

In [ ]:
# ============================================================
# STEP 7 — PATIENT-LEVEL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------
# Work only with our 500-record development dataset
# ------------------------------------------------------------

split_df = dev_df.copy()

print("Total ECG records:", len(split_df))
print("Unique patients:", split_df["patient_id"].nunique())

# ------------------------------------------------------------
# First split:
# 80% development
# 20% test
# ------------------------------------------------------------

gss_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_val_idx, test_idx = next(
    gss_test.split(
        split_df,
        groups=split_df["patient_id"]
    )
)

train_val_df = split_df.iloc[train_val_idx].copy()
test_df = split_df.iloc[test_idx].copy()

# ------------------------------------------------------------
# Second split:
# 80% train
# 20% validation
#
# This produces approximately:
# 64% train
# 16% validation
# 20% test
# ------------------------------------------------------------

gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    gss_val.split(
        train_val_df,
        groups=train_val_df["patient_id"]
    )
)

train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

# ------------------------------------------------------------
# Verify sizes
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SPLIT SUMMARY")
print("=" * 60)

print(
    f"Train      : {len(train_df)} ECGs | "
    f"{train_df['patient_id'].nunique()} patients"
)

print(
    f"Validation : {len(val_df)} ECGs | "
    f"{val_df['patient_id'].nunique()} patients"
)

print(
    f"Test       : {len(test_df)} ECGs | "
    f"{test_df['patient_id'].nunique()} patients"
)

# ------------------------------------------------------------
# Check patient leakage
# ------------------------------------------------------------

train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])
test_patients = set(test_df["patient_id"])

train_val_overlap = train_patients & val_patients
train_test_overlap = train_patients & test_patients
val_test_overlap = val_patients & test_patients

print("\n" + "=" * 60)
print("PATIENT LEAKAGE CHECK")
print("=" * 60)

print("Train ∩ Validation:", len(train_val_overlap))
print("Train ∩ Test      :", len(train_test_overlap))
print("Validation ∩ Test :", len(val_test_overlap))

if (
    len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
):
    print("\n✅ ZERO PATIENT LEAKAGE")
else:
    raise RuntimeError(
        "❌ Patient leakage detected!"
    )

In [ ]:
# ============================================================
# STEP 8 — CLASS DISTRIBUTION ACROSS SPLITS
# ============================================================

TARGET_CLASSES = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP"
]

print("=" * 75)
print("CLASS DISTRIBUTION")
print("=" * 75)

distribution = pd.DataFrame(
    index=TARGET_CLASSES,
    columns=["Train", "Validation", "Test"]
)

for label in TARGET_CLASSES:
    distribution.loc[label, "Train"] = int(train_df[label].sum())
    distribution.loc[label, "Validation"] = int(val_df[label].sum())
    distribution.loc[label, "Test"] = int(test_df[label].sum())

display(distribution)

print("\nPercentages:")
percentage_distribution = distribution.astype(float).copy()

percentage_distribution["Train"] = (
    percentage_distribution["Train"] / len(train_df) * 100
)

percentage_distribution["Validation"] = (
    percentage_distribution["Validation"] / len(val_df) * 100
)

percentage_distribution["Test"] = (
    percentage_distribution["Test"] / len(test_df) * 100
)

display(
    percentage_distribution.round(2)
)

In [ ]:
# ============================================================
# STEP 9 — VERIFY SPLIT ↔ WAVEFORM MAPPING
# ============================================================

def check_waveform_files(dataframe, split_name):

    missing = []

    for _, row in dataframe.iterrows():

        record = row["filename_lr"]

        hea = os.path.join(
            PTBXL_PATH,
            record + ".hea"
        )

        dat = os.path.join(
            PTBXL_PATH,
            record + ".dat"
        )

        if not os.path.exists(hea) or not os.path.exists(dat):
            missing.append(record)

    print(
        f"{split_name:12s}: "
        f"{len(dataframe)} records | "
        f"{len(missing)} missing"
    )

    return missing


print("=" * 60)
print("WAVEFORM FILE VERIFICATION")
print("=" * 60)

train_missing = check_waveform_files(
    train_df,
    "Train"
)

val_missing = check_waveform_files(
    val_df,
    "Validation"
)

test_missing = check_waveform_files(
    test_df,
    "Test"
)

total_missing = (
    len(train_missing)
    + len(val_missing)
    + len(test_missing)
)

print("\nTotal missing:", total_missing)

if total_missing == 0:
    print("✅ ALL SPLIT RECORDS HAVE WAVEFORM FILES")
else:
    print("⚠️ Some waveform files are missing.")

In [ ]:
# ============================================================
# STEP 10 — ECG PREPROCESSING PIPELINE
# ============================================================

import numpy as np
from scipy.signal import butter, filtfilt


def preprocess_ecg(
    signal,
    fs=100,
    lowcut=0.5,
    highcut=40.0,
    order=4
):
    """
    Preprocess one PTB-XL ECG.

    Input:
        signal : numpy array [1000, 12]

    Output:
        processed : numpy array [12, 1000]
    """

    signal = np.asarray(signal, dtype=np.float32)

    # --------------------------------------------------------
    # 1. Validate raw signal
    # --------------------------------------------------------

    if signal.shape != (1000, 12):
        raise ValueError(
            f"Expected (1000, 12), got {signal.shape}"
        )

    if not np.isfinite(signal).all():
        raise ValueError(
            "Input contains NaN or Inf values"
        )

    # --------------------------------------------------------
    # 2. Design Butterworth bandpass filter
    # --------------------------------------------------------

    nyquist = fs / 2.0

    low = lowcut / nyquist
    high = highcut / nyquist

    if not (0 < low < high < 1):
        raise ValueError(
            "Invalid filter frequencies"
        )

    b, a = butter(
        order,
        [low, high],
        btype="band"
    )

    # --------------------------------------------------------
    # 3. Filter each lead
    # --------------------------------------------------------

    filtered = np.zeros_like(signal)

    for lead in range(signal.shape[1]):

        filtered[:, lead] = filtfilt(
            b,
            a,
            signal[:, lead]
        )

    # --------------------------------------------------------
    # 4. Per-lead Z-score normalization
    # --------------------------------------------------------

    mean = filtered.mean(axis=0)
    std = filtered.std(axis=0)

    # Prevent division by zero
    std = np.where(std < 1e-8, 1.0, std)

    normalized = (
        filtered - mean
    ) / std

    # --------------------------------------------------------
    # 5. Convert [1000, 12] → [12, 1000]
    # --------------------------------------------------------

    processed = normalized.T

    # --------------------------------------------------------
    # 6. Final validation
    # --------------------------------------------------------

    if processed.shape != (12, 1000):
        raise ValueError(
            f"Unexpected output shape: {processed.shape}"
        )

    if not np.isfinite(processed).all():
        raise ValueError(
            "Preprocessing produced NaN or Inf"
        )

    return processed.astype(np.float32)


print("✅ Preprocessing function created.")

In [ ]:
# ============================================================
# STEP 11 — TEST PREPROCESSING ON REAL ECG
# ============================================================

import wfdb

# Select first training ECG
test_record = train_df.iloc[0]["filename_lr"]

record_path = os.path.join(
    PTBXL_PATH,
    test_record
)

# Load raw ECG
raw_signal, fields = wfdb.rdsamp(
    record_path
)

# Preprocess
processed_signal = preprocess_ecg(
    raw_signal,
    fs=fields["fs"]
)

print("=" * 60)
print("PREPROCESSING TEST")
print("=" * 60)

print("Record:", test_record)

print(
    "Raw shape:",
    raw_signal.shape
)

print(
    "Processed shape:",
    processed_signal.shape
)

print(
    "Processed dtype:",
    processed_signal.dtype
)

print(
    "Contains NaN:",
    np.isnan(processed_signal).any()
)

print(
    "Contains Inf:",
    np.isinf(processed_signal).any()
)

print(
    "Global mean:",
    processed_signal.mean()
)

print(
    "Global std:",
    processed_signal.std()
)

assert processed_signal.shape == (12, 1000)
assert np.isfinite(processed_signal).all()

print("\n✅ PREPROCESSING PASSED")

In [ ]:
# ============================================================
# STEP 12 — RAW VS PREPROCESSED ECG
# ============================================================

import matplotlib.pyplot as plt

lead = 1

time = np.arange(1000) / 100

fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 7),
    sharex=True
)

axes[0].plot(
    time,
    raw_signal[:, lead],
    linewidth=0.8
)

axes[0].set_title(
    f"Raw ECG — {fields['sig_name'][lead]}"
)

axes[0].set_ylabel("Amplitude")
axes[0].grid(alpha=0.2)


axes[1].plot(
    time,
    processed_signal[lead],
    linewidth=0.8
)

axes[1].set_title(
    f"Preprocessed ECG — {fields['sig_name'][lead]}"
)

axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Normalized")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 13A — VERIFY PYTORCH DATASET IMPORT
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import wfdb

from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)
print("Dataset:", Dataset)
print("DataLoader:", DataLoader)

print("✅ PyTorch Dataset/DataLoader imports are ready.")

In [ ]:
# Remove samples that have no diagnostic labels

def has_valid_labels(labels):
    return isinstance(labels, list) and len(labels) > 0

before = len(train_df)

train_df = train_df[
    train_df["diagnostic_classes"].apply(has_valid_labels)
].reset_index(drop=True)

after = len(train_df)

print("Training data cleaned")
print("Before:", before)
print("Removed:", before - after)
print("After:", after)

In [ ]:
# Recreate datasets after cleaning

train_dataset = ECGDataset(train_df)
val_dataset = ECGDataset(val_df)
test_dataset = ECGDataset(test_df)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
# ============================================================
# STEP 13 — ECG DATASET
# ============================================================

class ECGDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        signal, fields = wfdb.rdsamp(
            os.path.join(
                PTBXL_PATH,
                row["filename_lr"]
            )
        )

        signal = preprocess_ecg(
            signal,
            fs=fields["fs"]
        )

        signal = torch.tensor(
            signal,
            dtype=torch.float32
        )

        label = torch.tensor(
            row["label"],
            dtype=torch.float32
        )

        return signal, label


print("✅ ECGDataset class created.")

In [ ]:
# ============================================================
# STEP 14A — CHECK DATAFRAME COLUMNS
# ============================================================

print("train_df shape:", train_df.shape)

print("\nColumns:")
for col in train_df.columns:
    print(" -", col)

print("\nFirst 3 rows:")
display(train_df.head(3))

In [ ]:
# Check possible label-related columns
print("\nPossible label columns:")

for col in train_df.columns:
    if any(word in col.lower() for word in [
        "label", "target", "class", "diagnos", "superclass"
    ]):
        print(" -", col)

In [ ]:
# ============================================================
# DIAGNOSTIC CLASS ANALYSIS
# ============================================================

print("Diagnostic class examples:")
print(train_df["diagnostic_classes"].head(10).to_string())

# Convert list-based labels into individual diagnostic classes
all_diagnostic_classes = train_df["diagnostic_classes"].explode()

# Remove missing values
all_diagnostic_classes = all_diagnostic_classes.dropna()

print("\nNumber of unique diagnostic classes:")
print(all_diagnostic_classes.nunique())

print("\nClass distribution:")
print(all_diagnostic_classes.value_counts())

In [ ]:
!pip install wfdb -q

In [ ]:
# ============================================================
# STEP — FINAL DATAFRAME & LABEL VALIDATION
# ============================================================

print("=" * 60)
print("FINAL DATA VALIDATION")
print("=" * 60)

# Check split sizes
print("\nSplit sizes:")
print(f"Train      : {len(train_df)}")
print(f"Validation : {len(val_df)}")
print(f"Test       : {len(test_df)}")

# Check required columns
required_columns = [
    "patient_id",
    "filename_lr",
    "diagnostic_classes"
]

print("\nRequired columns:")

for col in required_columns:
    if col in train_df.columns:
        print(f"✅ {col}")
    else:
        print(f"❌ {col} MISSING")

# Check missing values
print("\nMissing values:")

for col in required_columns:
    if col in train_df.columns:
        print(f"{col}: {train_df[col].isna().sum()}")

# Check label format
print("\nLabel format:")
print(type(train_df["diagnostic_classes"].iloc[0]))

# Check all splits have labels
print("\nLabels available:")
print("Train:", train_df["diagnostic_classes"].notna().all())
print("Validation:", val_df["diagnostic_classes"].notna().all())
print("Test:", test_df["diagnostic_classes"].notna().all())

print("\n" + "=" * 60)
print("✅ DATAFRAME VALIDATION COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# STEP — ECG DATASET
# ============================================================

import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import wfdb


class ECGDataset(Dataset):

    def __init__(
        self,
        dataframe,
        base_path="/content/ptb-xl",
        sampling_rate=100
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.base_path = base_path
        self.sampling_rate = sampling_rate

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        # ----------------------------------------------------
        # ECG file path
        # ----------------------------------------------------

        record_path = row["filename_lr"]

        # PTB-XL stores paths relative to the dataset directory
        record_path = os.path.join(
            self.base_path,
            record_path
        )

        # Remove extension if present
        record_path = os.path.splitext(record_path)[0]

        # ----------------------------------------------------
        # Load ECG
        # ----------------------------------------------------

        signal, fields = wfdb.rdsamp(record_path)

        # signal shape:
        # (samples, 12 leads)

        signal = signal.astype(np.float32)

        # ----------------------------------------------------
        # Convert to PyTorch format
        # ----------------------------------------------------

        signal = torch.tensor(
            signal,
            dtype=torch.float32
        )

        # (samples, leads)
        # -> (leads, samples)

        signal = signal.transpose(0, 1)

        # ----------------------------------------------------
        # Labels
        # ----------------------------------------------------

        labels = row["diagnostic_classes"]

        return {
            "ecg": signal,
            "labels": labels,
            "ecg_id": int(row.name)
        }


print("=" * 60)
print("CREATING ECG DATASET")
print("=" * 60)

train_dataset = ECGDataset(train_df)
val_dataset = ECGDataset(val_df)
test_dataset = ECGDataset(test_df)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\n✅ ECGDataset created successfully")

In [ ]:
# Check for empty labels in the training dataset

bad_indices = []

for i in range(len(train_dataset)):
    sample = train_dataset[i]

    if not sample["labels"]:
        bad_indices.append(i)

print("Total training samples:", len(train_dataset))
print("Samples with empty labels:", len(bad_indices))
print("Bad indices:", bad_indices)

In [ ]:
# ============================================================
# TEST SINGLE ECG
# ============================================================

sample = train_dataset[0]

print("=" * 60)
print("SINGLE ECG TEST")
print("=" * 60)

print("ECG shape:", sample["ecg"].shape)
print("ECG dtype:", sample["ecg"].dtype)
print("Labels:", sample["labels"])
print("ECG ID:", sample["ecg_id"])

assert sample["ecg"].ndim == 2
assert sample["ecg"].shape[0] == 12
assert sample["ecg"].dtype == torch.float32

print("\n✅ SINGLE ECG TEST PASSED")

In [ ]:
import torch
from torch.utils.data import DataLoader

# Convert the multi-label diagnostic classes into one integer class
# NORM = 0
# STTC = 1
# CD   = 2
# MI   = 3
# HYP  = 4

CLASS_NAMES = ["NORM", "STTC", "CD", "MI", "HYP"]

CLASS_TO_ID = {
    "NORM": 0,
    "STTC": 1,
    "CD": 2,
    "MI": 3,
    "HYP": 4
}


def collate_fn(batch):
    ecgs = []
    labels = []
    ids = []

    for item in batch:
        ecg = item["ecg"]

        # item["labels"] looks like ['NORM']
        label_list = item["labels"]

        # Take the first diagnostic class
        label_name = label_list[0]

        label_id = CLASS_TO_ID[label_name]

        ecgs.append(ecg)
        labels.append(label_id)
        ids.append(item["ecg_id"])

    ecgs = torch.stack(ecgs)
    labels = torch.tensor(labels, dtype=torch.long)

    return {
        "ecg": ecgs,
        "labels": labels,
        "ecg_id": ids
    }


# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)


print("=" * 60)
print("DATALOADER VALIDATION")
print("=" * 60)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


# Test one batch
ecgs, labels, ids = None, None, None

batch = next(iter(train_loader))

ecgs = batch["ecg"]
labels = batch["labels"]
ids = batch["ecg_id"]

print("\nBatch ECG shape:", ecgs.shape)
print("Batch labels shape:", labels.shape)
print("Batch labels:", labels)
print("Batch ECG IDs:", ids[:5])

print("\nECG dtype:", ecgs.dtype)
print("Labels dtype:", labels.dtype)

assert ecgs.ndim == 3
assert ecgs.shape[1] == 12
assert ecgs.shape[2] == 1000
assert labels.ndim == 1
assert labels.dtype == torch.long

print("\n✅ DATALOADER TEST PASSED")

In [ ]:
# ============================================================
# STEP 4: TEST ONE TRAINING BATCH
# ============================================================

print("=" * 60)
print("TRAINING BATCH TEST")
print("=" * 60)

batch = next(iter(train_loader))

ecgs = batch["ecg"]
labels = batch["labels"]
ids = batch["ecg_id"]

print("ECG shape    :", ecgs.shape)
print("ECG dtype    :", ecgs.dtype)

print("Labels shape :", labels.shape)
print("Labels dtype :", labels.dtype)

print("Labels       :", labels.tolist())
print("Number of IDs:", len(ids))
print("ECG IDs      :", ids)

# Validation checks
assert ecgs.ndim == 3, "ECG should be 3-dimensional"
assert ecgs.shape[1] == 12, "ECG should have 12 leads"
assert ecgs.shape[2] == 1000, "ECG should contain 1000 samples"

assert labels.ndim == 1, "Labels should be 1-dimensional"
assert len(ecgs) == len(labels) == len(ids), "ECG, labels and IDs must match"

print("\n✅ TRAINING BATCH TEST PASSED")

In [ ]:
# ============================================================
# STEP 3: VALIDATION + TEST DATALOADER TEST
# ============================================================

def test_loader(loader, name):
    print("=" * 60)
    print(f"{name} DATALOADER TEST")
    print("=" * 60)

    batch = next(iter(loader))

    ecgs = batch["ecg"]
    labels = batch["labels"]
    ids = batch["ecg_id"]

    print("ECG shape    :", ecgs.shape)
    print("ECG dtype    :", ecgs.dtype)
    print("Labels shape :", labels.shape)
    print("Labels dtype :", labels.dtype)
    print("Number of IDs:", len(ids))
    print("ECG IDs      :", ids)

    # Check ECG dimensions
    assert ecgs.ndim == 3
    assert ecgs.shape[0] == len(labels)
    assert ecgs.shape[0] == len(ids)
    assert ecgs.shape[1] == 12
    assert ecgs.shape[2] == 1000

    # Check labels
    assert labels.ndim == 1

    print(f"\n✅ {name} BATCH TEST PASSED")


# Test validation loader
test_loader(val_loader, "VALIDATION")

# Test test loader
test_loader(test_loader, "TEST")

In [ ]:
print("=" * 60)
print("CHECKING ECG LABELS")
print("=" * 60)

for i in range(10):
    item = train_dataset[i]

    print(f"\nECG {i}")
    print("ECG ID :", item.get("ecg_id"))
    print("Labels :", item.get("labels"))
    print("Type   :", type(item.get("labels")))

print("\n" + "=" * 60)
print("LABEL CHECK COMPLETE")
print("=" * 60)

In [ ]:
print("=" * 60)
print("SEARCHING FOR EMPTY LABELS")
print("=" * 60)

bad_samples = []

for i in range(len(train_dataset)):
    item = train_dataset[i]
    labels = item["labels"]

    if not labels:
        bad_samples.append({
            "index": i,
            "ecg_id": item["ecg_id"],
            "labels": labels
        })

print("Total training samples:", len(train_dataset))
print("Samples with empty labels:", len(bad_samples))

if bad_samples:
    print("\n❌ EMPTY LABEL SAMPLES:")
    for x in bad_samples[:20]:
        print(x)
else:
    print("\n✅ NO EMPTY LABELS FOUND")

In [ ]:
# ============================================================
# REMOVE ECG RECORDS WITH EMPTY DIAGNOSTIC LABELS
# ============================================================

def remove_empty_labels(df, name):
    df = df.copy()

    before = len(df)

    # Keep only records that have at least one diagnostic label
    df = df[
        df["diagnostic_classes"].apply(
            lambda x: isinstance(x, list) and len(x) > 0
        )
    ].reset_index(drop=True)

    after = len(df)

    print(f"{name}:")
    print(f"  Before : {before}")
    print(f"  Removed: {before - after}")
    print(f"  After  : {after}")
    print()

    return df


train_df = remove_empty_labels(train_df, "TRAIN")
val_df   = remove_empty_labels(val_df, "VALIDATION")
test_df  = remove_empty_labels(test_df, "TEST")

In [ ]:
# ============================================================
# VERIFY NO EMPTY LABELS REMAIN
# ============================================================

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    empty = df["diagnostic_classes"].apply(
        lambda x: not isinstance(x, list) or len(x) == 0
    ).sum()

    print(f"{name}: {len(df)} records | Empty labels: {empty}")

In [ ]:
# ============================================================
# SAFE COLLATE FUNCTION
# ============================================================

import torch

CLASS_TO_ID = {
    "NORM": 0,
    "STTC": 1,
    "CD": 2,
    "MI": 3,
    "HYP": 4
}


def collate_fn(batch):

    ecgs = []
    labels = []
    ids = []

    for item in batch:

        label_list = item["labels"]

        # Safety check
        if not isinstance(label_list, list) or len(label_list) == 0:
            continue

        # ECG
        ecgs.append(item["ecg"])

        # Use first diagnostic class
        label_name = label_list[0]

        # Convert class name -> integer
        label_id = CLASS_TO_ID[label_name]

        labels.append(label_id)

        # ECG ID
        ids.append(item["ecg_id"])

    # Stack ECG tensors
    ecgs = torch.stack(ecgs)

    # Labels tensor
    labels = torch.tensor(labels, dtype=torch.long)

    return {
        "ecg": ecgs,
        "labels": labels,
        "ecg_id": ids
    }

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
# ============================================================
# FINAL DATALOADER VALIDATION
# ============================================================

print("=" * 60)
print("DATALOADER VALIDATION")
print("=" * 60)

print("Train samples      :", len(train_dataset))
print("Validation samples :", len(val_dataset))
print("Test samples       :", len(test_dataset))

print("\nNumber of batches:")
print("Train      :", len(train_loader))
print("Validation :", len(val_loader))
print("Test       :", len(test_loader))


# ------------------------------------------------------------
# GET ONE TRAINING BATCH
# ------------------------------------------------------------

batch = next(iter(train_loader))

ecgs = batch["ecg"]
labels = batch["labels"]
ids = batch["ecg_id"]

print("\n" + "=" * 60)
print("SAMPLE TRAINING BATCH")
print("=" * 60)

print("ECG shape       :", ecgs.shape)
print("ECG dtype       :", ecgs.dtype)

print("Labels shape    :", labels.shape)
print("Labels dtype    :", labels.dtype)

print("Labels          :", labels.tolist())

print("Number of IDs   :", len(ids))
print("ECG IDs         :", ids)


# ------------------------------------------------------------
# SAFETY CHECKS
# ------------------------------------------------------------

assert ecgs.ndim == 3, "ECG should be 3-dimensional"
assert ecgs.shape[1] == 12, "ECG should have 12 leads"
assert ecgs.shape[2] == 1000, "ECG should contain 1000 samples"

assert labels.ndim == 1, "Labels should be 1-dimensional"

assert ecgs.shape[0] == labels.shape[0], \
    "Number of ECGs and labels must match"

assert ecgs.shape[0] == len(ids), \
    "Number of ECGs and IDs must match"

assert labels.min() >= 0, \
    "Invalid negative label"

assert labels.max() < 5, \
    "Label ID outside 5-class range"


print("\n" + "=" * 60)
print("✅ DATALOADER TEST PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# RECREATE DATASETS AFTER REMOVING EMPTY LABELS
# ============================================================

train_dataset = ECGDataset(train_df)
val_dataset   = ECGDataset(val_df)
test_dataset  = ECGDataset(test_df)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
# ============================================================
# FINAL SAFE COLLATE FUNCTION
# ============================================================

import torch


CLASS_TO_ID = {
    "NORM": 0,
    "STTC": 1,
    "CD": 2,
    "MI": 3,
    "HYP": 4
}


def collate_fn(batch):

    # Make sure there are no invalid samples
    valid_batch = []

    for item in batch:

        label_list = item["labels"]

        if not isinstance(label_list, list):
            raise ValueError(
                f"Labels must be a list. Got: {type(label_list)}"
            )

        if len(label_list) == 0:
            raise ValueError(
                f"Empty label found for ECG ID: {item['ecg_id']}"
            )

        valid_batch.append(item)

    # ECG tensors
    ecgs = torch.stack([
        item["ecg"]
        for item in valid_batch
    ])

    # Convert diagnostic class -> integer
    labels = torch.tensor(
        [
            CLASS_TO_ID[item["labels"][0]]
            for item in valid_batch
        ],
        dtype=torch.long
    )

    # ECG IDs
    ids = [
        item["ecg_id"]
        for item in valid_batch
    ]

    return {
        "ecg": ecgs,
        "labels": labels,
        "ecg_id": ids
    }

In [ ]:
# ============================================================
# STEP 3 FIX: TEST DATALOADER
# ============================================================

batch = next(iter(test_loader))

ecgs = batch["ecg"]
labels = batch["labels"]
ids = batch["ecg_id"]

print("=" * 60)
print("TEST DATALOADER TEST")
print("=" * 60)

print("ECG shape    :", ecgs.shape)
print("ECG dtype    :", ecgs.dtype)
print("Labels shape :", labels.shape)
print("Labels dtype :", labels.dtype)
print("Number of IDs:", len(ids))
print("ECG IDs      :", ids)

assert ecgs.ndim == 3
assert ecgs.shape[1] == 12
assert ecgs.shape[2] == 1000

assert labels.ndim == 1
assert len(ecgs) == len(labels)
assert len(ecgs) == len(ids)

print("\n✅ TEST BATCH TEST PASSED")

In [ ]:
# ============================================================
# STEP 3 FIX: CREATE TEST DATALOADER WITH A NEW NAME
# ============================================================

from torch.utils.data import DataLoader

TEST_BATCH_SIZE = 16

test_loader_dl = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

print("Test DataLoader created successfully")
print("Test batches:", len(test_loader_dl))

In [ ]:
# ============================================================
# TEST THE TEST DATALOADER
# ============================================================

batch = next(iter(test_loader_dl))

ecgs = batch["ecg"]
labels = batch["labels"]
ids = batch["ecg_id"]

print("=" * 60)
print("TEST DATALOADER TEST")
print("=" * 60)

print("ECG shape    :", ecgs.shape)
print("ECG dtype    :", ecgs.dtype)
print("Labels shape :", labels.shape)
print("Labels dtype :", labels.dtype)
print("Number of IDs:", len(ids))
print("ECG IDs      :", ids)

assert ecgs.ndim == 3
assert ecgs.shape[1] == 12
assert ecgs.shape[2] == 1000

assert labels.ndim == 1
assert len(ecgs) == len(labels)
assert len(ecgs) == len(ids)

print("\n✅ TEST BATCH TEST PASSED")

In [ ]:
from collections import Counter

def get_class_distribution(dataset, name):
    print("=" * 60)
    print(f"{name} CLASS DISTRIBUTION")
    print("=" * 60)

    counts = Counter()

    for i in range(len(dataset)):
        sample = dataset[i]
        label = sample["labels"][0]

        # Labels are strings such as NORM, MI, etc.
        counts[str(label)] += 1

    total = sum(counts.values())

    for class_name, count in sorted(counts.items()):
        percentage = (count / total) * 100

        print(
            f"{class_name:10s} -> "
            f"{count:3d} samples ({percentage:.2f}%)"
        )

    print(f"\nTotal samples: {total}")

In [ ]:
get_class_distribution(train_dataset, "TRAINING")
get_class_distribution(val_dataset, "VALIDATION")
get_class_distribution(test_dataset, "TEST")

In [ ]:
print("=" * 60)
print("FINAL DATASET SIZE CHECK")
print("=" * 60)

print("Train dataset     :", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset      :", len(test_dataset))

print("\nDataFrame sizes:")
print("Train dataframe   :", len(train_df))
print("Validation dataframe:", len(val_df))
print("Test dataframe    :", len(test_df))

In [ ]:
print("\nChecking last sample of each dataset...")

for name, dataset in [
    ("TRAIN", train_dataset),
    ("VALIDATION", val_dataset),
    ("TEST", test_dataset)
]:
    sample = dataset[len(dataset) - 1]

    print(f"\n{name}")
    print("Index:", len(dataset) - 1)
    print("ECG shape:", sample["ecg"].shape)
    print("Label:", sample["labels"])
    print("ECG ID:", sample["ecg_id"])

In [ ]:
print("=" * 70)
print("DATAFRAME vs ECGDATASET SIZE CHECK")
print("=" * 70)

print("\nTRAIN")
print("DataFrame :", len(train_df))
print("Dataset   :", len(train_dataset))
print("Dropped   :", len(train_df) - len(train_dataset))

print("\nVALIDATION")
print("DataFrame :", len(val_df))
print("Dataset   :", len(val_dataset))
print("Dropped   :", len(val_df) - len(val_dataset))

print("\nTEST")
print("DataFrame :", len(test_df))
print("Dataset   :", len(test_dataset))
print("Dropped   :", len(test_df) - len(test_dataset))

In [ ]:
print("\n" + "=" * 70)
print("DATAFRAME COLUMNS")
print("=" * 70)

print("Train columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(val_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

In [ ]:
print("\n" + "=" * 70)
print("DATASET SAMPLE KEYS")
print("=" * 70)

print(train_dataset[0].keys())

print("\nFirst training sample:")
print(train_dataset[0])

print("\nLast training sample:")
print(train_dataset[len(train_dataset) - 1])

In [ ]:
# ============================================================
# FINAL LABEL CHECK
# ============================================================

CLASS_NAMES = ['NORM', 'MI', 'CD', 'HYP', 'STTC']

CLASS_TO_ID = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

print("=" * 60)
print("FINAL LABEL ENCODING CHECK")
print("=" * 60)

for i in range(min(10, len(train_dataset))):

    sample = train_dataset[i]

    original_labels = sample["labels"]

    # Convert labels to multi-hot vector
    encoded = torch.zeros(len(CLASS_NAMES), dtype=torch.float32)

    for label in original_labels:
        if label in CLASS_TO_ID:
            encoded[CLASS_TO_ID[label]] = 1.0

    print(f"\nECG ID       : {sample['ecg_id']}")
    print(f"Original     : {original_labels}")
    print(f"Encoded      : {encoded.tolist()}")

In [ ]:
# Find a sample containing multiple labels

for i in range(len(train_dataset)):

    sample = train_dataset[i]

    if len(sample["labels"]) > 1:

        print("MULTI-LABEL SAMPLE FOUND")
        print("=" * 60)
        print("Index      :", i)
        print("ECG ID     :", sample["ecg_id"])
        print("Labels     :", sample["labels"])

        encoded = torch.zeros(len(CLASS_NAMES))

        for label in sample["labels"]:
            encoded[CLASS_TO_ID[label]] = 1.0

        print("Multi-hot  :", encoded.tolist())

        break

In [ ]:
# ============================================================
# CONVERT DATASET LABELS TO MULTI-HOT FORMAT
# ============================================================

CLASS_NAMES = ['NORM', 'MI', 'CD', 'HYP', 'STTC']

CLASS_TO_ID = {
    name: i for i, name in enumerate(CLASS_NAMES)
}


def encode_labels(labels):
    """
    Convert ECG labels into a multi-hot tensor.

    Example:
    ['MI', 'STTC']
    -> [0, 1, 0, 0, 1]
    """

    encoded = torch.zeros(
        len(CLASS_NAMES),
        dtype=torch.float32
    )

    for label in labels:
        if label in CLASS_TO_ID:
            encoded[CLASS_TO_ID[label]] = 1.0

    return encoded


class MultiLabelECGDataset(torch.utils.data.Dataset):

    def __init__(self, original_dataset):
        self.dataset = original_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):

        sample = self.dataset[index]

        return {
            "ecg": sample["ecg"].float(),
            "labels": encode_labels(sample["labels"]),
            "ecg_id": sample["ecg_id"]
        }


# Create final datasets
train_dataset_final = MultiLabelECGDataset(train_dataset)
val_dataset_final = MultiLabelECGDataset(val_dataset)
test_dataset_final = MultiLabelECGDataset(test_dataset)

print("Train:", len(train_dataset_final))
print("Validation:", len(val_dataset_final))
print("Test:", len(test_dataset_final))

In [ ]:
# ============================================================
# VERIFY FINAL DATASET
# ============================================================

sample = train_dataset_final[25]

print("=" * 60)
print("FINAL DATASET TEST")
print("=" * 60)

print("ECG shape :", sample["ecg"].shape)
print("ECG dtype :", sample["ecg"].dtype)

print("Labels    :", sample["labels"])
print("Label shape:", sample["labels"].shape)

print("ECG ID    :", sample["ecg_id"])

assert sample["ecg"].shape == (12, 1000)
assert sample["labels"].shape == (5,)
assert sample["labels"].dtype == torch.float32

print("\n✅ FINAL DATASET TEST PASSED")

In [ ]:
# ============================================================
# CREATE FINAL DATALOADERS
# ============================================================

from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset_final,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset_final,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset_final,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("=" * 60)
print("DATALOADERS CREATED")
print("=" * 60)

print("Training batches   :", len(train_loader))
print("Validation batches :", len(val_loader))
print("Test batches       :", len(test_loader))

In [ ]:
# ============================================================
# FINAL DATALOADER TEST
# ============================================================

batch = next(iter(train_loader))

ecgs = batch["ecg"]
labels = batch["labels"]
ids = batch["ecg_id"]

print("=" * 60)
print("FINAL DATALOADER TEST")
print("=" * 60)

print("ECG shape    :", ecgs.shape)
print("ECG dtype    :", ecgs.dtype)

print("Labels shape :", labels.shape)
print("Labels dtype :", labels.dtype)

print("IDs shape    :", ids.shape)
print("Number of IDs:", len(ids))

print("\nFirst 5 label vectors:")
print(labels[:5])

# Safety checks
assert ecgs.shape[1:] == (12, 1000)
assert labels.shape[1] == 5
assert ecgs.dtype == torch.float32
assert labels.dtype == torch.float32
assert len(ecgs) == len(labels) == len(ids)

print("\n✅ DATALOADER TEST PASSED")

In [ ]:
import torch
import torch.nn as nn


# ============================================================
# ECG RESIDUAL BLOCK
# ============================================================

class ECGResBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):

        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=7,
            stride=stride,
            padding=3,
            bias=False
        )

        self.bn1 = nn.BatchNorm1d(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=7,
            stride=1,
            padding=3,
            bias=False
        )

        self.bn2 = nn.BatchNorm1d(out_channels)

        # Shortcut
        if in_channels != out_channels or stride != 1:

            self.shortcut = nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm1d(out_channels)
            )

        else:
            self.shortcut = nn.Identity()

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):

        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out


# ============================================================
# ECG MODEL
# ============================================================

class ECGResNet(nn.Module):

    def __init__(self, num_classes=5):

        super().__init__()

        # Input:
        # [batch, 12, 1000]

        self.stem = nn.Sequential(

            nn.Conv1d(
                in_channels=12,
                out_channels=64,
                kernel_size=7,
                stride=2,
                padding=3,
                bias=False
            ),

            nn.BatchNorm1d(64),

            nn.ReLU(inplace=True),

            nn.MaxPool1d(
                kernel_size=3,
                stride=2,
                padding=1
            )
        )

        self.block1 = ECGResBlock(
            64,
            64
        )

        self.block2 = ECGResBlock(
            64,
            128,
            stride=2
        )

        self.block3 = ECGResBlock(
            128,
            256,
            stride=2
        )

        self.block4 = ECGResBlock(
            256,
            512,
            stride=2
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(

            nn.Linear(512, 128),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # x:
        # [batch, 12, 1000]

        x = self.stem(x)

        x = self.block1(x)

        x = self.block2(x)

        x = self.block3(x)

        x = self.block4(x)

        # [batch, 512, sequence_length]
        x = self.global_pool(x)

        # [batch, 512, 1]
        x = torch.flatten(x, 1)

        # [batch, 5]
        x = self.classifier(x)

        return x

In [11]:
import torch

# Create model architecture
test_model = ECGResNet(
    num_classes=5
)

# Load checkpoint
state_dict = torch.load(
    checkpoint_path,
    map_location=device
)

# Handle checkpoints that may contain extra information
if isinstance(state_dict, dict) and "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]

test_model.load_state_dict(state_dict)

# Move model to CPU/GPU
test_model = test_model.to(device)

# Evaluation mode
test_model.eval()

print("✅ Best model loaded successfully!")

NameError: name 'ECGResNet' is not defined

In [ ]:
import os

print("=== CURRENT VARIABLES ===")

for name in [
    "train_loader",
    "val_loader",
    "test_loader",
    "train_dataset",
    "val_dataset",
    "test_dataset",
    "train_data",
    "test_data",
    "X_test",
    "y_test",
    "X_train",
    "y_train"
]:
    print(f"{name}: {name in globals()}")

print("\n=== /content/drive/MyDrive ===")

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    # Don't print everything
    if any(x in root.lower() for x in ["ecg", "ptb", "dataset", "physionet"]):
        print("\nFolder:", root)
        print("Files:", files[:10])

print("\n=== /content ===")
print(os.listdir("/content"))

In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure model is in evaluation mode
test_model.eval()

all_preds = []
all_labels = []

# Evaluate without calculating gradients
with torch.no_grad():

    for inputs, labels in test_loader:

        # Move data to CPU/GPU
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Model prediction
        outputs = test_model(inputs)

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        # Store predictions and actual labels
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Accuracy
accuracy = accuracy_score(all_labels, all_preds)

print("=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

print(f"\nAccuracy: {accuracy * 100:.2f}%")

# Classification Report
print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=["NORM", "STTC", "CD", "MI", "HYP"],
        zero_division=0
    )
)

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

print("\nConfusion Matrix:")
print(cm)

# Plot confusion matrix
plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["NORM", "STTC", "CD", "MI", "HYP"],
    yticklabels=["NORM", "STTC", "CD", "MI", "HYP"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("ECGResNet Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

In [ ]:
test_model = ECGResNet(
    num_classes=5
)

test_model.load_state_dict(
    torch.load(checkpoint_path, map_location=device)
)

test_model = test_model.to(device)
test_model.eval()

print("Best model loaded successfully!")

In [ ]:
import torch
import os

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Path to your saved model
checkpoint_path = "/content/drive/MyDrive/best_ecg_model.pth"

print("Checkpoint exists:", os.path.exists(checkpoint_path))

# Recreate the model architecture
test_model = ECGResNet(
    num_classes=5
)

# Load trained weights
state_dict = torch.load(
    checkpoint_path,
    map_location=device
)

test_model.load_state_dict(state_dict)

# Move model to device
test_model = test_model.to(device)

# Evaluation mode
test_model.eval()

print("✅ Best model loaded successfully!")

In [ ]:
import torch
import os

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Path to your saved model
checkpoint_path = "/content/drive/MyDrive/best_ecg_model.pth"

print("Checkpoint exists:", os.path.exists(checkpoint_path))

# Recreate the model architecture
test_model = ECGResNet(
    num_classes=5
)

# Load trained weights
state_dict = torch.load(
    checkpoint_path,
    map_location=device
)

test_model.load_state_dict(state_dict)

# Move model to device
test_model = test_model.to(device)

# Evaluation mode
test_model.eval()

print("✅ Best model loaded successfully!")

In [ ]:
import os

print("Files in /content:")
print(os.listdir("/content"))

print("\nFiles in /content/drive:")
print(os.listdir("/content/drive") if os.path.exists("/content/drive") else "Drive not mounted")

print("\nFiles in MyDrive:")
mydrive = "/content/drive/MyDrive"

if os.path.exists(mydrive):
    print(os.listdir(mydrive))
else:
    print("MyDrive not mounted")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import torch
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# Correct model path
checkpoint_path = "/content/drive/MyDrive/best_ecg_model.pth"

print("Checkpoint exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
    print(f"Model size: {size_mb:.2f} MB")
else:
    print("ERROR: Model checkpoint not found!")

In [ ]:
import os

print("Drive mounted:",
      os.path.exists("/content/drive/MyDrive"))

print("\nMyDrive contents:")
print(os.listdir("/content/drive/MyDrive")[:20])

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import torch

test_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for signals, labels in test_loader:
        signals = signals.to(device)
        labels = labels.to(device)

        outputs = test_model(signals)
        predictions = torch.argmax(outputs, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert to numpy
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Accuracy
accuracy = accuracy_score(all_labels, all_preds)

print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Detailed report
print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=["NORM", "STTC", "CD", "MI", "HYP"],
        zero_division=0
    )
)

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

print("\nConfusion Matrix:")
print(cm)

In [ ]:
test_model = ECGResNet(
    num_classes=5
)

test_model.load_state_dict(
    torch.load(checkpoint_path, map_location=device)
)

test_model = test_model.to(device)
test_model.eval()

print("Best model loaded successfully!")

In [ ]:
# ============================================================
# MODEL TEST
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ECGResNet(
    num_classes=5
).to(device)


# Fake ECG batch
x = torch.randn(
    16,
    12,
    1000
).to(device)


with torch.no_grad():

    output = model(x)


print("=" * 60)
print("MODEL TEST")
print("=" * 60)

print("Input shape :", x.shape)
print("Output shape:", output.shape)

print("=" * 60)


assert x.shape == (16, 12, 1000)

assert output.shape == (16, 5)

print("✅ MODEL TEST PASSED")

In [ ]:
print(model)
print(type(model).__name__)

In [ ]:
import torch
import torch.nn as nn


# ============================================================
# CLASS NAMES
# ============================================================

CLASS_NAMES = ["CD", "HYP", "MI", "NORM", "STTC"]

CLASS_TO_ID = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}


# ============================================================
# CALCULATE CLASS BALANCE
# ============================================================

positive_counts = torch.zeros(len(CLASS_NAMES))
negative_counts = torch.zeros(len(CLASS_NAMES))


for i in range(len(train_dataset)):

    # Original labels are strings
    labels = train_dataset[i]["labels"]

    # Convert labels into multi-hot vector
    target = torch.zeros(len(CLASS_NAMES))

    for label in labels:

        if label in CLASS_TO_ID:
            target[CLASS_TO_ID[label]] = 1.0

    positive_counts += target
    negative_counts += (1.0 - target)


# Avoid division by zero
positive_counts = torch.clamp(
    positive_counts,
    min=1
)


# ============================================================
# POSITIVE CLASS WEIGHTS
# ============================================================

pos_weights = negative_counts / positive_counts


print("=" * 60)
print("CLASS BALANCE")
print("=" * 60)

for i, class_name in enumerate(CLASS_NAMES):

    print(
        f"{class_name:5s} -> "
        f"Positive: {int(positive_counts[i]):3d} | "
        f"Negative: {int(negative_counts[i]):3d} | "
        f"Weight: {pos_weights[i]:.3f}"
    )


# ============================================================
# MOVE WEIGHTS TO DEVICE
# ============================================================

pos_weights = pos_weights.to(device)


# ============================================================
# LOSS
# ============================================================

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)


# ============================================================
# FINAL CHECK
# ============================================================

print("\n" + "=" * 60)
print("TRAINING SETUP")
print("=" * 60)

print("Loss         :", criterion.__class__.__name__)
print("Optimizer    :", optimizer.__class__.__name__)
print("Learning rate: 1e-3")
print("Pos weights  :", pos_weights.detach().cpu())

print("\n✅ LOSS + OPTIMIZER READY")

In [ ]:
# ============================================================
# LOSS SANITY TEST
# ============================================================

batch = next(iter(train_loader))

ecgs = batch["ecg"].to(device)
labels = batch["labels"].float().to(device)

logits = model(ecgs)

loss = criterion(logits, labels)


print("=" * 60)
print("LOSS SANITY TEST")
print("=" * 60)

print("ECG shape    :", ecgs.shape)
print("Labels shape :", labels.shape)
print("Logits shape :", logits.shape)
print("Loss         :", loss.item())


assert ecgs.shape[1:] == (12, 1000)
assert labels.shape == (ecgs.shape[0], 5)
assert logits.shape == labels.shape
assert torch.isfinite(loss)


print("\n✅ LOSS TEST PASSED")

In [ ]:
# ============================================================
# TRAINING LOOP - FIXED FOR DICTIONARY DATALOADER
# ============================================================

import torch

EPOCHS = 30

best_val_loss = float("inf")

train_losses = []
val_losses = []

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

for epoch in range(EPOCHS):

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    running_train_loss = 0.0

    for batch in train_loader:

        # Get data using dictionary keys
        ecgs = batch["ecg"].to(device)
        labels = batch["labels"].to(device)

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(ecgs)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_train_loss += loss.item()

    train_loss = running_train_loss / len(train_loader)

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0

    with torch.no_grad():

        for batch in val_loader:

            ecgs = batch["ecg"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(ecgs)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

    val_loss = running_val_loss / len(val_loader)

    # Store losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

if val_accuracy > best_val_accuracy:

    best_val_accuracy = val_accuracy

    torch.save(
        model.state_dict(),
        "best_ecg_model.pth"
    )

    saved = " ⭐ BEST"
else:
    saved = ""

    # ========================================================
    # PRINT PROGRESS
    # ========================================================

    print(
        f"Epoch [{epoch + 1:02d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
        f"{saved}"
    )


print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(f"Best validation loss: {best_val_loss:.4f}")
print("Best model saved as: best_ecg_model.pth")

In [ ]:
print("\n" + "=" * 70)
print("FINAL MODEL EVALUATION")
print("=" * 70)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load best model
model.load_state_dict(
    torch.load("best_ecg_model.pth", map_location=device)
)
model.to(device)
model.eval()

print("Best model loaded successfully.")

all_true = []
all_pred = []
all_prob = []
all_ids = []

with torch.no_grad():

    for batch in test_loader:

        # DataLoader returns a dictionary
        ecgs = batch["ecg"].to(device)
        labels = batch["labels"].to(device)
        ids = batch["ecg_id"]

        # Forward pass
        logits = model(ecgs)

        # Probabilities
        probabilities = torch.sigmoid(logits)

        # Multi-label prediction
        predictions = (probabilities >= 0.5).float()

        # Store results
        all_true.append(labels.cpu())
        all_pred.append(predictions.cpu())
        all_prob.append(probabilities.cpu())
        all_ids.extend(ids)

# Combine batches
y_true = torch.cat(all_true, dim=0).numpy()
y_pred = torch.cat(all_pred, dim=0).numpy()
y_prob = torch.cat(all_prob, dim=0).numpy()

print("\nEvaluation data collected successfully.")

print("True labels shape :", y_true.shape)
print("Predictions shape :", y_pred.shape)
print("Probabilities shape:", y_prob.shape)
print("Number of ECG IDs :", len(all_ids))

print("\n" + "=" * 70)
print("EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

CLASS_NAMES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

print("=" * 70)
print("FINAL PERFORMANCE METRICS")
print("=" * 70)

# --------------------------------------------------
# 1. Exact-match accuracy
# --------------------------------------------------
exact_match_accuracy = np.mean(
    np.all(y_true == y_pred, axis=1)
)

# --------------------------------------------------
# 2. Micro metrics
# --------------------------------------------------
micro_precision = precision_score(
    y_true, y_pred, average="micro", zero_division=0
)

micro_recall = recall_score(
    y_true, y_pred, average="micro", zero_division=0
)

micro_f1 = f1_score(
    y_true, y_pred, average="micro", zero_division=0
)

# --------------------------------------------------
# 3. Macro metrics
# --------------------------------------------------
macro_precision = precision_score(
    y_true, y_pred, average="macro", zero_division=0
)

macro_recall = recall_score(
    y_true, y_pred, average="macro", zero_division=0
)

macro_f1 = f1_score(
    y_true, y_pred, average="macro", zero_division=0
)

# --------------------------------------------------
# 4. ROC-AUC
# --------------------------------------------------
try:
    macro_auc = roc_auc_score(
        y_true,
        y_prob,
        average="macro"
    )
except ValueError:
    macro_auc = float("nan")

# --------------------------------------------------
# 5. Average Precision / AUPRC
# --------------------------------------------------
try:
    macro_auprc = average_precision_score(
        y_true,
        y_prob,
        average="macro"
    )
except ValueError:
    macro_auprc = float("nan")

# --------------------------------------------------
# Print overall metrics
# --------------------------------------------------

print(f"\nExact-match Accuracy : {exact_match_accuracy:.4f}")
print(f"Micro Precision      : {micro_precision:.4f}")
print(f"Micro Recall         : {micro_recall:.4f}")
print(f"Micro F1             : {micro_f1:.4f}")

print(f"\nMacro Precision      : {macro_precision:.4f}")
print(f"Macro Recall         : {macro_recall:.4f}")
print(f"Macro F1             : {macro_f1:.4f}")

print(f"\nMacro ROC-AUC        : {macro_auc:.4f}")
print(f"Macro AUPRC          : {macro_auprc:.4f}")

# --------------------------------------------------
# Per-class metrics
# --------------------------------------------------

print("\n" + "=" * 70)
print("PER-CLASS PERFORMANCE")
print("=" * 70)

report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
)

print(report)

print("=" * 70)
print("EVALUATION METRICS COMPLETE")
print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("=" * 70)
print("PER-CLASS CONFUSION MATRICES")
print("=" * 70)

for i, class_name in enumerate(CLASS_NAMES):

    cm = confusion_matrix(
        y_true[:, i],
        y_pred[:, i],
        labels=[0, 1]
    )

    print(f"\n{class_name}")
    print(cm)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Negative", "Positive"]
    )

    disp.plot()
    plt.title(f"Confusion Matrix - {class_name}")
    plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(9, 7))

for i, class_name in enumerate(CLASS_NAMES):

    fpr, tpr, _ = roc_curve(
        y_true[:, i],
        y_prob[:, i]
    )

    roc_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} (AUC = {roc_auc:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - ECG Multi-Label Classification")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# CONFUSION MATRICES - ECG MULTI-LABEL CLASSIFICATION
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix

CLASS_NAMES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

print("=" * 70)
print("CONFUSION MATRIX ANALYSIS")
print("=" * 70)

# Convert to numpy if necessary
y_true_np = np.asarray(y_true)
y_pred_np = np.asarray(y_pred)

print("\nTrue labels shape :", y_true_np.shape)
print("Predictions shape :", y_pred_np.shape)

# Generate one 2x2 confusion matrix per class
cm_all = multilabel_confusion_matrix(
    y_true_np,
    y_pred_np
)

for i, class_name in enumerate(CLASS_NAMES):

    tn, fp, fn, tp = cm_all[i].ravel()

    print(f"\n{'-' * 50}")
    print(f"CLASS: {class_name}")
    print(f"{'-' * 50}")

    print(f"True Negatives  : {tn}")
    print(f"False Positives : {fp}")
    print(f"False Negatives : {fn}")
    print(f"True Positives  : {tp}")

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    print(f"Precision       : {precision:.4f}")
    print(f"Recall          : {recall:.4f}")


# ============================================================
# VISUALIZE CONFUSION MATRICES
# ============================================================

for i, class_name in enumerate(CLASS_NAMES):

    tn, fp, fn, tp = cm_all[i].ravel()

    matrix = np.array([
        [tn, fp],
        [fn, tp]
    ])

    plt.figure(figsize=(5, 4))

    plt.imshow(matrix)

    plt.title(f"Confusion Matrix - {class_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.xticks([0, 1], ["Negative", "Positive"])
    plt.yticks([0, 1], ["Negative", "Positive"])

    for row in range(2):
        for col in range(2):
            plt.text(
                col,
                row,
                str(matrix[row, col]),
                ha="center",
                va="center",
                fontsize=14
            )

    plt.colorbar()
    plt.tight_layout()
    plt.show()


print("\n" + "=" * 70)
print("CONFUSION MATRIX ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 1: PER-CLASS CONFUSION MATRIX ANALYSIS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix

CLASS_NAMES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

# Make sure arrays are NumPy arrays
y_true_np = np.asarray(y_true).astype(int)
y_pred_np = np.asarray(y_pred).astype(int)

print("=" * 70)
print("CONFUSION MATRIX ANALYSIS")
print("=" * 70)

print("\nTrue labels shape :", y_true_np.shape)
print("Predictions shape :", y_pred_np.shape)

# One binary confusion matrix for each class
cms = multilabel_confusion_matrix(
    y_true_np,
    y_pred_np
)

for i, class_name in enumerate(CLASS_NAMES):

    tn, fp, fn, tp = cms[i].ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    print("\n" + "-" * 60)
    print(f"{class_name}")
    print("-" * 60)

    print(f"True Positive  : {tp}")
    print(f"False Positive : {fp}")
    print(f"True Negative  : {tn}")
    print(f"False Negative : {fn}")

    print(f"Precision      : {precision:.4f}")
    print(f"Recall         : {recall:.4f}")
    print(f"Specificity    : {specificity:.4f}")


# ============================================================
# VISUALIZATION
# ============================================================

for i, class_name in enumerate(CLASS_NAMES):

    tn, fp, fn, tp = cms[i].ravel()

    matrix = np.array([
        [tn, fp],
        [fn, tp]
    ])

    plt.figure(figsize=(5, 4))

    plt.imshow(matrix)

    plt.title(f"Confusion Matrix - {class_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.xticks(
        [0, 1],
        ["Negative", "Positive"]
    )

    plt.yticks(
        [0, 1],
        ["Negative", "Positive"]
    )

    for row in range(2):
        for col in range(2):
            plt.text(
                col,
                row,
                str(matrix[row, col]),
                ha="center",
                va="center",
                fontsize=14
            )

    plt.colorbar()
    plt.tight_layout()
    plt.show()


print("\n" + "=" * 70)
print("CONFUSION MATRIX ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 2: PRECISION-RECALL CURVES
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

CLASS_NAMES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

# Convert to NumPy
y_true_np = np.asarray(y_true).astype(int)
y_prob_np = np.asarray(y_prob)

print("=" * 70)
print("PRECISION-RECALL ANALYSIS")
print("=" * 70)

print("True labels shape :", y_true_np.shape)
print("Probabilities shape:", y_prob_np.shape)

ap_scores = []

# ============================================================
# INDIVIDUAL PR CURVES
# ============================================================

for i, class_name in enumerate(CLASS_NAMES):

    precision, recall, thresholds = precision_recall_curve(
        y_true_np[:, i],
        y_prob_np[:, i]
    )

    ap = average_precision_score(
        y_true_np[:, i],
        y_prob_np[:, i]
    )

    ap_scores.append(ap)

    plt.figure(figsize=(7, 5))

    plt.plot(
        recall,
        precision,
        linewidth=2,
        label=f'{class_name} (AP = {ap:.3f})'
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve - {class_name}")

    plt.xlim(0, 1)
    plt.ylim(0, 1.05)

    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.tight_layout()
    plt.show()


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("AVERAGE PRECISION BY CLASS")
print("=" * 70)

for class_name, ap in zip(CLASS_NAMES, ap_scores):
    print(f"{class_name:<8}: {ap:.4f}")

macro_ap = np.mean(ap_scores)

micro_ap = average_precision_score(
    y_true_np,
    y_prob_np,
    average="micro"
)

print("\nMacro Average Precision :", f"{macro_ap:.4f}")
print("Micro Average Precision :", f"{micro_ap:.4f}")

print("\n" + "=" * 70)
print("PRECISION-RECALL ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
import numpy as np
import torch
from sklearn.metrics import f1_score

# ============================================================
# PER-CLASS THRESHOLD OPTIMIZATION
# ============================================================

CLASS_NAMES = ["CD", "HYP", "MI", "NORM", "STTC"]

model.eval()

val_true = []
val_prob = []

with torch.no_grad():

    for batch in val_loader:

        # Handle dictionary-style batches
        if isinstance(batch, dict):
            ecgs = batch["ecg"]
            labels = batch["labels"]

        # Handle tuple/list-style batches
        else:
            ecgs = batch[0]
            labels = batch[1]

        ecgs = ecgs.to(device)
        labels = labels.to(device)

        outputs = model(ecgs)

        # Convert logits → probabilities
        probs = torch.sigmoid(outputs)

        val_true.append(labels.cpu().numpy())
        val_prob.append(probs.cpu().numpy())


val_true = np.concatenate(val_true, axis=0)
val_prob = np.concatenate(val_prob, axis=0)

print("Validation labels shape :", val_true.shape)
print("Validation probabilities shape :", val_prob.shape)


# ============================================================
# SEARCH BEST THRESHOLD FOR EACH CLASS
# ============================================================

best_thresholds = []
best_f1_scores = []

thresholds = np.arange(0.10, 0.91, 0.01)

print("\n" + "=" * 70)
print("PER-CLASS THRESHOLD OPTIMIZATION")
print("=" * 70)

for class_idx, class_name in enumerate(CLASS_NAMES):

    y_true_class = val_true[:, class_idx]
    y_prob_class = val_prob[:, class_idx]

    best_threshold = 0.50
    best_f1 = 0.0

    for threshold in thresholds:

        y_pred_class = (y_prob_class >= threshold).astype(int)

        score = f1_score(
            y_true_class,
            y_pred_class,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    best_thresholds.append(best_threshold)
    best_f1_scores.append(best_f1)

    baseline_pred = (y_prob_class >= 0.50).astype(int)

    baseline_f1 = f1_score(
        y_true_class,
        baseline_pred,
        zero_division=0
    )

    print(
        f"{class_name:5s} | "
        f"0.50 F1: {baseline_f1:.4f} | "
        f"Best Threshold: {best_threshold:.2f} | "
        f"Best F1: {best_f1:.4f}"
    )


print("\n" + "=" * 70)
print("OPTIMIZED THRESHOLDS")
print("=" * 70)

for name, threshold in zip(CLASS_NAMES, best_thresholds):
    print(f"{name:5s}: {threshold:.2f}")

In [ ]:
# ============================================================
# TEST EVALUATION WITH OPTIMIZED PER-CLASS THRESHOLDS
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    hamming_loss
)

CLASS_NAMES = ["CD", "HYP", "MI", "NORM", "STTC"]

# Thresholds learned ONLY from validation set
OPTIMIZED_THRESHOLDS = np.array([
    0.78,   # CD
    0.66,   # HYP
    0.69,   # MI
    0.77,   # NORM
    0.58    # STTC
])

model.eval()

test_true = []
test_prob = []

with torch.no_grad():

    for batch in test_loader:

        # ----------------------------------------------------
        # Handle dictionary batches
        # ----------------------------------------------------
        if isinstance(batch, dict):

            ecgs = batch["ecg"]
            labels = batch["labels"]

        # ----------------------------------------------------
        # Handle tuple/list batches
        # ----------------------------------------------------
        else:

            ecgs = batch[0]
            labels = batch[1]

        ecgs = ecgs.to(device)
        labels = labels.to(device)

        outputs = model(ecgs)

        probabilities = torch.sigmoid(outputs)

        test_true.append(labels.cpu().numpy())
        test_prob.append(probabilities.cpu().numpy())


# Combine batches
test_true = np.concatenate(test_true, axis=0)
test_prob = np.concatenate(test_prob, axis=0)


# ============================================================
# BASELINE PREDICTIONS (THRESHOLD = 0.50)
# ============================================================

test_pred_baseline = (test_prob >= 0.50).astype(int)


# ============================================================
# OPTIMIZED PREDICTIONS
# ============================================================

test_pred_optimized = (
    test_prob >= OPTIMIZED_THRESHOLDS
).astype(int)


# ============================================================
# BASIC SHAPE CHECK
# ============================================================

print("=" * 70)
print("TEST DATA")
print("=" * 70)

print("True labels shape       :", test_true.shape)
print("Probabilities shape     :", test_prob.shape)
print("Baseline predictions    :", test_pred_baseline.shape)
print("Optimized predictions   :", test_pred_optimized.shape)


# ============================================================
# FUNCTION FOR METRICS
# ============================================================

def calculate_metrics(y_true, y_pred, name):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    exact_match = np.mean(
        np.all(y_true == y_pred, axis=1)
    )

    micro_precision = precision_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    micro_recall = recall_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    micro_f1 = f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    macro_precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    hamming = hamming_loss(
        y_true,
        y_pred
    )

    print(f"Exact-match Accuracy : {exact_match:.4f}")
    print(f"Micro Precision      : {micro_precision:.4f}")
    print(f"Micro Recall         : {micro_recall:.4f}")
    print(f"Micro F1             : {micro_f1:.4f}")

    print(f"Macro Precision      : {macro_precision:.4f}")
    print(f"Macro Recall         : {macro_recall:.4f}")
    print(f"Macro F1             : {macro_f1:.4f}")

    print(f"Hamming Loss         : {hamming:.4f}")

    return {
        "exact_match": exact_match,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "hamming_loss": hamming
    }


# ============================================================
# BASELINE
# ============================================================

baseline_metrics = calculate_metrics(
    test_true,
    test_pred_baseline,
    "BASELINE — THRESHOLD 0.50"
)


# ============================================================
# OPTIMIZED
# ============================================================

optimized_metrics = calculate_metrics(
    test_true,
    test_pred_optimized,
    "OPTIMIZED — PER-CLASS THRESHOLDS"
)


# ============================================================
# PER-CLASS COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("PER-CLASS PERFORMANCE COMPARISON")
print("=" * 70)

for i, class_name in enumerate(CLASS_NAMES):

    baseline_f1 = f1_score(
        test_true[:, i],
        test_pred_baseline[:, i],
        zero_division=0
    )

    optimized_f1 = f1_score(
        test_true[:, i],
        test_pred_optimized[:, i],
        zero_division=0
    )

    baseline_precision = precision_score(
        test_true[:, i],
        test_pred_baseline[:, i],
        zero_division=0
    )

    optimized_precision = precision_score(
        test_true[:, i],
        test_pred_optimized[:, i],
        zero_division=0
    )

    baseline_recall = recall_score(
        test_true[:, i],
        test_pred_baseline[:, i],
        zero_division=0
    )

    optimized_recall = recall_score(
        test_true[:, i],
        test_pred_optimized[:, i],
        zero_division=0
    )

    print(f"\n{class_name}")
    print(f"  Precision : {baseline_precision:.4f} → {optimized_precision:.4f}")
    print(f"  Recall    : {baseline_recall:.4f} → {optimized_recall:.4f}")
    print(f"  F1        : {baseline_f1:.4f} → {optimized_f1:.4f}")


# ============================================================
# DETAILED CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("OPTIMIZED CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        test_true,
        test_pred_optimized,
        target_names=CLASS_NAMES,
        zero_division=0
    )
)


print("\n" + "=" * 70)
print("TEST THRESHOLD EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# ECG ROBUSTNESS EXPERIMENT
# Controlled Signal Degradation
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("ECG ROBUSTNESS EXPERIMENT")
print("=" * 70)


# ------------------------------------------------------------
# 1. Gaussian Noise
# ------------------------------------------------------------

def add_gaussian_noise(signal, noise_level=0.01):
    """
    Add controlled Gaussian noise to ECG signal.

    signal:
        ECG array with shape (..., time)

    noise_level:
        Relative noise strength
    """
    signal = np.asarray(signal, dtype=np.float32)

    signal_std = np.std(signal, axis=-1, keepdims=True)

    noise = np.random.normal(
        0,
        noise_level * (signal_std + 1e-8),
        size=signal.shape
    )

    return signal + noise


# ------------------------------------------------------------
# 2. Baseline Wander
# ------------------------------------------------------------

def add_baseline_wander(signal, amplitude=0.05, frequency=0.5):
    """
    Add low-frequency baseline drift.
    """

    signal = np.asarray(signal, dtype=np.float32)

    time = np.linspace(
        0,
        1,
        signal.shape[-1]
    )

    wander = amplitude * np.sin(
        2 * np.pi * frequency * time
    )

    return signal + wander


# ------------------------------------------------------------
# 3. Amplitude Scaling
# ------------------------------------------------------------

def add_amplitude_distortion(signal, scale=1.0):
    """
    Simulate amplitude variation.
    """

    signal = np.asarray(signal, dtype=np.float32)

    return signal * scale


# ------------------------------------------------------------
# 4. Combined Degradation
# ------------------------------------------------------------

def degrade_ecg(signal, severity="clean"):
    """
    Apply controlled ECG degradation.

    Levels:
        clean
        mild
        moderate
        severe
    """

    signal = np.asarray(signal, dtype=np.float32).copy()

    if severity == "clean":
        return signal

    elif severity == "mild":

        signal = add_gaussian_noise(
            signal,
            noise_level=0.02
        )

        signal = add_baseline_wander(
            signal,
            amplitude=0.02,
            frequency=0.5
        )

    elif severity == "moderate":

        signal = add_gaussian_noise(
            signal,
            noise_level=0.05
        )

        signal = add_baseline_wander(
            signal,
            amplitude=0.05,
            frequency=0.7
        )

    elif severity == "severe":

        signal = add_gaussian_noise(
            signal,
            noise_level=0.10
        )

        signal = add_baseline_wander(
            signal,
            amplitude=0.10,
            frequency=1.0
        )

        signal = add_amplitude_distortion(
            signal,
            scale=0.85
        )

    else:
        raise ValueError(
            f"Unknown severity: {severity}"
        )

    return signal


print("Degradation functions created successfully.")
print("Levels: clean → mild → moderate → severe")

In [ ]:
# ============================================================
# CHECK EXISTING TEST DATA
# ============================================================

print("Available variables related to ECG/test data:")
print()

for name in sorted(globals().keys()):
    name_lower = name.lower()

    if any(keyword in name_lower for keyword in [
        "test",
        "ecg",
        "signal",
        "dataset",
        "loader"
    ]):
        try:
            obj = globals()[name]

            if hasattr(obj, "shape"):
                print(f"{name:30s} shape = {obj.shape}")
            else:
                print(f"{name:30s} type  = {type(obj).__name__}")

        except Exception:
            pass

In [ ]:
# ============================================================
# INSPECT TEST DATASET SAMPLE
# ============================================================

print("=" * 70)
print("TEST DATASET STRUCTURE")
print("=" * 70)

print("Dataset type:", type(test_dataset_final).__name__)
print("Dataset length:", len(test_dataset_final))

sample = test_dataset_final[0]

print("\nSample type:", type(sample).__name__)

if isinstance(sample, dict):
    print("Dictionary keys:", list(sample.keys()))

    for key, value in sample.items():
        if hasattr(value, "shape"):
            print(f"{key:20s} shape = {value.shape}")
        else:
            print(f"{key:20s} type  = {type(value).__name__}")

elif isinstance(sample, (tuple, list)):
    print("Number of elements:", len(sample))

    for i, value in enumerate(sample):
        if hasattr(value, "shape"):
            print(f"sample[{i}] shape = {value.shape}")
        else:
            print(f"sample[{i}] type  = {type(value).__name__}")

else:
    print("Sample shape:", getattr(sample, "shape", None))
    print("Sample type:", type(sample).__name__)

In [ ]:
# ============================================================
# FIX MODEL REFERENCE FOR ROBUSTNESS EXPERIMENT
# ============================================================

print("=" * 70)
print("SEARCHING FOR AVAILABLE MODEL VARIABLES")
print("=" * 70)

# Find model-like variables already present in the notebook
model_candidates = {}

for name, value in globals().items():
    try:
        if isinstance(value, torch.nn.Module):
            model_candidates[name] = value
    except:
        pass

print("\nAvailable PyTorch model objects:")

if len(model_candidates) == 0:
    print("No PyTorch model object found.")
else:
    for name, model in model_candidates.items():
        print(f"  {name}: {type(model).__name__}")

# Automatically select the most likely trained model
if "best_model" in globals():
    best_model = best_model

elif "model" in globals() and isinstance(model, torch.nn.Module):
    best_model = model

elif len(model_candidates) == 1:
    best_model = list(model_candidates.values())[0]

else:
    raise RuntimeError(
        "Could not uniquely identify the trained model. "
        "Inspect the model variables printed above."
    )

best_model = best_model.to(device)
best_model.eval()

print("\n" + "=" * 70)
print("MODEL REFERENCE READY")
print("=" * 70)
print("Model type :", type(best_model).__name__)
print("Device     :", device)

In [ ]:
# ============================================================
# CORRECT TRAINED MODEL REFERENCE
# ============================================================

best_model = model.to(device)
best_model.eval()

print("=" * 70)
print("CORRECT MODEL REFERENCE")
print("=" * 70)

print("Model variable :", "model")
print("Model type     :", type(best_model).__name__)
print("Device         :", device)

assert isinstance(best_model, torch.nn.Module)
assert type(best_model).__name__ == "ECGResNet"

print("\n✓ ECGResNet successfully selected")
print("✓ Model is in evaluation mode")
print("✓ Ready for robustness experiments")

In [ ]:
# ============================================================
# RUNTIME RECOVERY CHECK
# ============================================================

import os
import glob
import torch

print("=" * 70)
print("COLAB RUNTIME RECOVERY CHECK")
print("=" * 70)

# Check important variables
important_vars = [
    "model",
    "best_model",
    "ECGResNet",
    "MultiLabelECGDataset",
    "test_dataset",
    "test_loader",
    "device",
    "CLASS_NAMES",
    "CLASS_NAMES"
]

print("\nCURRENT VARIABLES:")
for name in important_vars:
    print(f"{name:25s} : {'FOUND' if name in globals() else 'NOT FOUND'}")

# Search common locations for saved model checkpoints
print("\n" + "=" * 70)
print("SEARCHING FOR MODEL CHECKPOINTS")
print("=" * 70)

search_paths = [
    "/content/**/*.pth",
    "/content/**/*.pt",
    "/content/**/*.pth.tar",
    "/content/**/*.ckpt",
    "/content/drive/**/*.pth",
    "/content/drive/**/*.pt",
    "/content/drive/**/*.ckpt",
]

checkpoint_files = []

for pattern in search_paths:
    checkpoint_files.extend(
        glob.glob(pattern, recursive=True)
    )

checkpoint_files = sorted(set(checkpoint_files))

if checkpoint_files:
    print("\nFound checkpoint files:")
    for path in checkpoint_files:
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  {path}  ({size_mb:.2f} MB)")
else:
    print("\nNO CHECKPOINT FILES FOUND.")

print("\n" + "=" * 70)
print("PYTORCH VERSION")
print("=" * 70)
print(torch.__version__)

In [ ]:
import os

print("Searching for model/checkpoint files...")

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith((".pth", ".pt", ".ckpt", ".pkl")):
            print(os.path.join(root, file))

In [ ]:
import os

drive_path = "/content/drive/MyDrive"

print("=" * 70)
print("SEARCHING GOOGLE DRIVE FOR MODEL CHECKPOINTS")
print("=" * 70)

if os.path.exists(drive_path):
    found = False

    for root, dirs, files in os.walk(drive_path):
        for file in files:
            if file.endswith((".pth", ".pt", ".ckpt", ".pkl")):
                print(os.path.join(root, file))
                found = True

    if not found:
        print("❌ No model checkpoint found in Google Drive.")
else:
    print("❌ Google Drive is not mounted.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

drive_path = "/content/drive/MyDrive"

print("=" * 70)
print("SEARCHING GOOGLE DRIVE FOR MODEL CHECKPOINTS")
print("=" * 70)

found = False

for root, dirs, files in os.walk(drive_path):
    for file in files:
        if file.endswith((".pth", ".pt", ".ckpt", ".pkl")):
            print(os.path.join(root, file))
            found = True

if not found:
    print("❌ No model checkpoint found in Google Drive.")

In [ ]:
best_model = model.to(device)
best_model.eval()

print("Model type:", type(best_model).__name__)
print("Device:", device)

In [ ]:
import torch

best_model.eval()

x = torch.randn(2, 12, 1000).to(device)

with torch.no_grad():
    output = best_model(x)

print("Input shape :", x.shape)
print("Output shape:", output.shape)

In [ ]:
import os

checkpoint = "/content/best_ecg_model.pth"

print("Exists:", os.path.exists(checkpoint))

if os.path.exists(checkpoint):
    print("Found:", checkpoint)
    print("Size:", os.path.getsize(checkpoint) / (1024 * 1024), "MB")
else:
    print("❌ Checkpoint not found")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

print("Checking /content...")
for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith((".pth", ".pt", ".ckpt", ".pkl")):
            print(os.path.join(root, file))

In [ ]:
print("\nChecking Google Drive...")

drive_path = "/content/drive/MyDrive"

for root, dirs, files in os.walk(drive_path):
    for file in files:
        if file.endswith((".pth", ".pt", ".ckpt", ".pkl")):
            print(os.path.join(root, file))

In [ ]:
print("model exists:", "model" in globals())
print("best_model exists:", "best_model" in globals())

if "model" in globals():
    print("Model type:", type(model).__name__)

if "best_model" in globals():
    print("Best model type:", type(best_model).__name__)

In [ ]:
import torch
import os

save_path = "/content/drive/MyDrive/best_ecg_model.pth"

torch.save(best_model.state_dict(), save_path)

print("Saved:", save_path)
print("Exists:", os.path.exists(save_path))
print("Size:", os.path.getsize(save_path) / (1024 * 1024), "MB")

In [ ]:
import os

checkpoint = "/content/drive/MyDrive/best_ecg_model.pth"

print("Checkpoint exists:", os.path.exists(checkpoint))

if os.path.exists(checkpoint):
    print("Checkpoint size:",
          os.path.getsize(checkpoint) / (1024 * 1024), "MB")

In [ ]:
import os
import torch

save_path = "/content/drive/MyDrive/best_ecg_model.pth"

torch.save(best_model.state_dict(), save_path)

print("Saved:", os.path.exists(save_path))
print("Path:", save_path)

if os.path.exists(save_path):
    print("Size:", os.path.getsize(save_path) / (1024 * 1024), "MB")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

print(os.path.exists("/content/drive/MyDrive/best_ecg_model.pth"))

In [ ]:
import os

drive_model = "/content/drive/MyDrive/best_ecg_model.pth"

print("Google Drive mounted:", os.path.exists("/content/drive/MyDrive"))
print("Best model in Drive:", os.path.exists(drive_model))

if os.path.exists(drive_model):
    print("Model found!")
    print("Size:", round(os.path.getsize(drive_model) / (1024 * 1024), 2), "MB")

In [ ]:
import torch
import os

checkpoint_path = "/content/drive/MyDrive/best_ecg_model.pth"

print("Checkpoint exists:", os.path.exists(checkpoint_path))

In [ ]:
test_model = ECGResNet(
    num_classes=5
)

test_model.load_state_dict(
    torch.load(checkpoint_path, map_location=device)
)

test_model = test_model.to(device)
test_model.eval()

print("Best model loaded successfully!")

In [ ]:
import shutil
import os

source = "/content/best_ecg_model.pth"
destination = "/content/drive/MyDrive/best_ecg_model.pth"

shutil.copy(source, destination)

print("Saved to Google Drive:", os.path.exists(destination))
print("Path:", destination)

In [ ]:
import os
import torch

save_path = "/content/drive/MyDrive/best_ecg_model.pth"

torch.save(best_model.state_dict(), save_path)

print("Saved:", os.path.exists(save_path))

if os.path.exists(save_path):
    print("Path:", save_path)
    print("Size:", round(os.path.getsize(save_path) / (1024 * 1024), 2), "MB")

In [ ]:
import os

print("Current folder:")
print(os.path.exists("best_ecg_model.pth"))

print("\nGoogle Drive:")
print(os.path.exists("/content/drive/MyDrive/best_ecg_model.pth"))

In [ ]:
import os
import torch

checkpoint_path = "/content/drive/MyDrive/best_ecg_model.pth"

print("Checkpoint exists:", os.path.exists(checkpoint_path))

# Create model using the SAME architecture
test_model = ECGResNet(
    num_classes=5
).to(device)

# Load trained weights
state_dict = torch.load(
    checkpoint_path,
    map_location=device
)

test_model.load_state_dict(state_dict)

# Evaluation mode
test_model.eval()

print("✅ Model loaded successfully!")
print("Model type:", type(test_model).__name__)

In [ ]:
import inspect

print(inspect.signature(ECGResNet))

In [ ]:
print("Variables related to test/validation:")

for name in globals():
    if "test" in name.lower() or "val" in name.lower():
        print(name)

In [ ]:
print("Test dataset size:", len(test_dataset))
print("Test loader batches:", len(test_loader))

# Check one batch
x, y = next(iter(test_loader))

print("Input shape:", x.shape)
print("Labels shape:", y.shape)
print("Labels:", y[:10])

In [ ]:
batch = next(iter(test_loader))

print("Batch type:", type(batch))

if isinstance(batch, (list, tuple)):
    print("Number of items:", len(batch))

    for i, item in enumerate(batch):
        print(
            f"Item {i}: "
            f"type={type(item)}, "
            f"shape={getattr(item, 'shape', 'N/A')}"
        )
else:
    print("Batch shape:", getattr(batch, "shape", "N/A"))

In [ ]:
batch = next(iter(test_loader))

print("Batch type:", type(batch))

if isinstance(batch, dict):
    print("\nBatch keys:")
    for key in batch.keys():
        value = batch[key]
        print(
            f"  {key}: "
            f"type={type(value)}, "
            f"shape={getattr(value, 'shape', 'N/A')}"
        )
else:
    print("Batch contents:")
    print(batch)

In [ ]:
# Get one test batch
batch = next(iter(test_loader))

# Extract ECG and labels
x = batch["ecg"]
y = batch["labels"]

# Move to the same device as the model
x = x.to(device)
y = y.to(device)

print("ECG input shape :", x.shape)
print("Labels shape    :", y.shape)
print("Labels          :", y)

# Run one forward pass
with torch.no_grad():
    outputs = test_model(x)

print("\nModel output shape:", outputs.shape)
print("Predictions:", torch.argmax(outputs, dim=1))

In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

test_model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in test_loader:

        # Get ECG and labels
        x = batch["ecg"].to(device)
        y = batch["labels"].to(device)

        # Prediction
        outputs = test_model(x)
        predictions = torch.argmax(outputs, dim=1)

        # Store results
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Convert to numpy
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

print("=" * 60)
print("FINAL TEST SET EVALUATION")
print("=" * 60)

print("\nNumber of test samples:", len(all_labels))

# Accuracy
accuracy = accuracy_score(all_labels, all_predictions)

print("\nAccuracy:", round(accuracy, 4))

# Precision
precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

# Recall
recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

# F1
f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

print("Weighted Precision:", round(precision, 4))
print("Weighted Recall   :", round(recall, 4))
print("Weighted F1       :", round(f1, 4))

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        all_labels,
        all_predictions,
        digits=4,
        zero_division=0
    )
)

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(all_labels, all_predictions)

print(cm)

print("\n" + "=" * 60)
print("PREDICTED CLASS DISTRIBUTION")
print("=" * 60)

unique, counts = np.unique(all_predictions, return_counts=True)

for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count}")

In [ ]:
batch = next(iter(test_loader))

print("Batch keys:", batch.keys())

print("\nECG:")
print(type(batch["ecg"]))
print(batch["ecg"].shape)

print("\nLabels:")
print(type(batch["labels"]))
print(batch["labels"])
print(batch["labels"].shape)

print("\nECG IDs:")
print(type(batch["ecg_id"]))
print(batch["ecg_id"])

In [ ]:
print("Dataset length:", len(test_dataset))

for i in range(min(10, len(test_dataset))):
    item = test_dataset[i]

    print("\nITEM", i)
    print("Type:", type(item))

    if isinstance(item, dict):
        print("Keys:", item.keys())

        for key, value in item.items():
            if hasattr(value, "shape"):
                print(key, "shape:", value.shape)
            else:
                print(key, ":", value)

In [ ]:
def collate_fn(batch):

    ecgs = []
    labels = []
    ecg_ids = []

    for item in batch:

        ecg = item["ecg"]
        label_list = item["labels"]
        ecg_id = item["ecg_id"]

        # Skip samples without labels
        if not label_list:
            continue

        label_name = label_list[0]

        if label_name not in CLASS_TO_ID:
            continue

        label_id = CLASS_TO_ID[label_name]

        ecgs.append(ecg)
        labels.append(label_id)
        ecg_ids.append(ecg_id)

    return {
        "ecg": torch.stack(ecgs),
        "labels": torch.tensor(labels, dtype=torch.long),
        "ecg_id": ecg_ids
    }

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader

In [ ]:
import torch
from torch.utils.data import DataLoader

print("DataLoader imported successfully")

In [ ]:
CLASS_TO_ID = {
    "NORM": 0,
    "MI": 1,
    "STTC": 2,
    "CD": 3,
    "HYP": 4
}

print(CLASS_TO_ID)

In [ ]:
from torch.utils.data import DataLoader

print("DataLoader imported successfully")

In [ ]:
print("test_dataset exists:", "test_dataset" in globals())

if "test_dataset" in globals():
    print("Test dataset length:", len(test_dataset))
    print("First sample:", test_dataset[0])

In [ ]:
print("Checking dataset variables...")

print("test_dataset exists:", "test_dataset" in globals())
print("train_dataset exists:", "train_dataset" in globals())

if "test_dataset" in globals():
    print("Test dataset length:", len(test_dataset))

In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

print("Test dataset size:", len(test_dataset))
print("Test loader batches:", len(test_loader))

In [ ]:
print("Variables currently available:")
for name in sorted(globals()):
    if any(word in name.lower() for word in ["data", "dataset", "ecg", "test", "train"]):
        print(name)

In [ ]:
print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

x = train_dataset[0]

print("\nFirst sample:")
print("ECG shape:", x["ecg"].shape)
print("Labels:", x["labels"])
print("ECG ID:", x["ecg_id"])

In [ ]:
# Check DataLoader
batch = next(iter(train_loader))

print("Batch loaded successfully!")

print("ECG batch shape:", batch["ecg"].shape)
print("Labels:", batch["labels"][:5])
print("ECG IDs:", batch["ecg_id"][:5])

In [ ]:
# Find ECGs with empty diagnostic labels

bad_samples = []

for i in range(len(train_dataset)):
    sample = train_dataset[i]

    if not sample["labels"]:
        bad_samples.append((i, sample["ecg_id"], sample["labels"]))

print("Number of samples with empty labels:", len(bad_samples))

if bad_samples:
    print("\nFirst few problematic samples:")
    for item in bad_samples[:20]:
        print(item)
else:
    print("✅ No empty labels found in train_dataset")

In [ ]:
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

print("\nNumber of classes:", len(CLASS_TO_ID))
print("Class mapping:")
print(CLASS_TO_ID)

In [ ]:
# Check all training labels

all_labels = []

for batch in train_loader:
    all_labels.extend(batch["labels"].tolist())

print("Total labels checked:", len(all_labels))
print("Unique label IDs:", sorted(set(all_labels)))

In [ ]:
import torch
import torch.nn as nn


class ECG1DCNN(nn.Module):

    def __init__(self, num_classes=5):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv1d(12, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            # Block 2
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            # Block 3
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            # Block 4
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            # Convert variable-length features to fixed-size vector
            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = ECG1DCNN(num_classes=5)

print(model)

In [ ]:
# Get one batch
batch = next(iter(train_loader))

ecg = batch["ecg"]

print("Input shape:", ecg.shape)

# Forward pass
with torch.no_grad():
    output = model(ecg)

print("Output shape:", output.shape)

In [ ]:
# ============================================================
# STEP 15 - TRAINING SETUP
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Move model to device
model = model.to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Model moved to:", device)

In [ ]:
# ============================================================
# STEP 16 - TRAINING FUNCTION
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:

        # Get ECG and labels
        ecg = batch["ecg"].to(device)
        labels = batch["labels"].to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(ecg)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        total_loss += loss.item()

        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / len(loader)
    accuracy = correct / total

    return average_loss, accuracy

In [ ]:
# ============================================================
# STEP 17 - VALIDATION FUNCTION
# ============================================================

def validate(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch in loader:

            ecg = batch["ecg"].to(device)
            labels = batch["labels"].to(device)

            # Forward pass
            outputs = model(ecg)

            # Loss
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            # Predictions
            predictions = torch.argmax(outputs, dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    average_loss = total_loss / len(loader)
    accuracy = correct / total

    return average_loss, accuracy

In [ ]:
# ============================================================
# STEP 15 - CLEAN EMPTY LABELS FROM ALL SPLITS
# ============================================================

def has_valid_labels(labels):
    return isinstance(labels, list) and len(labels) > 0


# Clean TRAIN
before_train = len(train_df)

train_df = train_df[
    train_df["diagnostic_classes"].apply(has_valid_labels)
].reset_index(drop=True)

print("TRAIN")
print("Before :", before_train)
print("After  :", len(train_df))
print("Removed:", before_train - len(train_df))


# Clean VALIDATION
before_val = len(val_df)

val_df = val_df[
    val_df["diagnostic_classes"].apply(has_valid_labels)
].reset_index(drop=True)

print("\nVALIDATION")
print("Before :", before_val)
print("After  :", len(val_df))
print("Removed:", before_val - len(val_df))


# Clean TEST
before_test = len(test_df)

test_df = test_df[
    test_df["diagnostic_classes"].apply(has_valid_labels)
].reset_index(drop=True)

print("\nTEST")
print("Before :", before_test)
print("After  :", len(test_df))
print("Removed:", before_test - len(test_df))

In [ ]:
# ============================================================
# STEP 16 - RECREATE DATASETS AFTER CLEANING
# ============================================================

train_dataset = ECGDataset(train_df)
val_dataset = ECGDataset(val_df)
test_dataset = ECGDataset(test_df)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
# ============================================================
# STEP 17 - RECREATE DATALOADERS
# ============================================================

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("DataLoaders recreated successfully!")

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# ============================================================
# STEP 18 - CHECK VALIDATION LOADER
# ============================================================

batch = next(iter(val_loader))

print("Validation batch loaded successfully!")
print("ECG shape:", batch["ecg"].shape)
print("Labels:", batch["labels"][:10])
print("ECG IDs:", batch["ecg_id"][:10])

In [ ]:
# ============================================================
# STEP 19 - TEST VALIDATION FORWARD PASS
# ============================================================

model.eval()

batch = next(iter(val_loader))

ecg = batch["ecg"].to(device)

with torch.no_grad():
    output = model(ecg)

print("Validation input shape :", ecg.shape)
print("Validation output shape:", output.shape)

In [ ]:
# ============================================================
# STEP 20 - TRAINING SETUP
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Move model to GPU/CPU
model = model.to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print("Model ready for training!")
print("Loss:", criterion)
print("Optimizer: Adam")

In [ ]:
# ============================================
# STEP 20 - CHECK ECG SHAPES
# ============================================

from collections import Counter

train_shapes = []

for i in range(len(train_dataset)):
    sample = train_dataset[i]
    train_shapes.append(tuple(sample["ecg"].shape))

print("Unique ECG shapes in training data:")
print(Counter(train_shapes))

In [ ]:
def collate_fn(batch):

    ecgs = []
    labels = []
    ecg_ids = []

    for sample in batch:

        ecg = sample["ecg"]

        # Make sure ECG is [12, length]
        if ecg.dim() == 1:
            ecg = ecg.unsqueeze(0)

        # Force exactly 1000 time points
        if ecg.shape[-1] > 1000:
            ecg = ecg[:, :1000]

        elif ecg.shape[-1] < 1000:
            padding = 1000 - ecg.shape[-1]
            ecg = torch.nn.functional.pad(ecg, (0, padding))

        ecgs.append(ecg)

        # Get first diagnostic label
        label_list = sample["labels"]

        if isinstance(label_list, (list, tuple)) and len(label_list) > 0:
            label_name = label_list[0]
        else:
            label_name = "NORM"

        label_id = CLASS_TO_ID[label_name]

        labels.append(label_id)
        ecg_ids.append(sample["ecg_id"])

    return {
        "ecg": torch.stack(ecgs),
        "labels": torch.tensor(labels, dtype=torch.long),
        "ecg_id": torch.tensor(ecg_ids, dtype=torch.long)
    }

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

In [ ]:
print("Testing training loader...")

for batch in train_loader:
    print("ECG shape:", batch["ecg"].shape)
    print("Labels shape:", batch["labels"].shape)
    break

print("Training DataLoader works!")

In [ ]:
# ============================================
# STEP 21 - TRAIN THE ECG MODEL
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim

# Make sure model is on the correct device
model = model.to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

# Number of epochs
EPOCHS = 20

best_val_accuracy = 0.0

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

for epoch in range(EPOCHS):

    # -------------------------
    # TRAINING
    # -------------------------
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch in train_loader:

        ecg = batch["ecg"].to(device)
        labels = batch["labels"].to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        output = model(ecg)

        # Calculate loss
        loss = criterion(output, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        train_loss += loss.item()

        predictions = torch.argmax(output, dim=1)

        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= len(train_loader)
    train_accuracy = train_correct / train_total


    # -------------------------
    # VALIDATION
    # -------------------------
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch in val_loader:

            ecg = batch["ecg"].to(device)
            labels = batch["labels"].to(device)

            output = model(ecg)

            loss = criterion(output, labels)

            val_loss += loss.item()

            predictions = torch.argmax(output, dim=1)

            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= len(val_loader)
    val_accuracy = val_correct / val_total


    # -------------------------
    # SAVE BEST MODEL
    # -------------------------

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "best_ecg_model.pth"
        )

        best_marker = " ⭐ BEST"

    else:
        best_marker = ""


    # -------------------------
    # PRINT RESULTS
    # -------------------------

    print(
        f"Epoch [{epoch + 1:02d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
        f"{best_marker}"
    )


print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print(f"Best Validation Accuracy: {best_val_accuracy:.4f}")

In [ ]:
# ============================================
# STEP 22 - TRAINING HISTORY
# ============================================

import matplotlib.pyplot as plt

# Store training history
# If these lists were not created during training,
# we will recreate them from the values above.

train_losses = [
    0.8480, 0.7814, 0.7307, 0.6004, 0.6185,
    0.5440, 0.4576, 0.4671, 0.4184, 0.3950,
    0.4462, 0.4249, 0.3708, 0.2584, 0.2353,
    0.3139, 0.3508, 0.4213, 0.2456, 0.2288
]

val_losses = [
    1.0298, 0.9754, 0.9060, 0.9390, 0.9066,
    1.3637, 0.8453, 1.0039, 1.1069, 1.0866,
    0.9365, 1.0468, 1.2987, 1.0794, 1.9319,
    1.1436, 1.4762, 0.8940, 0.9488, 1.4085
]

train_accs = [
    0.6990, 0.6926, 0.7249, 0.7443, 0.7799,
    0.7864, 0.8285, 0.8382, 0.8576, 0.8447,
    0.8479, 0.8544, 0.8770, 0.9223, 0.9223,
    0.9126, 0.8576, 0.8770, 0.8964, 0.9223
]

val_accs = [
    0.6250, 0.7000, 0.6750, 0.6500, 0.6875,
    0.4625, 0.6375, 0.5750, 0.7125, 0.6500,
    0.6500, 0.6875, 0.6375, 0.6750, 0.6625,
    0.6125, 0.6375, 0.7125, 0.7250, 0.6625
]

epochs = range(1, 21)

# -----------------------------
# Loss curve
# -----------------------------

plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    train_losses,
    marker="o",
    label="Training Loss"
)

plt.plot(
    epochs,
    val_losses,
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")

plt.legend()
plt.grid(True)

plt.show()


# -----------------------------
# Accuracy curve
# -----------------------------

plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    train_accs,
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    epochs,
    val_accs,
    marker="o",
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# STEP 20 - FINAL TEST EVALUATION
# ============================================================

import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import numpy as np

print("=" * 60)
print("FINAL TEST EVALUATION")
print("=" * 60)

# Use the best model if it was saved during training
if "best_model_state" in globals():
    model.load_state_dict(best_model_state)
    print("Loaded best validation model.")
else:
    print("Best model checkpoint not found.")
    print("Using the current trained model.")

model.eval()

all_predictions = []
all_true_labels = []

with torch.no_grad():

    for batch in test_loader:

        ecg = batch["ecg"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(ecg)

        # Predicted class
        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_true_labels.extend(labels.cpu().numpy())


# Convert to NumPy arrays
all_predictions = np.array(all_predictions)
all_true_labels = np.array(all_true_labels)

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    all_true_labels,
    all_predictions
)

print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# ------------------------------------------------------------
# Class names
# ------------------------------------------------------------

ID_TO_CLASS = {
    value: key
    for key, value in CLASS_TO_ID.items()
}

class_names = [
    ID_TO_CLASS[i]
    for i in range(len(ID_TO_CLASS))
]

print("\nClass names:")
print(class_names)

# ------------------------------------------------------------
# Classification Report
# ------------------------------------------------------------

print("\nClassification Report:")
print(
    classification_report(
        all_true_labels,
        all_predictions,
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    all_true_labels,
    all_predictions,
    labels=list(range(len(class_names)))
)

print("Confusion Matrix:")
print(cm)

print("\n" + "=" * 60)
print("TEST EVALUATION COMPLETED")
print("=" * 60)

In [ ]:
# ============================================
# STEP 1: Imports + Google Drive
# ============================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

In [ ]:
# ============================================
# STEP 2: Find model checkpoints
# ============================================

import os

drive_root = "/content/drive/MyDrive"

checkpoint_files = []

for root, dirs, files in os.walk(drive_root):
    for file in files:
        if file.endswith((".pth", ".pt", ".ckpt")):
            full_path = os.path.join(root, file)
            checkpoint_files.append(full_path)

print("Found checkpoints:", len(checkpoint_files))

for path in checkpoint_files:
    print(path)

In [ ]:
# ============================================
# STEP 3: Set up PyTorch device
# ============================================

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

In [12]:
# ============================================
# STEP 4: Define ECGResNet architecture
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class ECGResNet(nn.Module):

    def __init__(self, num_classes=5):
        super(ECGResNet, self).__init__()

        self.conv1 = nn.Conv1d(
            in_channels=1,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3
        )
        self.bn1 = nn.BatchNorm1d(64)

        self.conv2 = nn.Conv1d(
            in_channels=64,
            out_channels=128,
            kernel_size=5,
            stride=2,
            padding=2
        )
        self.bn2 = nn.BatchNorm1d(128)

        self.conv3 = nn.Conv1d(
            in_channels=128,
            out_channels=256,
            kernel_size=5,
            stride=2,
            padding=2
        )
        self.bn3 = nn.BatchNorm1d(256)

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)

        x = self.pool(x)

        x = x.squeeze(-1)

        x = self.classifier(x)

        return x


print("ECGResNet defined successfully!")

ECGResNet defined successfully!


In [ ]:
# ============================================
# STEP 5: Inspect checkpoint
# ============================================

import torch

checkpoint_path = "/content/drive/MyDrive/best_ecg_model.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint.keys():
        print(" -", key)

    # If this is a raw state_dict
    if all(isinstance(v, torch.Tensor) for v in checkpoint.values()):
        print("\nThis appears to be a raw state_dict.")
        print("Number of parameters:", len(checkpoint))

        print("\nFirst 15 parameter names:")
        for i, key in enumerate(checkpoint.keys()):
            if i >= 15:
                break
            print(key, checkpoint[key].shape)

In [ ]:
# ============================================================
# STEP 4: Find ECG dataset files
# ============================================================

import os

drive_root = "/content/drive/MyDrive"

print("Searching for ECG dataset files...\n")

# Search common ECG-related files/folders
matches = []

for root, dirs, files in os.walk(drive_root):
    # Avoid unnecessary hidden/system folders
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for file in files:
        lower = file.lower()

        if any(keyword in lower for keyword in [
            "ecg",
            "ptb",
            "physionet",
            "mit",
            "dataset",
            "test",
            "train"
        ]):
            matches.append(os.path.join(root, file))

print(f"Found {len(matches)} possible dataset files:\n")

for path in matches[:100]:
    print(path)

if len(matches) > 100:
    print(f"\n...and {len(matches) - 100} more files")

In [ ]:
# ============================================================
# STEP 5: Inspect Google Drive structure
# ============================================================

import os

drive_root = "/content/drive/MyDrive"

print("=== TOP-LEVEL ITEMS IN MYDRIVE ===\n")

for item in os.listdir(drive_root):
    path = os.path.join(drive_root, item)

    if os.path.isdir(path):
        print("[FOLDER] ", item)
    else:
        print("[FILE]   ", item)

print("\n=== POSSIBLE DATASET / ARCHIVE FILES ===\n")

extensions = (
    ".zip",
    ".rar",
    ".7z",
    ".tar",
    ".gz",
    ".csv",
    ".npy",
    ".npz",
    ".pt",
    ".pth",
    ".pkl",
    ".pickle",
    ".mat",
    ".hea",
    ".dat",
    ".edf"
)

found = []

for root, dirs, files in os.walk(drive_root):
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for file in files:
        if file.lower().endswith(extensions):
            found.append(os.path.join(root, file))

for path in found:
    print(path)

print(f"\nTotal candidate files: {len(found)}")

In [ ]:
# ============================================================
# STEP 6: Inspect checkpoint contents
# ============================================================

import torch
import os

checkpoint_path = "/content/drive/MyDrive/best_ecg_model.pth"

print("Loading checkpoint...")
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

print("\nCheckpoint type:")
print(type(checkpoint))

print("\nCheckpoint information:")

if isinstance(checkpoint, dict):
    print("Number of keys:", len(checkpoint))

    for key in checkpoint.keys():
        print(" -", key)
else:
    print("Checkpoint is not a dictionary.")

print("\nModel checkpoint successfully inspected.")

In [ ]:
# ============================================================
# STEP 4: Load the trained ECG model
# ============================================================

import torch
import os

# Path to trained model
MODEL_PATH = "/content/drive/MyDrive/best_ecg_model.pth"

# Number of ECG classes
NUM_CLASSES = 5

# Class names
CLASS_NAMES = ["NORM", "STTC", "CD", "MI", "HYP"]

print("Loading ECG model...")
print("Model path:", MODEL_PATH)
print("Classes:", CLASS_NAMES)

# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

model = ECGRsNet(num_classes=NUM_CLASSES)

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

# The checkpoint is a raw state_dict
if isinstance(checkpoint, dict):
    state_dict = checkpoint
else:
    raise ValueError("Unexpected checkpoint format")

# Remove possible 'module.' prefix
clean_state_dict = {}

for key, value in state_dict.items():
    new_key = key.replace("module.", "")
    clean_state_dict[new_key] = value

# ------------------------------------------------------------
# Load weights
# ------------------------------------------------------------

missing_keys, unexpected_keys = model.load_state_dict(
    clean_state_dict,
    strict=False
)

# Move model to device
model = model.to(device)

# Evaluation mode
model.eval()

print("\n==============================")
print("MODEL LOADED SUCCESSFULLY")
print("==============================")

print("Device:", device)
print("Missing keys:", len(missing_keys))
print("Unexpected keys:", len(unexpected_keys))

if missing_keys:
    print("\nMissing keys:")
    for key in missing_keys:
        print(" -", key)

if unexpected_keys:
    print("\nUnexpected keys:")
    for key in unexpected_keys:
        print(" -", key)

print("\nModel is ready for inference.")

In [13]:
# ============================================================
# STEP 4: Define ECGResNet
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class ECGResNet(nn.Module):

    def __init__(self, num_classes):
        ...
        # actual layers

    def forward(self, x):
        ...
        # actual forward pass

print("ECGResNet defined successfully!")

ECGResNet defined successfully!


In [14]:
# ============================================
# STEP 4: Define ECGResNet architecture
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class ECGResNet(nn.Module):

    def __init__(self, num_classes=5):
        super(ECGResNet, self).__init__()

        self.conv1 = nn.Conv1d(
            in_channels=1,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3
        )
        self.bn1 = nn.BatchNorm1d(64)

        self.conv2 = nn.Conv1d(
            in_channels=64,
            out_channels=128,
            kernel_size=5,
            stride=2,
            padding=2
        )
        self.bn2 = nn.BatchNorm1d(128)

        self.conv3 = nn.Conv1d(
            in_channels=128,
            out_channels=256,
            kernel_size=5,
            stride=2,
            padding=2
        )
        self.bn3 = nn.BatchNorm1d(256)

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)

        x = self.pool(x)

        x = x.squeeze(-1)

        x = self.classifier(x)

        return x


print("ECGResNet defined successfully!")

ECGResNet defined successfully!


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# STEP 5: Reconstruct ECGResNet from checkpoint
# ============================================================

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=7,
            stride=stride,
            padding=3,
            bias=False
        )
        self.bn1 = nn.BatchNorm1d(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=7,
            stride=1,
            padding=3,
            bias=False
        )
        self.bn2 = nn.BatchNorm1d(out_channels)

        # Residual shortcut
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Sequential()

    def forward(self, x):
        identity = self.shortcut(x)

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)

        x = x + identity
        x = F.relu(x)

        return x


class ECGResNet(nn.Module):

    def __init__(self, num_classes=5):
        super().__init__()

        # Input: 12-lead ECG
        self.stem = nn.Sequential(
            nn.Conv1d(
                12,
                64,
                kernel_size=7,
                stride=1,
                padding=3,
                bias=False
            ),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        # Residual blocks
        self.block1 = ResidualBlock(64, 64)

        self.block2 = ResidualBlock(
            64,
            128,
            stride=2
        )

        self.block3 = ResidualBlock(
            128,
            256,
            stride=2
        )

        self.block4 = ResidualBlock(
            256,
            512,
            stride=2
        )

        # Global average pooling
        self.pool = nn.AdaptiveAvgPool1d(1)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = self.pool(x)

        x = x.squeeze(-1)

        x = self.classifier(x)

        return x


print("ECGResNet reconstructed successfully!")

ECGResNet reconstructed successfully!


In [16]:
# ============================================================
# STEP 5: Load the trained ECGResNet checkpoint
# ============================================================

import torch
import os

MODEL_PATH = "/content/drive/MyDrive/best_ecg_model.pth"

CLASSES = ['NORM', 'STTC', 'CD', 'MI', 'HYP']
NUM_CLASSES = len(CLASSES)

print("Loading trained ECG model...")
print("Model path:", MODEL_PATH)
print("Classes:", CLASSES)

# Create the model
model = ECGResNet(num_classes=NUM_CLASSES)

# Load checkpoint
checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu"
)

# The checkpoint is a raw state_dict
model.load_state_dict(checkpoint, strict=True)

# Evaluation mode
model.eval()

print("\n✅ Model loaded successfully!")
print("Architecture:", type(model).__name__)
print("Number of classes:", NUM_CLASSES)
print("Classes:", CLASSES)

Loading trained ECG model...
Model path: /content/drive/MyDrive/best_ecg_model.pth
Classes: ['NORM', 'STTC', 'CD', 'MI', 'HYP']

✅ Model loaded successfully!
Architecture: ECGResNet
Number of classes: 5
Classes: ['NORM', 'STTC', 'CD', 'MI', 'HYP']


In [17]:
# ============================================================
# STEP 6: Test model with dummy 12-lead ECG
# ============================================================

import torch

# Dummy ECG:
# batch = 1
# leads = 12
# signal length = 1000
dummy_ecg = torch.randn(1, 12, 1000)

print("Input shape:", dummy_ecg.shape)

with torch.no_grad():
    output = model(dummy_ecg)

print("Output shape:", output.shape)
print("Output:", output)

Input shape: torch.Size([1, 12, 1000])
Output shape: torch.Size([1, 5])
Output: tensor([[-0.0207, -0.0317, -0.0757, -0.0230, -0.0535]])


In [18]:
# ============================================================
# STEP 7: Convert model output into probabilities
# ============================================================

import torch

with torch.no_grad():
    logits = model(dummy_ecg)

# Convert logits to probabilities
probabilities = torch.softmax(logits, dim=1)

# Get the predicted class
predicted_index = torch.argmax(probabilities, dim=1).item()
predicted_class = CLASSES[predicted_index]

print("Prediction probabilities:\n")

for i, class_name in enumerate(CLASSES):
    print(f"{class_name}: {probabilities[0][i].item() * 100:.2f}%")

print("\nPredicted class:", predicted_class)

Prediction probabilities:

NORM: 20.40%
STTC: 20.18%
CD: 19.31%
MI: 20.36%
HYP: 19.75%

Predicted class: NORM


In [19]:
# ============================================================
# STEP 8: Search Drive for ECG training/preprocessing files
# ============================================================

import os

DRIVE_ROOT = "/content/drive/MyDrive"

keywords = [
    "ecg",
    "ptb",
    "train",
    "dataset",
    "preprocess",
    "normaliz",
    "label"
]

matches = []

print("Searching Drive for relevant ECG/training files...\n")

for root, dirs, files in os.walk(DRIVE_ROOT):
    for file in files:
        name = file.lower()

        if any(keyword in name for keyword in keywords):
            matches.append(os.path.join(root, file))

print(f"Found {len(matches)} relevant files:\n")

for path in matches[:200]:
    print(path)

if len(matches) > 200:
    print(f"\n...and {len(matches) - 200} more files")

Searching Drive for relevant ECG/training files...

Found 1 relevant files:

/content/drive/MyDrive/best_ecg_model.pth


In [20]:
import torch

MODEL_PATH = "/content/drive/MyDrive/best_ecg_model.pth"

checkpoint = torch.load(MODEL_PATH, map_location="cpu")

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint.keys():
        print(" -", key)

    # Find the actual state dictionary
    if "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    print("\nModel weight keys:")
    for key, value in state_dict.items():
        print(key, "->", tuple(value.shape))
else:
    print("Checkpoint is not a dictionary.")

Checkpoint type: <class 'collections.OrderedDict'>

Checkpoint keys:
 - stem.0.weight
 - stem.1.weight
 - stem.1.bias
 - stem.1.running_mean
 - stem.1.running_var
 - stem.1.num_batches_tracked
 - block1.conv1.weight
 - block1.bn1.weight
 - block1.bn1.bias
 - block1.bn1.running_mean
 - block1.bn1.running_var
 - block1.bn1.num_batches_tracked
 - block1.conv2.weight
 - block1.bn2.weight
 - block1.bn2.bias
 - block1.bn2.running_mean
 - block1.bn2.running_var
 - block1.bn2.num_batches_tracked
 - block2.conv1.weight
 - block2.bn1.weight
 - block2.bn1.bias
 - block2.bn1.running_mean
 - block2.bn1.running_var
 - block2.bn1.num_batches_tracked
 - block2.conv2.weight
 - block2.bn2.weight
 - block2.bn2.bias
 - block2.bn2.running_mean
 - block2.bn2.running_var
 - block2.bn2.num_batches_tracked
 - block2.shortcut.0.weight
 - block2.shortcut.1.weight
 - block2.shortcut.1.bias
 - block2.shortcut.1.running_mean
 - block2.shortcut.1.running_var
 - block2.shortcut.1.num_batches_tracked
 - block3.conv1.w

In [ ]:
print("\n" + "=" * 60)
print("ALL MODEL LAYERS")
print("=" * 60)

for key, value in state_dict.items():
    if hasattr(value, "shape"):
        print(f"{key:45s} {tuple(value.shape)}")

In [21]:
# ============================================
# STEP 5: Load the Trained ECGResNet Model
# ============================================

import torch

# Path to your trained model
MODEL_PATH = "/content/drive/MyDrive/best_ecg_model.pth"

# Create the model
model = ECGResNet(num_classes=5)

# Load trained weights
checkpoint = torch.load(MODEL_PATH, map_location="cpu")

# Check what type of checkpoint we have
print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    print(checkpoint.keys())

# Load weights
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    model.load_state_dict(checkpoint["state_dict"])
elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

# Evaluation mode
model.eval()

print("\n✅ Trained ECGResNet loaded successfully!")

Checkpoint type: <class 'collections.OrderedDict'>

Checkpoint keys:
odict_keys(['stem.0.weight', 'stem.1.weight', 'stem.1.bias', 'stem.1.running_mean', 'stem.1.running_var', 'stem.1.num_batches_tracked', 'block1.conv1.weight', 'block1.bn1.weight', 'block1.bn1.bias', 'block1.bn1.running_mean', 'block1.bn1.running_var', 'block1.bn1.num_batches_tracked', 'block1.conv2.weight', 'block1.bn2.weight', 'block1.bn2.bias', 'block1.bn2.running_mean', 'block1.bn2.running_var', 'block1.bn2.num_batches_tracked', 'block2.conv1.weight', 'block2.bn1.weight', 'block2.bn1.bias', 'block2.bn1.running_mean', 'block2.bn1.running_var', 'block2.bn1.num_batches_tracked', 'block2.conv2.weight', 'block2.bn2.weight', 'block2.bn2.bias', 'block2.bn2.running_mean', 'block2.bn2.running_var', 'block2.bn2.num_batches_tracked', 'block2.shortcut.0.weight', 'block2.shortcut.1.weight', 'block2.shortcut.1.bias', 'block2.shortcut.1.running_mean', 'block2.shortcut.1.running_var', 'block2.shortcut.1.num_batches_tracked', 'bloc

In [22]:
import os

print("Google Drive mounted:", os.path.exists("/content/drive"))
print("MyDrive exists:", os.path.exists("/content/drive/MyDrive"))

print("\nFiles in MyDrive:")
if os.path.exists("/content/drive/MyDrive"):
    for f in os.listdir("/content/drive/MyDrive"):
        print(f)

Google Drive mounted: True
MyDrive exists: True

Files in MyDrive:
Colab Notebooks
Sac internship
Saved from the Google app
Google AI Studio
Screenshot 2025-10-29 170148.png
Screenshots
hti
Copyright_1.pdf
field_report_odd_sem (1).pdf
CVS_Startup_Project_Presentation.pdf
2nd YEAR END EXAM SUMMER TERM PAPER.gdoc
IMG_1124.png
vitar.project
SURYA RAJENDRA (2).resume1111.pdf
SURYA RAJENDRA (1).resume1111.pdf
SURYA RAJENDRA.resume1111.pdf
SURYA_RAJENDRA_FlowCV_Resume_2026-05-14.pdf
Surya_Resume___ (4).pdf
Surya_Resume___ (3).pdf
Surya_Resume___ (2).pdf
Surya_Resume___ (1).pdf
Surya_Resume___.pdf
Surya_Resume (3).pdf
Surya_Resume (2).pdf
Surya_Resume (1).pdf
Surya-resume
Surya.resume
Surya_Resume.pdf
best_ecg_model.pth
Rag.concept
Surya_Resume_research.pdf
Surya_Resume_compressed.pdf


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

MODEL_PATH = "/content/drive/MyDrive/best_ecg_model.pth"

print("Model exists:", os.path.exists(MODEL_PATH))
print("Model path:", MODEL_PATH)

In [ ]:
# ============================================
# STEP 6: Test Trained Model
# ============================================

import torch

# Class names
CLASSES = ["NORM", "STIC", "CD", "MI", "HYP"]

# Create a dummy 12-lead ECG
# Shape = [batch, leads, signal_length]
dummy_ecg = torch.randn(1, 12, 1000)

# Run trained model
model.eval()

with torch.no_grad():
    logits = model(dummy_ecg)
    probabilities = torch.softmax(logits, dim=1)

# Get predicted class
predicted_index = torch.argmax(probabilities, dim=1).item()
predicted_class = CLASSES[predicted_index]
confidence = probabilities[0][predicted_index].item() * 100

# Display results
print("Input shape:", dummy_ecg.shape)
print("Output shape:", logits.shape)

print("\nPrediction probabilities:")
for i, class_name in enumerate(CLASSES):
    print(f"{class_name}: {probabilities[0][i].item() * 100:.2f}%")

print("\nPredicted class:", predicted_class)
print(f"Confidence: {confidence:.2f}%")

In [23]:
# ============================================
# STEP 7: Search Google Drive for ECG datasets
# ============================================

import os

DRIVE_ROOT = "/content/drive/MyDrive"

keywords = [
    "ecg",
    "ptb",
    "physionet",
    "mit",
    "arrhythmia",
    "heartbeat",
    "cardio",
    "signal",
    "record"
]

matches = []

print("Searching Google Drive for ECG/data files...\n")

for root, dirs, files in os.walk(DRIVE_ROOT):
    for file in files:
        name = file.lower()

        if any(keyword in name for keyword in keywords):
            matches.append(os.path.join(root, file))

print(f"Found {len(matches)} possible ECG/data files:\n")

for path in matches[:100]:
    print(path)

if len(matches) > 100:
    print(f"\n...and {len(matches) - 100} more files.")

Searching Google Drive for ECG/data files...

Found 1 possible ECG/data files:

/content/drive/MyDrive/best_ecg_model.pth


In [24]:
# ============================================
# STEP 8: Download PTB-XL metadata
# ============================================

import pandas as pd
import os

PTB_URL = "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"

metadata_path = "/content/ptbxl_database.csv"

print("Downloading PTB-XL metadata...")

df = pd.read_csv(PTB_URL)

print("\n✅ Metadata loaded!")
print("Number of ECG records:", len(df))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())


✅ Metadata loaded!
Number of ECG records: 21799

Columns:
['ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']

First 5 records:


,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


In [25]:
# STEP 1: Check whether PTB-XL waveform files are available

import os

print("Searching for PTB-XL waveform files...")

waveform_files = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith((".dat", ".hea")):
            waveform_files.append(os.path.join(root, file))

print(f"\nFound {len(waveform_files)} waveform/header files.")

for path in waveform_files[:30]:
    print(path)

Searching for PTB-XL waveform files...

Found 0 waveform/header files.


In [ ]:
# ============================================================
# STEP 4: VERIFY PTB-XL WAVEFORM DOWNLOAD
# ============================================================

import os

PTBXL_PATH = "/content/ptbxl"

dat_files = []
hea_files = []

for root, dirs, files in os.walk(PTBXL_PATH):
    for file in files:
        if file.endswith(".dat"):
            dat_files.append(os.path.join(root, file))
        elif file.endswith(".hea"):
            hea_files.append(os.path.join(root, file))

print("ECG waveform files:", len(dat_files))
print("Header files:", len(hea_files))

print("\nFirst 10 waveform files:")
for f in dat_files[:10]:
    print(f)

if dat_files and hea_files:
    print("\n✅ PTB-XL waveforms are ready.")
else:
    print("\n❌ Waveforms not found.")

In [ ]:
# ============================================================
# TEST PTB-XL WAVEFORM DOWNLOAD
# ============================================================

import os
import requests

TEST_URL = "https://physionet.org/files/ptb-xl/1.0.3/records100/00000/00001_lr.dat"
TEST_PATH = "/content/test_ecg.dat"

print("Testing PTB-XL waveform download...")

response = requests.get(TEST_URL)

print("HTTP status:", response.status_code)
print("Downloaded bytes:", len(response.content))

if response.status_code == 200 and len(response.content) > 0:
    with open(TEST_PATH, "wb") as f:
        f.write(response.content)

    print("✅ Test ECG downloaded successfully!")
    print("File:", TEST_PATH)
else:
    print("❌ Download failed.")
    print(response.text[:500])

In [ ]:
# ============================================================
# STEP 4 — DOWNLOAD A SMALL PTB-XL WAVEFORM SAMPLE
# ============================================================

import os
import requests
import pandas as pd
from tqdm.auto import tqdm

PTBXL_PATH = "/content/ptbxl"
os.makedirs(PTBXL_PATH, exist_ok=True)

BASE_URL = "https://physionet.org/files/ptb-xl/1.0.3/records100"

# Start small — 100 records only
# We will increase this after confirming everything works.

records_to_download = []

for _, row in metadata.iterrows():
    ecg_id = int(row["ecg_id"])

    # PTB-XL record path
    record = f"{ecg_id:05d}"
    folder = record[:3]

    dat_url = f"{BASE_URL}/{folder}/{record}_lr.dat"
    hea_url = f"{BASE_URL}/{folder}/{record}_lr.hea"

    records_to_download.append(
        (ecg_id, dat_url, hea_url)
    )

    if len(records_to_download) >= 100:
        break

print(f"Preparing to download {len(records_to_download)} records...")


def download_file(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return True

    try:
        r = requests.get(url, timeout=60)

        if r.status_code == 200 and len(r.content) > 0:
            with open(path, "wb") as f:
                f.write(r.content)
            return True

    except Exception as e:
        print("Download error:", e)

    return False


successful = 0

for ecg_id, dat_url, hea_url in tqdm(records_to_download):

    record = f"{ecg_id:05d}"
    folder = record[:3]

    record_dir = os.path.join(PTBXL_PATH, folder)
    os.makedirs(record_dir, exist_ok=True)

    dat_path = os.path.join(record_dir, f"{record}_lr.dat")
    hea_path = os.path.join(record_dir, f"{record}_lr.hea")

    dat_ok = download_file(dat_url, dat_path)
    hea_ok = download_file(hea_url, hea_path)

    if dat_ok and hea_ok:
        successful += 1


print()
print("===================================")
print("DOWNLOAD COMPLETE")
print("===================================")
print("Requested:", len(records_to_download))
print("Successful:", successful)
print("Failed:", len(records_to_download) - successful)

In [ ]:
# ============================================================
# STEP 4 — LOAD METADATA + DOWNLOAD 100 PTB-XL WAVEFORMS
# ============================================================

import os
import requests
import pandas as pd
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Load PTB-XL metadata
# ------------------------------------------------------------

METADATA_URL = "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"

print("Loading PTB-XL metadata...")

metadata = pd.read_csv(METADATA_URL)

print("✅ Metadata loaded!")
print("Number of records:", len(metadata))
print("Columns:", list(metadata.columns)[:10], "...")


# ------------------------------------------------------------
# 2. Prepare download directory
# ------------------------------------------------------------

PTBXL_PATH = "/content/ptbxl"
os.makedirs(PTBXL_PATH, exist_ok=True)

BASE_URL = "https://physionet.org/files/ptb-xl/1.0.3/records100"


# ------------------------------------------------------------
# 3. Select first 100 ECG records
# ------------------------------------------------------------

records_to_download = []

for _, row in metadata.iterrows():

    ecg_id = int(row["ecg_id"])

    # PTB-XL uses zero-padded record IDs
    record = f"{ecg_id:05d}"

    # Example:
    # 00001 -> records100/00000/00001_lr.dat
    folder = record[:3] + "00"

    dat_url = f"{BASE_URL}/{folder}/{record}_lr.dat"
    hea_url = f"{BASE_URL}/{folder}/{record}_lr.hea"

    records_to_download.append(
        (ecg_id, dat_url, hea_url)
    )

    if len(records_to_download) >= 100:
        break


print()
print("Records selected:", len(records_to_download))


# ------------------------------------------------------------
# 4. Download function
# ------------------------------------------------------------

def download_file(url, path):

    if os.path.exists(path) and os.path.getsize(path) > 0:
        return True

    try:
        response = requests.get(url, timeout=60)

        if response.status_code == 200 and len(response.content) > 0:

            with open(path, "wb") as f:
                f.write(response.content)

            return True

        return False

    except Exception as e:
        print("Download error:", e)
        return False


# ------------------------------------------------------------
# 5. Download records
# ------------------------------------------------------------

successful = 0
failed = 0

print("\nDownloading PTB-XL waveforms...\n")

for ecg_id, dat_url, hea_url in tqdm(records_to_download):

    record = f"{ecg_id:05d}"

    folder = record[:3] + "00"

    record_dir = os.path.join(
        PTBXL_PATH,
        folder
    )

    os.makedirs(record_dir, exist_ok=True)

    dat_path = os.path.join(
        record_dir,
        f"{record}_lr.dat"
    )

    hea_path = os.path.join(
        record_dir,
        f"{record}_lr.hea"
    )

    dat_ok = download_file(
        dat_url,
        dat_path
    )

    hea_ok = download_file(
        hea_url,
        hea_path
    )

    if dat_ok and hea_ok:
        successful += 1
    else:
        failed += 1


# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print()
print("===================================")
print("DOWNLOAD COMPLETE")
print("===================================")

print("Requested :", len(records_to_download))
print("Successful:", successful)
print("Failed    :", failed)

print()

if successful > 0:
    print("✅ PTB-XL waveform files are available.")
else:
    print("❌ No waveform files were downloaded.")

In [ ]:
# ============================================================
# STEP 5 — READ ONE REAL PTB-XL ECG WAVEFORM
# ============================================================

import os
import wfdb
import numpy as np
import matplotlib.pyplot as plt

PTBXL_PATH = "/content/ptbxl"

# Find downloaded .hea files
hea_files = []

for root, dirs, files in os.walk(PTBXL_PATH):
    for file in files:
        if file.endswith("_lr.hea"):
            hea_files.append(
                os.path.join(root, file)
            )

print("Found header files:", len(hea_files))

if len(hea_files) == 0:
    print("❌ No ECG header files found.")
else:

    # Take first available ECG
    hea_path = hea_files[0]

    # wfdb wants the path WITHOUT .hea
    record_path = hea_path[:-4]

    print("Reading:")
    print(record_path)

    record = wfdb.rdrecord(record_path)

    ecg = record.p_signal

    print()
    print("===================================")
    print("ECG INFORMATION")
    print("===================================")

    print("Signal shape:", ecg.shape)
    print("Sampling frequency:", record.fs)
    print("Number of leads:", ecg.shape[1])

    print()
    print("Lead names:")
    print(record.sig_name)

    # Plot first 5 seconds
    samples = int(record.fs * 5)

    plt.figure(figsize=(15, 5))
    plt.plot(
        np.arange(samples) / record.fs,
        ecg[:samples, 0]
    )

    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.title("PTB-XL ECG — Lead I")
    plt.grid(True)
    plt.show()

In [ ]:
# ============================================================
# STEP 8 — DETAILED MODEL EVALUATION
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("=" * 60)
print("DETAILED ECGResNet EVALUATION")
print("=" * 60)

# ------------------------------------------------------------
# IMPORTANT:
# Use the predictions and true labels generated by your
# EXISTING evaluation code.
# ------------------------------------------------------------

print("\nClasses:")
for i, name in enumerate(CLASSES):
    print(f"{i}: {name}")

In [ ]:
print("Checking existing evaluation variables...")

for name in ["y_true", "y_pred", "all_labels", "all_predictions",
             "true_labels", "predicted_labels"]:
    print(f"{name}: {'EXISTS' if name in globals() else 'not found'}")

In [ ]:
print("===== CURRENT COLAB STATE =====")

variables = [
    "model",
    "ECGResNet",
    "metadata",
    "dat_files",
    "hea_files",
    "PTBXL_PATH",
    "PTBXL_PATH",
    "CLASS_NAMES",
    "CLASSES"
]

for v in variables:
    print(f"{v}: {'YES' if v in globals() else 'NO'}")

print("\n===== DOWNLOADED FILES =====")

import os

for path in ["/content/ptbxl", "/content"]:
    if os.path.exists(path):
        print(f"\n{path}:")
        files = []
        for root, dirs, fs in os.walk(path):
            for f in fs:
                if f.endswith((".dat", ".hea")):
                    files.append(os.path.join(root, f))
        print("ECG files found:", len(files))
        print("Examples:", files[:5])

In [ ]:
# ============================================================
# RESTORE PTB-XL METADATA
# ============================================================

import os
import pandas as pd

METADATA_URL = "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"
metadata_path = "/content/ptbxl_database.csv"

print("Loading PTB-XL metadata...")

if not os.path.exists(metadata_path):
    metadata = pd.read_csv(METADATA_URL)
else:
    metadata = pd.read_csv(metadata_path)

print("✅ Metadata loaded")
print("Number of records:", len(metadata))
print("Columns:", list(metadata.columns))

print("\nFirst 3 records:")
display(metadata.head(3))

In [ ]:
# ============================================================
# STEP NEXT — MATCH DOWNLOADED ECGs WITH PTB-XL METADATA
# ============================================================

import os
import re

PTBXL_PATH = "/content/ptbxl"

# Find all .dat waveform files
dat_files = []

for root, dirs, files in os.walk(PTBXL_PATH):
    for file in files:
        if file.endswith(".dat"):
            dat_files.append(os.path.join(root, file))

print("Downloaded ECG waveform files:", len(dat_files))

# Extract PTB-XL record IDs
record_ids = []

for path in dat_files:
    filename = os.path.basename(path)

    # Example: 00000/00001_lr.dat -> 1
    match = re.search(r"(\d+)_lr\.dat$", filename)

    if match:
        record_ids.append(int(match.group(1)))

print("Record IDs found:", len(record_ids))
print("First 10 record IDs:", record_ids[:10])

# Match against metadata
test_metadata = metadata[
    metadata["ecg_id"].astype(int).isin(record_ids)
].copy()

print("\nMatched metadata records:", len(test_metadata))

display(
    test_metadata[
        ["ecg_id", "patient_id", "age", "sex", "scp_codes"]
    ].head(10)
)

In [ ]:
# ============================================================
# STEP — LOAD AND INSPECT A REAL PTB-XL ECG
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

# Install/import WFDB
!pip -q install wfdb

import wfdb

PTBXL_PATH = "/content/ptbxl"

# Find downloaded .dat files
dat_files = []

for root, dirs, files in os.walk(PTBXL_PATH):
    for file in files:
        if file.endswith(".dat"):
            dat_files.append(os.path.join(root, file))

dat_files = sorted(dat_files)

print("ECG waveform files:", len(dat_files))

# Take the first ECG
ecg_path = dat_files[0]

# Remove .dat extension for WFDB
record_path = ecg_path[:-4]

print("Loading:")
print(record_path)

# Read ECG
signal, fields = wfdb.rdsamp(record_path)

print("\nECG loaded successfully!")
print("Signal shape:", signal.shape)
print("Sampling frequency:", fields["fs"])
print("Number of leads:", fields["n_sig"])
print("Lead names:", fields["sig_name"])
print("Duration:", len(signal) / fields["fs"], "seconds")

# Convert to PyTorch format
ecg = torch.tensor(signal, dtype=torch.float32).T

print("\nPyTorch shape:", ecg.shape)

In [ ]:
# ============================================================
# CHECK CURRENT MODEL STATUS
# ============================================================

print("MODEL:", "YES" if "model" in globals() else "NO")
print("ECGResNet:", "YES" if "ECGResNet" in globals() else "NO")
print("CLASSES:", CLASSES if "CLASSES" in globals() else "NO")

print("\n/content files:")
for f in os.listdir("/content"):
    if f.endswith((".pth", ".pt", ".ckpt")):
        print("  ", f)

print("\nPTB-XL model files:")
for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith((".pth", ".pt", ".ckpt")):
            print(os.path.join(root, f))

In [ ]:
# ============================================================
# VERIFY TRAINED MODEL CHECKPOINT
# ============================================================

import os

MODEL_PATH = "/content/drive/MyDrive/best_ecg_model.pth"

print("Checking trained model checkpoint...\n")

print("Path:", MODEL_PATH)
print("Exists:", os.path.exists(MODEL_PATH))

if os.path.exists(MODEL_PATH):
    print("File size:", round(os.path.getsize(MODEL_PATH) / (1024**2), 2), "MB")
    print("✅ Trained model checkpoint is accessible.")
else:
    print("❌ Checkpoint is NOT accessible.")
    print("Google Drive may need to be mounted.")

In [ ]:
# ============================================================
# STEP — LOAD THE TRAINED ECGResNet WEIGHTS
# ============================================================

import torch

MODEL_PATH = "/content/drive/MyDrive/best_ecg_model.pth"

print("Loading trained ECGResNet...")

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu"
)

print("Checkpoint type:", type(checkpoint))

# Handle the common checkpoint formats
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]

elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]

else:
    state_dict = checkpoint

# Remove possible DataParallel prefix
state_dict = {
    k.replace("module.", "", 1) if k.startswith("module.") else k: v
    for k, v in state_dict.items()
}

# Load weights into the existing model
model.load_state_dict(state_dict)

# Evaluation mode
model.eval()

print("✅ Trained ECGResNet weights loaded successfully!")
print("Total parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# ============================================================
# STEP — RUN TRAINED MODEL ON THE REAL PTB-XL ECGs
# ============================================================

import os
import numpy as np
import torch
import wfdb

print("Running ECGResNet on real PTB-XL ECGs...")

# Make sure model is in evaluation mode
model.eval()

# Use the ECG files already downloaded
ecg_files = []

for root, dirs, files in os.walk("/content/ptbxl"):
    for f in files:
        if f.endswith("_lr.dat"):
            ecg_files.append(os.path.join(root, f))

ecg_files = sorted(ecg_files)

print("ECG records found:", len(ecg_files))

predictions = []
record_ids = []

with torch.no_grad():

    for record_path in ecg_files:

        # Load 12-lead ECG
        signal, fields = wfdb.rdsamp(
            record_path.replace("_lr.dat", "_lr")
        )

        # signal shape: (1000, 12)
        # Convert to: (12, 1000)
        ecg = torch.tensor(
            signal,
            dtype=torch.float32
        ).T

        # Add batch dimension
        ecg = ecg.unsqueeze(0)

        # Model prediction
        logits = model(ecg)

        # Convert logits → probabilities
        probabilities = torch.softmax(logits, dim=1)

        # Predicted class
        predicted_index = torch.argmax(
            probabilities,
            dim=1
        ).item()

        predictions.append(predicted_index)

        # Extract ECG ID from filename
        filename = os.path.basename(record_path)
        ecg_id = int(filename.split("_")[0])

        record_ids.append(ecg_id)

print("\n✅ REAL ECG INFERENCE COMPLETE")

print("Records processed:", len(predictions))

print("\nFirst 20 predictions:")

for ecg_id, pred in zip(record_ids[:20], predictions[:20]):
    print(
        f"ECG {ecg_id:05d} → "
        f"{CLASSES[pred]}"
    )

In [ ]:
# ============================================================
# STEP — MATCH MODEL PREDICTIONS WITH PTB-XL GROUND TRUTH
# ============================================================

import ast
import pandas as pd
import numpy as np

print("=" * 60)
print("MATCHING PREDICTIONS WITH PTB-XL GROUND TRUTH")
print("=" * 60)

# ------------------------------------------------------------
# 1. Make sure metadata exists
# ------------------------------------------------------------

if "metadata" not in globals():
    print("Loading PTB-XL metadata...")

    metadata = pd.read_csv(
        "/content/ptbxl/ptbxl_database.csv"
    )

print("Metadata records:", len(metadata))

# ------------------------------------------------------------
# 2. Create ECG ID → SCP diagnostic code mapping
# ------------------------------------------------------------

metadata_lookup = {}

for _, row in metadata.iterrows():

    ecg_id = int(row["ecg_id"])

    scp_codes = row["scp_codes"]

    # Convert string representation to dictionary
    if isinstance(scp_codes, str):
        try:
            scp_codes = ast.literal_eval(scp_codes)
        except:
            scp_codes = {}

    metadata_lookup[ecg_id] = scp_codes


# ------------------------------------------------------------
# 3. Convert PTB-XL SCP codes into our 5 classes
# ------------------------------------------------------------

def get_diagnostic_class(scp_codes):

    if not isinstance(scp_codes, dict):
        return "UNKNOWN"

    codes = set(scp_codes.keys())

    # PTB-XL diagnostic superclass mapping
    if any(code.startswith("NORM") for code in codes):
        return "NORM"

    if any(code.startswith("MI") for code in codes):
        return "MI"

    if any(code.startswith("STTC") for code in codes):
        return "STIC"

    if any(code.startswith("CD") for code in codes):
        return "CD"

    if any(code.startswith("HYP") for code in codes):
        return "HYP"

    return "UNKNOWN"


# ------------------------------------------------------------
# 4. Match predictions with true labels
# ------------------------------------------------------------

results = []

for ecg_id, predicted_index in zip(
    record_ids,
    predictions
):

    true_codes = metadata_lookup.get(ecg_id, {})

    true_class = get_diagnostic_class(true_codes)

    predicted_class = CLASSES[predicted_index]

    results.append({
        "ecg_id": ecg_id,
        "true_class": true_class,
        "predicted_class": predicted_class,
        "scp_codes": true_codes
    })


results_df = pd.DataFrame(results)


# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\nMatched records:", len(results_df))

print("\nFirst 20 matched predictions:")

display(
    results_df[
        ["ecg_id", "true_class", "predicted_class"]
    ].head(20)
)

print("\nTrue-label distribution:")
print(
    results_df["true_class"].value_counts()
)

print("\nPredicted-label distribution:")
print(
    results_df["predicted_class"].value_counts()
)

In [ ]:
# ============================================================
# STEP — CORRECT PTB-XL DIAGNOSTIC SUPERCLASS MAPPING
# ============================================================

import pandas as pd
import ast

print("=" * 60)
print("CORRECTING PTB-XL GROUND-TRUTH LABELS")
print("=" * 60)

# ------------------------------------------------------------
# 1. Load PTB-XL SCP statement mapping
# ------------------------------------------------------------

SCP_PATH = "/content/ptbxl/scp_statements.csv"

scp_df = pd.read_csv(SCP_PATH, index_col=0)

print("SCP statements loaded:", len(scp_df))
print("Columns:")
print(scp_df.columns.tolist())


# ------------------------------------------------------------
# 2. Keep only diagnostic statements
# ------------------------------------------------------------

diagnostic_map = scp_df[
    scp_df["diagnostic"] == 1
]

print("\nDiagnostic SCP statements:",
      len(diagnostic_map))


# ------------------------------------------------------------
# 3. Create SCP code → diagnostic superclass mapping
# ------------------------------------------------------------

scp_to_class = {}

for code, row in diagnostic_map.iterrows():

    superclass = row["diagnostic_class"]

    if pd.notna(superclass):
        scp_to_class[code] = superclass


print("\nExample SCP mappings:")

for code in list(scp_to_class.keys())[:20]:
    print(
        f"{code} → {scp_to_class[code]}"
    )


# ------------------------------------------------------------
# 4. Convert each ECG's SCP codes to superclasses
# ------------------------------------------------------------

def get_ptbxl_superclasses(scp_codes):

    if isinstance(scp_codes, str):
        try:
            scp_codes = ast.literal_eval(scp_codes)
        except:
            return []

    if not isinstance(scp_codes, dict):
        return []

    classes = set()

    for code in scp_codes.keys():

        if code in scp_to_class:

            superclass = scp_to_class[code]

            classes.add(superclass)

    return sorted(classes)


# ------------------------------------------------------------
# 5. Rebuild ground-truth labels
# ------------------------------------------------------------

correct_results = []

for ecg_id, predicted_index in zip(
    record_ids,
    predictions
):

    row = metadata[
        metadata["ecg_id"] == ecg_id
    ]

    if len(row) == 0:
        continue

    scp_codes = row.iloc[0]["scp_codes"]

    true_classes = get_ptbxl_superclasses(
        scp_codes
    )

    predicted_class = CLASSES[predicted_index]

    correct_results.append({
        "ecg_id": ecg_id,
        "true_classes": true_classes,
        "predicted_class": predicted_class
    })


correct_results_df = pd.DataFrame(
    correct_results
)


# ------------------------------------------------------------
# 6. Display corrected labels
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CORRECTED LABELS")
print("=" * 60)

display(
    correct_results_df[
        ["ecg_id", "true_classes", "predicted_class"]
    ].head(20)
)


# ------------------------------------------------------------
# 7. Distribution of diagnostic classes
# ------------------------------------------------------------

print("\nDiagnostic superclass distribution:")

class_counts = {
    "NORM": 0,
    "MI": 0,
    "STTC": 0,
    "CD": 0,
    "HYP": 0
}

for classes in correct_results_df["true_classes"]:

    for c in classes:

        if c in class_counts:
            class_counts[c] += 1


for c, count in class_counts.items():

    print(
        f"{c:5s}: {count}"
    )


print("\nPredicted distribution:")

print(
    correct_results_df[
        "predicted_class"
    ].value_counts()
)

print("\n✅ Ground-truth mapping corrected.")

In [ ]:
import os

print("Searching for scp_statements.csv...\n")

matches = []

for root, dirs, files in os.walk("/content"):
    for f in files:
        if f == "scp_statements.csv":
            matches.append(os.path.join(root, f))

if matches:
    print("Found:")
    for path in matches:
        print(path)
else:
    print("❌ scp_statements.csv is not present in /content")

In [ ]:
import requests

SCP_URL = "https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv"
SCP_PATH = "/content/scp_statements.csv"

print("Downloading PTB-XL SCP mapping...")

response = requests.get(SCP_URL)

print("HTTP status:", response.status_code)
print("Downloaded bytes:", len(response.content))

if response.status_code == 200:
    with open(SCP_PATH, "wb") as f:
        f.write(response.content)

    print("✅ scp_statements.csv downloaded successfully")
    print("File:", SCP_PATH)
else:
    print("❌ Download failed")

In [ ]:
import pandas as pd

scp_df = pd.read_csv(
    "/content/scp_statements.csv",
    index_col=0
)

print("SCP statements:", len(scp_df))
print("Columns:", scp_df.columns.tolist())

display(scp_df.head())

In [ ]:
# ============================================================
# STEP — BUILD CORRECT PTB-XL GROUND-TRUTH LABELS
# ============================================================

import ast
import pandas as pd

print("=" * 60)
print("BUILDING CORRECT PTB-XL GROUND-TRUTH LABELS")
print("=" * 60)

# ------------------------------------------------------------
# 1. Diagnostic SCP statements only
# ------------------------------------------------------------

diagnostic_map = scp_df[
    scp_df["diagnostic"] == 1
].copy()

print("Diagnostic SCP statements:", len(diagnostic_map))


# ------------------------------------------------------------
# 2. SCP CODE → PTB-XL DIAGNOSTIC CLASS
# ------------------------------------------------------------

scp_to_class = {}

for code, row in diagnostic_map.iterrows():

    diagnostic_class = row["diagnostic_class"]

    if pd.notna(diagnostic_class):

        scp_to_class[code] = diagnostic_class


print("\nExample mappings:")

for code in list(scp_to_class.keys())[:20]:
    print(
        f"{code:10s} → {scp_to_class[code]}"
    )


# ------------------------------------------------------------
# 3. Function to extract diagnostic classes
# ------------------------------------------------------------

def get_diagnostic_classes(scp_codes):

    if isinstance(scp_codes, str):

        try:
            scp_codes = ast.literal_eval(scp_codes)

        except Exception:
            return []


    if not isinstance(scp_codes, dict):
        return []


    classes = []

    for code in scp_codes.keys():

        if code in scp_to_class:

            diagnostic_class = scp_to_class[code]

            if diagnostic_class not in classes:

                classes.append(diagnostic_class)


    return sorted(classes)


# ------------------------------------------------------------
# 4. Build corrected results
# ------------------------------------------------------------

correct_results = []

for ecg_id, predicted_index in zip(
    record_ids,
    predictions
):

    matching = metadata[
        metadata["ecg_id"] == ecg_id
    ]

    if len(matching) == 0:
        continue


    scp_codes = matching.iloc[0]["scp_codes"]

    true_classes = get_diagnostic_classes(
        scp_codes
    )

    predicted_class = CLASSES[predicted_index]


    correct_results.append({

        "ecg_id": ecg_id,

        "true_classes": true_classes,

        "predicted_class": predicted_class

    })


correct_results_df = pd.DataFrame(
    correct_results
)


# ------------------------------------------------------------
# 5. Display first 20
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CORRECTED LABELS")
print("=" * 60)

display(
    correct_results_df[
        [
            "ecg_id",
            "true_classes",
            "predicted_class"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 6. Count diagnostic classes
# ------------------------------------------------------------

class_counts = {}

for classes in correct_results_df[
    "true_classes"
]:

    for c in classes:

        class_counts[c] = (
            class_counts.get(c, 0) + 1
        )


print("\n" + "=" * 60)
print("TRUE DIAGNOSTIC CLASS DISTRIBUTION")
print("=" * 60)

for c, count in sorted(
    class_counts.items()
):

    print(
        f"{c:10s}: {count}"
    )


# ------------------------------------------------------------
# 7. Prediction distribution
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MODEL PREDICTION DISTRIBUTION")
print("=" * 60)

print(
    correct_results_df[
        "predicted_class"
    ].value_counts()
)


print("\n✅ Ground-truth labels successfully rebuilt.")

In [ ]:
# ============================================================
# STEP — ECGResNet PERFORMANCE EVALUATION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score
)

print("=" * 70)
print("ECGResNet PERFORMANCE EVALUATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. Convert ground truth into NORMAL vs ABNORMAL
# ------------------------------------------------------------

def is_abnormal(classes):

    if not classes:
        return None

    # NORM = normal
    if classes == ["NORM"]:
        return 0

    # Anything else = abnormal
    return 1


correct_results_df["true_abnormal"] = (
    correct_results_df["true_classes"]
    .apply(is_abnormal)
)


correct_results_df["pred_abnormal"] = (
    correct_results_df["predicted_class"]
    .apply(
        lambda x: 0 if x == "NORM" else 1
    )
)


# Remove records with no diagnostic ground truth
binary_df = correct_results_df.dropna(
    subset=["true_abnormal"]
).copy()


y_true_binary = binary_df[
    "true_abnormal"
].astype(int)

y_pred_binary = binary_df[
    "pred_abnormal"
].astype(int)


# ------------------------------------------------------------
# 2. Binary confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true_binary,
    y_pred_binary,
    labels=[0, 1]
)


tn, fp, fn, tp = cm.ravel()


print("\nCONFUSION MATRIX")
print("----------------")
print("                Predicted")
print("              Normal  Abnormal")
print(f"True Normal    {tn:5d}    {fp:5d}")
print(f"True Abnormal  {fn:5d}    {tp:5d}")


# ------------------------------------------------------------
# 3. Calculate metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_true_binary,
    y_pred_binary
)

sensitivity = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else 0
)

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0
)

balanced_acc = balanced_accuracy_score(
    y_true_binary,
    y_pred_binary
)


print("\nBINARY PERFORMANCE")
print("------------------")

print(
    f"Accuracy:           {accuracy * 100:.2f}%"
)

print(
    f"Sensitivity:        {sensitivity * 100:.2f}%"
)

print(
    f"Specificity:        {specificity * 100:.2f}%"
)

print(
    f"Balanced Accuracy:  {balanced_acc * 100:.2f}%"
)


# ------------------------------------------------------------
# 4. Classification report
# ------------------------------------------------------------

print("\nCLASSIFICATION REPORT")
print("---------------------")

print(
    classification_report(
        y_true_binary,
        y_pred_binary,
        target_names=[
            "NORMAL",
            "ABNORMAL"
        ],
        zero_division=0
    )
)


# ------------------------------------------------------------
# 5. Class distribution
# ------------------------------------------------------------

print("\nGROUND TRUTH")
print("------------")

print(
    binary_df["true_abnormal"]
    .value_counts()
    .sort_index()
    .rename(
        index={
            0: "NORMAL",
            1: "ABNORMAL"
        }
    )
)


print("\nPREDICTIONS")
print("-----------")

print(
    binary_df["pred_abnormal"]
    .value_counts()
    .sort_index()
    .rename(
        index={
            0: "NORMAL",
            1: "ABNORMAL"
        }
    )
)


print("\n" + "=" * 70)
print("EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP — DIAGNOSE ECGResNet PREDICTION COLLAPSE
# ============================================================

import torch
import numpy as np
import pandas as pd

print("=" * 70)
print("ECGResNet PREDICTION DIAGNOSTICS")
print("=" * 70)

print("\nModel output classes:")
for i, c in enumerate(CLASSES):
    print(f"  {i}: {c}")

# ------------------------------------------------------------
# Find several ECGs from different diagnostic classes
# ------------------------------------------------------------

diagnostic_samples = correct_results_df[
    correct_results_df["true_classes"].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    )
].copy()

print("\nDiagnostic records available:", len(diagnostic_samples))

print("\nTrue-class distribution:")
print(
    diagnostic_samples["true_classes"]
    .explode()
    .value_counts()
)


# ------------------------------------------------------------
# Run model and inspect RAW probabilities
# ------------------------------------------------------------

records_to_test = []

for _, row in diagnostic_samples.head(20).iterrows():

    ecg_id = int(row["ecg_id"])

    # Find corresponding waveform
    possible_files = []

    for root, dirs, files in os.walk("/content/ptbxl"):
        for f in files:
            if f.endswith(".hea"):
                if f.startswith(str(ecg_id).zfill(5)):
                    possible_files.append(
                        os.path.join(root, f)
                    )

    if len(possible_files) == 0:
        continue

    record_path = possible_files[0].replace(".hea", "")

    try:

        signal, fields = wfdb.rdsamp(record_path)

        # Convert to PyTorch
        ecg = torch.tensor(
            signal,
            dtype=torch.float32
        ).T

        # Ensure expected shape [channels, samples]
        if ecg.shape[0] != 12:
            continue

        # Resize / crop to 1000 samples
        if ecg.shape[1] > 1000:
            ecg = ecg[:, :1000]

        elif ecg.shape[1] < 1000:
            padded = torch.zeros(
                (12, 1000),
                dtype=torch.float32
            )
            padded[:, :ecg.shape[1]] = ecg
            ecg = padded

        ecg = ecg.unsqueeze(0)

        with torch.no_grad():
            logits = model(ecg)

            probabilities = torch.softmax(
                logits,
                dim=1
            )[0]

        records_to_test.append({
            "ecg_id": ecg_id,
            "true_class": row["true_classes"],
            "NORM": probabilities[0].item(),
            "STTC": probabilities[1].item(),
            "CD": probabilities[2].item(),
            "MI": probabilities[3].item(),
            "HYP": probabilities[4].item(),
        })

    except Exception as e:
        print(
            f"Could not process ECG {ecg_id}: {e}"
        )


prob_df = pd.DataFrame(records_to_test)


print("\nRAW MODEL PROBABILITIES")
print("=" * 70)

if len(prob_df) > 0:

    display(
        prob_df.style.format({
            "NORM": "{:.4f}",
            "STTC": "{:.4f}",
            "CD": "{:.4f}",
            "MI": "{:.4f}",
            "HYP": "{:.4f}",
        })
    )

    print("\nAverage predicted probability:")
    print(
        prob_df[
            ["NORM", "STTC", "CD", "MI", "HYP"]
        ].mean()
    )

else:
    print("No probability records generated.")


print("\n" + "=" * 70)
print("DIAGNOSTIC CHECK COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CHECKPOINT WEIGHT DIAGNOSTICS
# ============================================================

import torch
import numpy as np

print("=" * 70)
print("ECGResNet CHECKPOINT WEIGHT DIAGNOSTICS")
print("=" * 70)

total_params = 0
zero_params = 0

for name, param in model.named_parameters():

    data = param.detach().cpu()

    total = data.numel()
    zeros = (data == 0).sum().item()

    total_params += total
    zero_params += zeros

    print(
        f"{name:55s} "
        f"shape={str(tuple(data.shape)):20s} "
        f"mean={data.mean().item():.6f} "
        f"std={data.std().item():.6f} "
        f"min={data.min().item():.6f} "
        f"max={data.max().item():.6f}"
    )

print("\n" + "-" * 70)

print("Total parameters:", total_params)
print("Zero parameters:", zero_params)
print(
    "Zero percentage:",
    f"{zero_params / total_params * 100:.4f}%"
)

print("\n" + "=" * 70)
print("CHECK COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 2 — RAW LOGIT / PROBABILITY DIAGNOSTICS
# ============================================================

import torch
import torch.nn.functional as F

print("=" * 70)
print("ECGResNet RAW OUTPUT DIAGNOSTICS")
print("=" * 70)

model.eval()

# ecg is already [12, 1000]
# Add ONLY batch dimension
x = ecg.unsqueeze(0).to(next(model.parameters()).device)

print("Input shape:", x.shape)
print("Input dtype:", x.dtype)
print("Input min:", x.min().item())
print("Input max:", x.max().item())
print("Input mean:", x.mean().item())
print("Input std:", x.std().item())

with torch.no_grad():
    logits = model(x)

print("\nRAW LOGITS:")
print(logits.detach().cpu().numpy())

print("\nLOGIT STATISTICS:")
print("Min :", logits.min().item())
print("Max :", logits.max().item())
print("Mean:", logits.mean().item())
print("Std :", logits.std().item())

probs = F.softmax(logits, dim=1)

print("\nSOFTMAX PROBABILITIES:")

for i, p in enumerate(probs[0].detach().cpu().numpy()):
    print(f"{CLASSES[i]:5s}: {p:.8f}")

pred = torch.argmax(logits, dim=1).item()

print("\nPredicted class:")
print(pred, "->", CLASSES[pred])

print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# FIX INPUT SHAPE + RUN RAW MODEL OUTPUT
# ============================================================

import torch
import torch.nn.functional as F

model.eval()

print("Original ECG shape:", ecg.shape)

# Remove unnecessary batch dimensions
x = ecg.squeeze()

# At this point we want [12, 1000]
print("After squeeze:", x.shape)

# Add exactly ONE batch dimension
if x.ndim == 2:
    x = x.unsqueeze(0)

print("Final model input shape:", x.shape)

# Safety check
assert x.ndim == 3, f"Expected 3D input, got {x.shape}"
assert x.shape[1] == 12, f"Expected 12 ECG leads, got {x.shape}"

x = x.to(next(model.parameters()).device).float()

with torch.no_grad():
    logits = model(x)

print("\n" + "=" * 60)
print("RAW LOGITS")
print("=" * 60)

print(logits.detach().cpu().numpy())

print("\nLOGIT STATISTICS")
print("Min :", logits.min().item())
print("Max :", logits.max().item())
print("Mean:", logits.mean().item())
print("Std :", logits.std().item())

probs = F.softmax(logits, dim=1)

print("\n" + "=" * 60)
print("CLASS PROBABILITIES")
print("=" * 60)

for i, p in enumerate(probs[0].detach().cpu().numpy()):
    print(f"{CLASSES[i]:5s}: {p:.8f}")

pred = torch.argmax(logits, dim=1).item()

print("\nPredicted class:", pred)
print("Predicted label:", CLASSES[pred])

In [ ]:
# ============================================================
# STEP 3 — CHECK FEATURES BEFORE FINAL CLASSIFIER
# ============================================================

import torch

model.eval()

# Prepare correct input
x = ecg.squeeze()

if x.ndim == 2:
    x = x.unsqueeze(0)

x = x.to(next(model.parameters()).device).float()

# Store classifier input
classifier_input = {}

def capture_classifier_input(module, inputs, output):
    classifier_input["x"] = inputs[0].detach().cpu()

# Register hook
hook = model.classifier.register_forward_hook(capture_classifier_input)

with torch.no_grad():
    logits = model(x)

hook.remove()

features = classifier_input["x"]

print("=" * 70)
print("CLASSIFIER FEATURE DIAGNOSTICS")
print("=" * 70)

print("Feature shape:", features.shape)

print("\nFeature statistics:")
print("Min :", features.min().item())
print("Max :", features.max().item())
print("Mean:", features.mean().item())
print("Std :", features.std().item())

print("\nFirst 20 feature values:")
print(features.flatten()[:20].numpy())

print("\nNumber of near-zero features:")
near_zero = (features.abs() < 1e-6).sum().item()
total = features.numel()

print(f"{near_zero} / {total}")
print(f"Percentage: {(near_zero / total) * 100:.2f}%")

print("\n" + "=" * 70)
print("CHECK COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 4 — CHECK WHETHER FEATURES CHANGE BETWEEN ECGs
# ============================================================

import torch
import numpy as np

model.eval()

# We will use several ECG records already available
# from the PTB-XL files.

print("=" * 70)
print("FEATURE VARIATION TEST")
print("=" * 70)

# Get ECG files
ecg_files = []

for root, dirs, files in os.walk("/content/ptbxl"):
    for f in files:
        if f.endswith(".dat"):
            ecg_files.append(os.path.join(root, f))

ecg_files = sorted(ecg_files)

print("ECG files available:", len(ecg_files))

# Take first 10 ECGs
test_files = ecg_files[:10]

all_features = []
all_logits = []

# Hook classifier input
classifier_input = {}

def capture_features(module, inputs, output):
    classifier_input["x"] = inputs[0].detach()

hook = model.classifier.register_forward_hook(capture_features)

for i, record_path in enumerate(test_files):

    try:
        signal, fields = wfdb.rdsamp(record_path)

        # signal = [1000, 12]
        # Convert to [12, 1000]
        x = torch.tensor(
            signal,
            dtype=torch.float32
        ).T

        # Add batch dimension
        x = x.unsqueeze(0)

        x = x.to(next(model.parameters()).device)

        with torch.no_grad():
            logits = model(x)

        features = classifier_input["x"]

        all_features.append(
            features.squeeze(0).cpu().numpy()
        )

        all_logits.append(
            logits.squeeze(0).cpu().numpy()
        )

        print(
            f"ECG {i+1:02d} | "
            f"feature mean = {features.mean().item():.6f} | "
            f"feature std = {features.std().item():.6f} | "
            f"logit std = {logits.std().item():.6f}"
        )

    except Exception as e:
        print(f"ECG {i+1:02d} ERROR:", e)

hook.remove()

# Convert to arrays
all_features = np.array(all_features)
all_logits = np.array(all_logits)

print("\n" + "=" * 70)
print("ACROSS-ECG FEATURE VARIATION")
print("=" * 70)

print("Feature matrix shape:", all_features.shape)

print("\nMean feature variation:")
print("Mean:", all_features.mean())

print("Std:", all_features.std())

print("Min:", all_features.min())
print("Max:", all_features.max())

# Variation for each feature across ECGs
feature_variance = np.var(all_features, axis=0)

print("\nAverage variance across 512 features:")
print(feature_variance.mean())

print("Maximum feature variance:")
print(feature_variance.max())

# Compare first and second ECG
if len(all_features) >= 2:

    difference = np.abs(
        all_features[0] - all_features[1]
    )

    print("\nDifference between ECG 1 and ECG 2:")
    print("Mean absolute difference:", difference.mean())
    print("Maximum absolute difference:", difference.max())

print("\n" + "=" * 70)
print("LOGIT VARIATION")
print("=" * 70)

print("Logit matrix shape:", all_logits.shape)

print("Logit mean:", all_logits.mean())
print("Logit std:", all_logits.std())

print("\nFirst 5 ECG logits:")

for i in range(min(5, len(all_logits))):
    print(f"ECG {i+1}:", all_logits[i])

print("\n" + "=" * 70)
print("TEST COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 4A — FIND PTB-XL ECG FILES
# ============================================================

import os

print("=" * 70)
print("SEARCHING FOR PTB-XL ECG FILES")
print("=" * 70)

# Search common locations
search_roots = [
    "/content",
    "/content/drive/MyDrive"
]

dat_files = []
hea_files = []

for root in search_roots:

    if not os.path.exists(root):
        continue

    print(f"\nSearching: {root}")

    for current_root, dirs, files in os.walk(root):

        # Avoid unnecessary hidden/system directories
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
        ]

        for f in files:

            full_path = os.path.join(current_root, f)

            if f.lower().endswith(".dat"):
                dat_files.append(full_path)

            elif f.lower().endswith(".hea"):
                hea_files.append(full_path)


print("\n" + "=" * 70)
print("SEARCH RESULT")
print("=" * 70)

print("DAT files found:", len(dat_files))
print("HEA files found:", len(hea_files))

print("\nFirst 20 DAT files:")

for f in dat_files[:20]:
    print(f)

print("\nFirst 20 HEA files:")

for f in hea_files[:20]:
    print(f)

print("\n" + "=" * 70)
print("CHECK COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 4B — LOAD REAL PTB-XL ECG RECORDS
# ============================================================

import os
import numpy as np
import wfdb

PTBXL_ROOT = "/content/ptbxl"

# Find all ECG records recursively
records = []

for root, dirs, files in os.walk(PTBXL_ROOT):
    for f in files:
        if f.endswith("_lr.dat"):
            record_path = os.path.join(root, f[:-4])  # remove .dat
            header_path = record_path + ".hea"

            if os.path.exists(header_path):
                records.append(record_path)

print("=" * 70)
print("PTB-XL RECORD DISCOVERY")
print("=" * 70)

print("Complete records found:", len(records))

for r in records[:10]:
    print(r)

# ------------------------------------------------------------
# Load first few ECGs
# ------------------------------------------------------------

ecgs = []

for record_path in records[:20]:

    try:
        signal, fields = wfdb.rdsamp(record_path)

        ecgs.append(signal)

        print(
            os.path.basename(record_path),
            "shape =", signal.shape,
            "sampling frequency =", fields["fs"]
        )

    except Exception as e:
        print(
            "ERROR:",
            record_path,
            "|",
            str(e)
        )

print("\nSuccessfully loaded ECGs:", len(ecgs))

print("=" * 70)
print("LOAD COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 5 — PREPARE REAL ECG FOR ECGResNet
# ============================================================

import torch
import numpy as np

# Load one real ECG
record_path = records[0]

signal, fields = wfdb.rdsamp(record_path)

print("Original ECG shape:", signal.shape)

# Convert:
# (1000, 12)
#      ↓
# (12, 1000)

ecg = torch.tensor(signal, dtype=torch.float32).T

print("After transpose:", ecg.shape)

# Model expects:
# (batch, channels, time)
# (1, 12, 1000)

x = ecg.unsqueeze(0)

print("Final model input:", x.shape)
print("dtype:", x.dtype)

# Basic statistics
print("\nECG statistics:")
print("Min :", x.min().item())
print("Max :", x.max().item())
print("Mean:", x.mean().item())
print("Std :", x.std().item())

In [ ]:
# ============================================================
# STEP 6 — TEST MODEL OUTPUT VARIATION
# ============================================================

import torch
import pandas as pd

model.eval()

results = []

for i, record_path in enumerate(records[:20]):

    signal, fields = wfdb.rdsamp(record_path)

    # (1000, 12) -> (12, 1000)
    ecg = torch.tensor(signal, dtype=torch.float32).T

    # (12, 1000) -> (1, 12, 1000)
    x = ecg.unsqueeze(0)

    with torch.no_grad():
        logits = model(x)

        # Convert logits to probabilities
        probs = torch.softmax(logits, dim=1)

        pred = torch.argmax(probs, dim=1).item()

    results.append({
        "record": os.path.basename(record_path),
        "predicted": CLASSES[pred],
        "NORM": probs[0, 0].item(),
        "STIC": probs[0, 1].item(),
        "CD":   probs[0, 2].item(),
        "MI":   probs[0, 3].item(),
        "HYP":  probs[0, 4].item()
    })

results_df = pd.DataFrame(results)

display(results_df)

In [ ]:
# ============================================================
# STEP 6B — OUTPUT VARIATION ANALYSIS
# ============================================================

prob_cols = ["NORM", "STIC", "CD", "MI", "HYP"]

print("=" * 70)
print("MODEL OUTPUT VARIATION")
print("=" * 70)

print("\nProbability standard deviation:")
print(results_df[prob_cols].std())

print("\nProbability minimum:")
print(results_df[prob_cols].min())

print("\nProbability maximum:")
print(results_df[prob_cols].max())

print("\nPredicted class distribution:")
print(results_df["predicted"].value_counts())

print("\nFirst 5 predictions:")
display(results_df.head())

In [ ]:
# ============================================================
# STEP 7 — CHECK WHETHER FEATURES ARE COLLAPSING
# ============================================================

import torch

model.eval()

feature_outputs = []
logit_outputs = []

for record_path in records[:20]:

    signal, fields = wfdb.rdsamp(record_path)

    # (1000, 12) -> (12, 1000) -> (1, 12, 1000)
    x = torch.tensor(signal, dtype=torch.float32).T.unsqueeze(0)

    with torch.no_grad():

        # Get feature representation
        features = model.stem(x)

        features = model.block1(features)
        features = model.block2(features)
        features = model.block3(features)
        features = model.block4(features)

        # Global average pooling
        features = torch.mean(features, dim=-1)

        # Classifier
        logits = model.classifier(features)

    feature_outputs.append(features.squeeze(0).cpu())
    logit_outputs.append(logits.squeeze(0).cpu())

features_tensor = torch.stack(feature_outputs)
logits_tensor = torch.stack(logit_outputs)

print("=" * 70)
print("FEATURE COLLAPSE DIAGNOSTIC")
print("=" * 70)

print("\nFeature shape:")
print(features_tensor.shape)

print("\nFeature statistics:")
print("Min :", features_tensor.min().item())
print("Max :", features_tensor.max().item())
print("Mean:", features_tensor.mean().item())
print("Std :", features_tensor.std().item())

print("\nVariation across ECGs:")
print("Mean feature std:",
      features_tensor.std(dim=0).mean().item())

print("Maximum feature std:",
      features_tensor.std(dim=0).max().item())

print("\nLogit variation:")
print("Mean logit std:",
      logits_tensor.std(dim=0).mean().item())

print("Maximum logit std:",
      logits_tensor.std(dim=0).max().item())

print("\nFirst ECG logits:")
print(logits_tensor[0])

print("\nLast ECG logits:")
print(logits_tensor[-1])

print("=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 8 — CHECK TRAINED CHECKPOINT
# ============================================================

import torch
import os

MODEL_PATH = "/content/drive/MyDrive/best_ecg_model.pth"

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu"
)

print("=" * 70)
print("CHECKPOINT STRUCTURE")
print("=" * 70)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):

    print("\nCheckpoint keys:")
    print(list(checkpoint.keys()))

    for key, value in checkpoint.items():

        if isinstance(value, dict):
            print(f"\n{key}: dictionary with {len(value)} entries")

        elif torch.is_tensor(value):
            print(
                f"\n{key}: tensor",
                value.shape
            )

        else:
            print(
                f"\n{key}:",
                type(value),
                value
            )

print("=" * 70)

In [ ]:
# ============================================================
# STEP 8B — CHECK CLASSIFIER WEIGHTS
# ============================================================

state_dict = checkpoint

print("=" * 70)
print("CLASSIFIER WEIGHT DIAGNOSTICS")
print("=" * 70)

for key, value in state_dict.items():

    if "classifier" in key:

        print(
            key,
            "| shape =", tuple(value.shape),
            "| mean =", value.float().mean().item(),
            "| std =", value.float().std().item(),
            "| min =", value.float().min().item(),
            "| max =", value.float().max().item()
        )

print("=" * 70)

In [ ]:
# ============================================================
# STEP 9 — ECGResNet LAYER-BY-LAYER ACTIVATION DIAGNOSTIC
# ============================================================

import torch

model.eval()

print("=" * 75)
print("ECGRESNET LAYER ACTIVATION DIAGNOSTIC")
print("=" * 75)

# ------------------------------------------------------------
# Use the existing ECG tensor
# ------------------------------------------------------------

if "x" not in globals():
    raise RuntimeError(
        "Variable 'x' is not available. "
        "Use the ECG tensor from your previous diagnostic."
    )

x_test = x.detach().clone().float()

print("\nOriginal input shape:", x_test.shape)

# Convert [12,1000] -> [1,12,1000]
if x_test.ndim == 2:
    x_test = x_test.unsqueeze(0)

# If accidentally [1,1000,12], transpose it
if x_test.shape[-1] == 12 and x_test.shape[-2] == 1000:
    x_test = x_test.transpose(-1, -2)

print("Model input shape:", x_test.shape)

# ------------------------------------------------------------
# Activation storage
# ------------------------------------------------------------

activations = {}

def hook_fn(name):

    def hook(module, inputs, output):

        if isinstance(output, tuple):
            output = output[0]

        activations[name] = output.detach().cpu()

    return hook


# ------------------------------------------------------------
# Register hooks
# ------------------------------------------------------------

hooks = []

for layer_name in [
    "stem",
    "block1",
    "block2",
    "block3",
    "block4",
    "classifier"
]:

    if hasattr(model, layer_name):

        layer = getattr(model, layer_name)

        hooks.append(
            layer.register_forward_hook(
                hook_fn(layer_name)
            )
        )

        print("Hooked:", layer_name)

# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

with torch.no_grad():

    logits = model(x_test)

# ------------------------------------------------------------
# Remove hooks
# ------------------------------------------------------------

for h in hooks:
    h.remove()

# ------------------------------------------------------------
# Analyze activations
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("ACTIVATION STATISTICS")
print("=" * 75)

for name, activation in activations.items():

    a = activation.float()

    print("\n" + name)
    print("-" * 60)

    print("Shape :", tuple(a.shape))
    print("Min   :", a.min().item())
    print("Max   :", a.max().item())
    print("Mean  :", a.mean().item())
    print("Std   :", a.std().item())

    near_zero = (
        (a.abs() < 1e-6).float().mean().item() * 100
    )

    print(
        "Near-zero:",
        f"{near_zero:.2f}%"
    )

# ------------------------------------------------------------
# Final logits
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL OUTPUT")
print("=" * 75)

print("Logits:")
print(logits.detach().cpu())

probs = torch.softmax(logits, dim=1)

print("\nProbabilities:")
print(probs.detach().cpu())

print("\nPredicted class:")
print(torch.argmax(probs, dim=1).item())

print("=" * 75)
print("DIAGNOSTIC COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# STEP 10 — CROSS-ECG REPRESENTATION VARIATION TEST
# ============================================================

import torch
import numpy as np

model.eval()

print("=" * 75)
print("CROSS-ECG REPRESENTATION VARIATION TEST")
print("=" * 75)

# ------------------------------------------------------------
# Find existing tensors safely
# ------------------------------------------------------------

tensor_candidates = []

for name, obj in list(globals().items()):

    if isinstance(obj, torch.Tensor):

        shape = tuple(obj.shape)

        # Look for ECG-like tensors
        if len(shape) >= 2:

            if 1000 in shape and 12 in shape:

                tensor_candidates.append(
                    (name, shape)
                )

print("\nECG-like tensors currently available:")

for name, shape in tensor_candidates:
    print(f"  {name:30s} {shape}")

# ------------------------------------------------------------
# Select a suitable tensor
# ------------------------------------------------------------

batch_tensor = None
batch_name = None

# Prefer a real batch/test dataset
preferred_names = [
    "X_test",
    "x_test",
    "X",
    "x",
    "test_X",
    "test_x",
    "ecg_batch",
    "batch_x"
]

for name in preferred_names:

    if name in globals():

        obj = globals()[name]

        if isinstance(obj, torch.Tensor):

            if 1000 in obj.shape and 12 in obj.shape:

                batch_tensor = obj
                batch_name = name
                break

# ------------------------------------------------------------
# If no preferred tensor, use the largest ECG-like tensor
# ------------------------------------------------------------

if batch_tensor is None and tensor_candidates:

    best_name, best_shape = max(
        tensor_candidates,
        key=lambda z: np.prod(z[1])
    )

    batch_tensor = globals()[best_name]
    batch_name = best_name

# ------------------------------------------------------------
# Stop if we only have a single ECG
# ------------------------------------------------------------

if batch_tensor is None:

    raise RuntimeError(
        "No ECG batch tensor was found.\n"
        "We need a tensor containing multiple ECG records."
    )

print("\nUsing tensor:", batch_name)
print("Original shape:", tuple(batch_tensor.shape))

# ------------------------------------------------------------
# Convert to [N, 12, 1000]
# ------------------------------------------------------------

data = batch_tensor.detach().clone().float()

# Case: [12,1000]
if data.ndim == 2:

    if data.shape == (12, 1000):

        data = data.unsqueeze(0)

    elif data.shape == (1000, 12):

        data = data.transpose(0, 1).unsqueeze(0)

# Case: [N,1000,12]
elif data.ndim == 3:

    if data.shape[-2:] == (1000, 12):

        data = data.transpose(1, 2)

# Case: [N,12,1000]
elif data.ndim == 3:

    pass

# Case: [N,1,12,1000]
elif data.ndim == 4:

    if data.shape[1] == 1:

        data = data.squeeze(1)

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

print("Converted shape:", tuple(data.shape))

if data.ndim != 3:

    raise RuntimeError(
        f"Expected 3D tensor [N,12,1000], got {data.shape}"
    )

if data.shape[1] != 12 or data.shape[2] != 1000:

    raise RuntimeError(
        f"Expected [N,12,1000], got {data.shape}"
    )

# ------------------------------------------------------------
# Use first 20 ECGs
# ------------------------------------------------------------

n = min(20, data.shape[0])

data = data[:n]

print("\nNumber of ECGs tested:", n)

# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

with torch.no_grad():

    logits = model(data)

    probabilities = torch.softmax(
        logits,
        dim=1
    )

# ------------------------------------------------------------
# Extract feature vectors
# ------------------------------------------------------------

features = None

def feature_hook(module, inputs, output):

    global features

    if isinstance(output, tuple):
        output = output[0]

    features = output.detach().cpu()


# Hook the layer immediately before classifier
hook = None

if hasattr(model, "classifier"):

    classifier = model.classifier

    # Usually classifier is Sequential
    if isinstance(classifier, torch.nn.Sequential):

        # Find the first Linear layer
        for layer in classifier:

            if isinstance(layer, torch.nn.Linear):

                hook = layer.register_forward_hook(
                    feature_hook
                )

                break

# Run again to capture features
with torch.no_grad():

    _ = model(data)

if hook is not None:
    hook.remove()

# ------------------------------------------------------------
# Print logits
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("LOGIT VARIATION")
print("=" * 75)

logit_cpu = logits.detach().cpu()

print("Shape:", tuple(logit_cpu.shape))

print(
    "Mean logit std across ECGs:",
    logit_cpu.std(dim=0).mean().item()
)

print(
    "Maximum logit std:",
    logit_cpu.std(dim=0).max().item()
)

print("\nLogits for each ECG:")

for i in range(n):

    print(
        f"ECG {i+1:02d}:",
        np.round(
            logit_cpu[i].numpy(),
            5
        )
    )

# ------------------------------------------------------------
# Probability variation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PROBABILITY VARIATION")
print("=" * 75)

prob_cpu = probabilities.detach().cpu()

prob_std = prob_cpu.std(dim=0)

print(
    "Mean probability std:",
    prob_std.mean().item()
)

print(
    "Maximum probability std:",
    prob_std.max().item()
)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

predictions = torch.argmax(
    prob_cpu,
    dim=1
)

print("\nPredicted classes:")

print(
    predictions.numpy()
)

print(
    "\nPrediction distribution:"
)

unique, counts = torch.unique(
    predictions,
    return_counts=True
)

for cls, count in zip(unique, counts):

    print(
        f"class {cls.item()}: {count.item()}"
    )

# ------------------------------------------------------------
# Feature variation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FEATURE VARIATION")
print("=" * 75)

if features is not None:

    f = features.float()

    print(
        "Feature shape:",
        tuple(f.shape)
    )

    feature_std_across_ecgs = f.std(
        dim=0
    )

    print(
        "Mean feature std across ECGs:",
        feature_std_across_ecgs.mean().item()
    )

    print(
        "Maximum feature std:",
        feature_std_across_ecgs.max().item()
    )

    # Pairwise distance from first ECG
    first = f[0]

    distances = torch.norm(
        f - first,
        dim=1
    )

    print(
        "\nFeature distance from ECG #1:"
    )

    for i, d in enumerate(distances):

        print(
            f"ECG {i+1:02d}: {d.item():.8f}"
        )

else:

    print(
        "Could not capture pre-classifier features."
    )

print("\n" + "=" * 75)
print("CROSS-ECG TEST COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# STEP 10A — LOAD MULTIPLE REAL PTB-XL ECG RECORDS
# ============================================================

import os
import glob
import numpy as np
import torch

try:
    import wfdb
except ImportError:
    !pip -q install wfdb
    import wfdb

print("=" * 75)
print("LOADING MULTIPLE PTB-XL ECG RECORDS")
print("=" * 75)

BASE_DIR = "/content/ptbxl"

# Find all .hea files
hea_files = sorted(
    glob.glob(
        os.path.join(BASE_DIR, "**", "*.hea"),
        recursive=True
    )
)

print("\nHeader files found:", len(hea_files))

if len(hea_files) == 0:
    raise RuntimeError(
        f"No .hea files found inside {BASE_DIR}"
    )

# ------------------------------------------------------------
# Load records that can actually be read
# ------------------------------------------------------------

ecg_records = []
record_names = []

for hea_path in hea_files:

    if len(ecg_records) >= 20:
        break

    record_path = os.path.splitext(hea_path)[0]

    try:

        signal, fields = wfdb.rdsamp(record_path)

        # Expected PTB-XL shape
        if signal.shape == (1000, 12):

            ecg_records.append(signal)
            record_names.append(
                os.path.basename(record_path)
            )

    except Exception as e:

        print(
            "Skipping:",
            record_path,
            "|",
            type(e).__name__
        )

print("\nSuccessfully loaded:", len(ecg_records))

if len(ecg_records) < 2:

    raise RuntimeError(
        "Could not load at least 2 ECG records."
    )

# ------------------------------------------------------------
# Stack into batch
# ------------------------------------------------------------

ecg_np = np.stack(
    ecg_records,
    axis=0
)

print(
    "Loaded NumPy shape:",
    ecg_np.shape
)

# ------------------------------------------------------------
# Convert [N,1000,12] -> [N,12,1000]
# ------------------------------------------------------------

ecg_batch = torch.tensor(
    ecg_np,
    dtype=torch.float32
).transpose(1, 2)

print(
    "Model input shape:",
    ecg_batch.shape
)

print(
    "dtype:",
    ecg_batch.dtype
)

# ------------------------------------------------------------
# Basic statistics
# ------------------------------------------------------------

print("\nBatch statistics:")

print(
    "Min:",
    ecg_batch.min().item()
)

print(
    "Max:",
    ecg_batch.max().item()
)

print(
    "Mean:",
    ecg_batch.mean().item()
)

print(
    "Std:",
    ecg_batch.std().item()
)

print("\nRecords loaded:")

for i, name in enumerate(record_names):

    print(
        f"{i+1:02d}: {name}"
    )

print("\n" + "=" * 75)
print("STEP 10A COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# STEP 10B — TRUE CROSS-ECG MODEL VARIATION TEST
# ============================================================

model.eval()

print("=" * 75)
print("TRUE CROSS-ECG MODEL VARIATION TEST")
print("=" * 75)

# ------------------------------------------------------------
# Use loaded real ECG batch
# ------------------------------------------------------------

x_batch = ecg_batch.to(
    next(model.parameters()).device
)

print(
    "\nInput shape:",
    tuple(x_batch.shape)
)

# ------------------------------------------------------------
# Capture features BEFORE final classifier
# ------------------------------------------------------------

captured_features = None

def capture_features(module, inputs, output):

    global captured_features

    if isinstance(inputs, tuple):

        captured_features = (
            inputs[0]
            .detach()
            .cpu()
        )

    else:

        captured_features = (
            inputs
            .detach()
            .cpu()
        )

# Find first Linear layer in classifier
hook = None

if hasattr(model, "classifier"):

    if isinstance(
        model.classifier,
        torch.nn.Sequential
    ):

        for layer in model.classifier:

            if isinstance(
                layer,
                torch.nn.Linear
            ):

                hook = layer.register_forward_hook(
                    capture_features
                )

                print(
                    "Hooked classifier input:",
                    layer
                )

                break

# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

with torch.no_grad():

    logits = model(x_batch)

if hook is not None:

    hook.remove()

probabilities = torch.softmax(
    logits,
    dim=1
)

# ------------------------------------------------------------
# LOGIT ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("1. LOGIT VARIATION")
print("=" * 75)

logits_cpu = logits.detach().cpu()

print(
    "Logit shape:",
    tuple(logits_cpu.shape)
)

logit_std = logits_cpu.std(
    dim=0,
    unbiased=False
)

print(
    "Mean logit std:",
    logit_std.mean().item()
)

print(
    "Maximum logit std:",
    logit_std.max().item()
)

print(
    "Minimum logit std:",
    logit_std.min().item()
)

print("\nFirst 10 ECG logits:")

for i in range(
    min(10, len(logits_cpu))
):

    print(
        f"{record_names[i]:15s}",
        np.round(
            logits_cpu[i].numpy(),
            5
        )
    )

# ------------------------------------------------------------
# PROBABILITY ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("2. PROBABILITY VARIATION")
print("=" * 75)

prob_cpu = probabilities.detach().cpu()

prob_std = prob_cpu.std(
    dim=0,
    unbiased=False
)

print(
    "Mean probability std:",
    prob_std.mean().item()
)

print(
    "Maximum probability std:",
    prob_std.max().item()
)

print("\nFirst 10 probability vectors:")

for i in range(
    min(10, len(prob_cpu))
):

    print(
        f"{record_names[i]:15s}",
        np.round(
            prob_cpu[i].numpy(),
            5
        )
    )

# ------------------------------------------------------------
# PREDICTIONS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("3. PREDICTION DISTRIBUTION")
print("=" * 75)

predictions = torch.argmax(
    prob_cpu,
    dim=1
)

for i in range(
    len(predictions)
):

    print(
        f"{record_names[i]:15s}"
        f" -> class {predictions[i].item()}"
    )

unique, counts = torch.unique(
    predictions,
    return_counts=True
)

print("\nDistribution:")

for cls, count in zip(
    unique,
    counts
):

    print(
        f"class {cls.item()}: "
        f"{count.item()} ECGs"
    )

# ------------------------------------------------------------
# FEATURE ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("4. PRE-CLASSIFIER FEATURE VARIATION")
print("=" * 75)

if captured_features is None:

    print(
        "WARNING: Features were not captured."
    )

else:

    features = captured_features.float()

    print(
        "Feature shape:",
        tuple(features.shape)
    )

    feature_std = features.std(
        dim=0,
        unbiased=False
    )

    print(
        "Mean feature std:",
        feature_std.mean().item()
    )

    print(
        "Maximum feature std:",
        feature_std.max().item()
    )

    # --------------------------------------------------------
    # Distance between every ECG and ECG #1
    # --------------------------------------------------------

    reference = features[0]

    distances = torch.norm(
        features - reference,
        dim=1
    )

    print(
        "\nFeature distance from ECG #1:"
    )

    for i, distance in enumerate(
        distances
    ):

        print(
            f"{record_names[i]:15s}"
            f" -> {distance.item():.8f}"
        )

# ------------------------------------------------------------
# FINAL VERDICT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 10B COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# STEP 10C — LOGIT + PROBABILITY VARIATION
# ============================================================

print("=" * 75)
print("STEP 10C — LOGIT AND PROBABILITY VARIATION")
print("=" * 75)

model.eval()

x_batch = ecg_batch.to(
    next(model.parameters()).device
)

with torch.no_grad():
    logits = model(x_batch)

    probabilities = torch.softmax(
        logits,
        dim=1
    )

logits = logits.cpu()
probabilities = probabilities.cpu()

# ------------------------------------------------------------
# LOGITS
# ------------------------------------------------------------

print("\nLOGIT STATISTICS")
print("-" * 75)

logit_std = logits.std(
    dim=0,
    unbiased=False
)

print("Logit shape:", tuple(logits.shape))

print(
    "Mean logit std:",
    logit_std.mean().item()
)

print(
    "Maximum logit std:",
    logit_std.max().item()
)

print(
    "Minimum logit std:",
    logit_std.min().item()
)

# ------------------------------------------------------------
# PROBABILITIES
# ------------------------------------------------------------

print("\nPROBABILITY STATISTICS")
print("-" * 75)

prob_std = probabilities.std(
    dim=0,
    unbiased=False
)

print(
    "Mean probability std:",
    prob_std.mean().item()
)

print(
    "Maximum probability std:",
    prob_std.max().item()
)

# ------------------------------------------------------------
# SHOW EVERY ECG
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PER-ECG OUTPUT")
print("=" * 75)

CLASS_NAMES = [
    "NORM",
    "STIC",
    "CD",
    "MI",
    "HYP"
]

for i in range(len(logits)):

    pred = torch.argmax(
        probabilities[i]
    ).item()

    print(
        f"\n{record_names[i]}"
    )

    print(
        "  Logits:",
        np.round(
            logits[i].numpy(),
            5
        )
    )

    print(
        "  Probabilities:",
        np.round(
            probabilities[i].numpy(),
            5
        )
    )

    print(
        "  Prediction:",
        pred,
        CLASS_NAMES[pred]
    )

# ------------------------------------------------------------
# RANGE OF EACH CLASS PROBABILITY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PROBABILITY RANGES")
print("=" * 75)

for c, name in enumerate(CLASS_NAMES):

    values = probabilities[:, c]

    print(
        f"{name:5s} | "
        f"min={values.min().item():.6f} | "
        f"max={values.max().item():.6f} | "
        f"std={values.std(unbiased=False).item():.6f}"
    )

print("\n" + "=" * 75)
print("STEP 10C COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# STEP 11 — CHECKPOINT + MODEL INTEGRITY
# ============================================================

print("=" * 75)
print("STEP 11 — CHECKPOINT / MODEL INTEGRITY")
print("=" * 75)

# ------------------------------------------------------------
# 1. CLASSIFIER WEIGHTS
# ------------------------------------------------------------

print("\n1. CLASSIFIER WEIGHTS")
print("-" * 75)

for name, param in model.named_parameters():

    if "classifier" in name:

        print(
            f"{name:30s} "
            f"shape={tuple(param.shape)} "
            f"mean={param.mean().item():.8f} "
            f"std={param.std().item():.8f} "
            f"min={param.min().item():.8f} "
            f"max={param.max().item():.8f}"
        )

# ------------------------------------------------------------
# 2. MODEL TRAIN / EVAL MODE
# ------------------------------------------------------------

print("\n2. MODEL MODE")
print("-" * 75)

print("model.training =", model.training)

model.eval()

print("After model.eval():", model.training)

# ------------------------------------------------------------
# 3. CHECKPOINT OBJECT
# ------------------------------------------------------------

print("\n3. CHECKPOINT INFORMATION")
print("-" * 75)

print("Checkpoint variable types:")

for name in [
    "checkpoint",
    "ckpt",
    "state_dict",
    "model_state_dict"
]:

    if name in globals():

        obj = globals()[name]

        print(
            name,
            "->",
            type(obj)
        )

# ------------------------------------------------------------
# 4. CHECK CURRENT MODEL PARAMETERS
# ------------------------------------------------------------

print("\n4. TOTAL MODEL PARAMETERS")
print("-" * 75)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Total parameters:",
    total_params
)

print(
    "Trainable parameters:",
    trainable_params
)

# ------------------------------------------------------------
# 5. CLASSIFIER BIAS
# ------------------------------------------------------------

print("\n5. CLASSIFIER BIAS")
print("-" * 75)

for name, param in model.named_parameters():

    if "classifier" in name and "bias" in name:

        print(
            name,
            "bias =",
            param.detach().cpu().numpy()
        )

# ------------------------------------------------------------
# 6. RE-RUN SAME ECG MULTIPLE TIMES
# ------------------------------------------------------------

print("\n6. DETERMINISM TEST")
print("-" * 75)

sample = ecg_batch[:1].to(
    next(model.parameters()).device
)

outputs = []

with torch.no_grad():

    for _ in range(5):

        outputs.append(
            model(sample).cpu()
        )

for i, output in enumerate(outputs):

    print(
        f"Run {i+1}:",
        np.round(
            output.numpy(),
            8
        )
    )

# ------------------------------------------------------------
# 7. DIFFERENT ECG PAIRWISE LOGIT DISTANCE
# ------------------------------------------------------------

print("\n7. LOGIT DISTANCE")
print("-" * 75)

with torch.no_grad():

    batch_logits = model(
        ecg_batch.to(
            next(model.parameters()).device
        )
    ).cpu()

reference_logit = batch_logits[0]

for i in range(
    min(20, len(batch_logits))
):

    distance = torch.norm(
        batch_logits[i] - reference_logit
    ).item()

    print(
        f"{record_names[i]:15s}"
        f" -> {distance:.10f}"
    )

print("\n" + "=" * 75)
print("STEP 11 COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# STEP 12 — CHECKPOINT FORENSICS
# ============================================================

print("=" * 75)
print("STEP 12 — CHECKPOINT FORENSICS")
print("=" * 75)

# ------------------------------------------------------------
# 1. CHECKPOINT KEYS
# ------------------------------------------------------------

print("\n1. CHECKPOINT KEYS")
print("-" * 75)

print("Number of checkpoint entries:", len(checkpoint))

for i, key in enumerate(checkpoint.keys()):

    print(f"{i:4d}: {key}")

    if i >= 30:
        print("... remaining keys omitted")
        break


# ------------------------------------------------------------
# 2. STATE_DICT KEYS
# ------------------------------------------------------------

print("\n2. STATE_DICT KEYS")
print("-" * 75)

print("Number of state_dict entries:", len(state_dict))

for i, key in enumerate(state_dict.keys()):

    print(f"{i:4d}: {key}")

    if i >= 30:
        print("... remaining keys omitted")
        break


# ------------------------------------------------------------
# 3. CHECKPOINT vs MODEL KEY MATCH
# ------------------------------------------------------------

print("\n3. KEY MATCH")
print("-" * 75)

model_keys = set(model.state_dict().keys())
ckpt_keys = set(state_dict.keys())

print("Model keys:", len(model_keys))
print("Checkpoint keys:", len(ckpt_keys))

print("\nMissing from checkpoint:")
print(
    sorted(model_keys - ckpt_keys)[:50]
)

print("\nExtra in checkpoint:")
print(
    sorted(ckpt_keys - model_keys)[:50]
)


# ------------------------------------------------------------
# 4. COMPARE ACTUAL PARAMETERS
# ------------------------------------------------------------

print("\n4. PARAMETER COMPARISON")
print("-" * 75)

common_keys = model_keys & ckpt_keys

print(
    "Common parameter keys:",
    len(common_keys)
)

differences = []

for key in common_keys:

    model_param = model.state_dict()[key].detach().cpu()
    ckpt_param = state_dict[key].detach().cpu()

    if model_param.shape != ckpt_param.shape:
        differences.append(
            (key, "SHAPE_MISMATCH")
        )
        continue

    diff = torch.max(
        torch.abs(model_param - ckpt_param)
    ).item()

    differences.append(
        (key, diff)
    )


print("\nFirst 30 parameter differences:")

for item in differences[:30]:

    print(item)


# ------------------------------------------------------------
# 5. CHECK CLASSIFIER WEIGHTS IN CHECKPOINT
# ------------------------------------------------------------

print("\n5. CHECKPOINT CLASSIFIER")
print("-" * 75)

for key in state_dict.keys():

    if "classifier" in key:

        tensor = state_dict[key].detach().cpu()

        print(
            f"{key:30s}"
            f" shape={tuple(tensor.shape)}"
            f" mean={tensor.mean().item():.8f}"
            f" std={tensor.std().item():.8f}"
            f" min={tensor.min().item():.8f}"
            f" max={tensor.max().item():.8f}"
        )


# ------------------------------------------------------------
# 6. CHECK FOR TRAINING METADATA
# ------------------------------------------------------------

print("\n6. CHECKPOINT METADATA")
print("-" * 75)

for key in checkpoint.keys():

    if key not in state_dict:

        value = checkpoint[key]

        print(
            key,
            "->",
            type(value)
        )

        if isinstance(
            value,
            (int, float, str, list, tuple)
        ):
            print(
                "   value:",
                value
            )


# ------------------------------------------------------------
# 7. FINAL CHECK
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 12 COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# STEP 13 — BATCHNORM / TRAIN-vs-EVAL DIAGNOSTIC
# ============================================================

print("=" * 75)
print("STEP 13 — BATCHNORM DIAGNOSTIC")
print("=" * 75)

import torch
import numpy as np

# ------------------------------------------------------------
# 1. FIND ALL BATCHNORM LAYERS
# ------------------------------------------------------------

print("\n1. BATCHNORM RUNNING STATISTICS")
print("-" * 75)

bn_layers = []

for name, module in model.named_modules():

    if isinstance(module, torch.nn.BatchNorm1d):

        bn_layers.append((name, module))

        running_mean = module.running_mean.detach().cpu()
        running_var = module.running_var.detach().cpu()

        print(f"\n{name}")
        print("  num_features :", module.num_features)
        print("  mean range   :",
              running_mean.min().item(),
              "to",
              running_mean.max().item())
        print("  var range    :",
              running_var.min().item(),
              "to",
              running_var.max().item())
        print("  mean avg     :", running_mean.mean().item())
        print("  var avg      :", running_var.mean().item())
        print("  batches seen :", module.num_batches_tracked.item())


print("\nTotal BatchNorm layers:", len(bn_layers))


# ------------------------------------------------------------
# 2. CREATE A BATCH OF THE 20 ECGs
# ------------------------------------------------------------

print("\n2. TEST BATCH")
print("-" * 75)

# x_test should already contain your ECG tensor.
# Expected shape for x_test:
# (1, 12, 1000)

test_batch = x_test.detach().clone()

# If only one ECG exists, use the previously discovered ECG tensor
# variables if available.

candidate_tensors = []

for name in ["x_test", "x", "ecg", "dummy_ecg"]:

    if name in globals():

        obj = globals()[name]

        if isinstance(obj, torch.Tensor):

            print(
                f"Found tensor: {name} -> {tuple(obj.shape)}"
            )

            candidate_tensors.append(
                (name, obj)
            )


# ------------------------------------------------------------
# 3. TEST CURRENT EVAL MODE
# ------------------------------------------------------------

print("\n3. CURRENT EVAL MODE")
print("-" * 75)

model.eval()

with torch.no_grad():

    out_eval = model(test_batch)

print("Input shape :", tuple(test_batch.shape))
print("Output shape:", tuple(out_eval.shape))

print("\nEval logits:")
print(out_eval.detach().cpu())


# ------------------------------------------------------------
# 4. TEST TRAIN MODE WITHOUT TRAINING
# ------------------------------------------------------------

print("\n4. TRAIN MODE — NO GRADIENTS / NO TRAINING")
print("-" * 75)

# IMPORTANT:
# This does NOT train the model.
# We are only checking BatchNorm behavior.

model.train()

with torch.no_grad():

    out_train = model(test_batch)

print("Train-mode logits:")
print(out_train.detach().cpu())


# ------------------------------------------------------------
# 5. COMPARE EVAL vs TRAIN OUTPUT
# ------------------------------------------------------------

print("\n5. EVAL vs TRAIN DIFFERENCE")
print("-" * 75)

eval_cpu = out_eval.detach().cpu()
train_cpu = out_train.detach().cpu()

difference = torch.abs(
    eval_cpu - train_cpu
)

print("Maximum difference:",
      difference.max().item())

print("Mean difference:",
      difference.mean().item())


# ------------------------------------------------------------
# 6. PROBABILITY COMPARISON
# ------------------------------------------------------------

print("\n6. PROBABILITY COMPARISON")
print("-" * 75)

eval_probs = torch.softmax(
    eval_cpu,
    dim=1
)

train_probs = torch.softmax(
    train_cpu,
    dim=1
)

print("\nEval probabilities:")
print(eval_probs)

print("\nTrain-mode probabilities:")
print(train_probs)


# ------------------------------------------------------------
# 7. PREDICTION COMPARISON
# ------------------------------------------------------------

print("\n7. PREDICTION COMPARISON")
print("-" * 75)

eval_pred = torch.argmax(
    eval_probs,
    dim=1
)

train_pred = torch.argmax(
    train_probs,
    dim=1
)

print("Eval prediction :",
      eval_pred.tolist())

print("Train prediction:",
      train_pred.tolist())


# ------------------------------------------------------------
# 8. RESTORE EVAL MODE
# ------------------------------------------------------------

model.eval()

print("\n" + "=" * 75)
print("STEP 13 COMPLETE")
print("=" * 75)

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("=" * 75)
print("STEP 14 — CONTROLLED BATCHNORM EXPERIMENT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Find the ECG tensor
# ------------------------------------------------------------
candidate_names = ["x_test", "x", "ecg", "dummy_ecg"]

ecg_tensor = None
ecg_name = None

for name in candidate_names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, torch.Tensor) and obj.ndim == 3:
            ecg_tensor = obj
            ecg_name = name
            break

if ecg_tensor is None:
    raise RuntimeError("Could not find an ECG tensor.")

print(f"\nUsing tensor: {ecg_name}")
print(f"Original shape: {tuple(ecg_tensor.shape)}")

# Make sure shape is [N, 12, 1000]
if ecg_tensor.shape[0] == 12 and ecg_tensor.shape[1] == 1000:
    ecg_tensor = ecg_tensor.unsqueeze(0)

print(f"Final shape:    {tuple(ecg_tensor.shape)}")

# Use up to 20 ECGs
N = min(20, ecg_tensor.shape[0])
test_ecgs = ecg_tensor[:N].clone()

print(f"ECGs tested: {N}")


# ------------------------------------------------------------
# 2. Helper: get logits
# ------------------------------------------------------------
def get_logits(model, x):
    with torch.no_grad():
        out = model(x)

        # Handle tuple/list outputs
        if isinstance(out, (tuple, list)):
            out = out[0]

        return out.detach().cpu()


# ------------------------------------------------------------
# 3. EXPERIMENT A — Normal EVAL mode
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("A. NORMAL EVAL MODE")
print("=" * 75)

model.eval()

eval_logits = get_logits(model, test_ecgs)
eval_probs = torch.softmax(eval_logits, dim=1)
eval_preds = torch.argmax(eval_logits, dim=1)

print("Logits shape:", tuple(eval_logits.shape))
print("Predictions:", eval_preds.tolist())

print("\nPrediction distribution:")
unique, counts = torch.unique(eval_preds, return_counts=True)

for cls, count in zip(unique.tolist(), counts.tolist()):
    print(f"Class {cls}: {count} ECGs")


# ------------------------------------------------------------
# 4. EXPERIMENT B — TRAIN mode, NO gradients
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("B. TRAIN MODE — NO GRADIENTS")
print("=" * 75)

model.train()

train_logits = get_logits(model, test_ecgs)
train_probs = torch.softmax(train_logits, dim=1)
train_preds = torch.argmax(train_logits, dim=1)

print("Logits shape:", tuple(train_logits.shape))
print("Predictions:", train_preds.tolist())

print("\nPrediction distribution:")
unique, counts = torch.unique(train_preds, return_counts=True)

for cls, count in zip(unique.tolist(), counts.tolist()):
    print(f"Class {cls}: {count} ECGs")


# ------------------------------------------------------------
# 5. Compare EVAL vs TRAIN
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("C. EVAL vs TRAIN")
print("=" * 75)

logit_diff = torch.abs(eval_logits - train_logits)

print(f"Maximum logit difference : {logit_diff.max().item():.8f}")
print(f"Mean logit difference    : {logit_diff.mean().item():.8f}")

prediction_changes = (eval_preds != train_preds).sum().item()

print(f"Prediction changes       : {prediction_changes}/{N}")


# ------------------------------------------------------------
# 6. Per-ECG comparison
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("D. PER-ECG COMPARISON")
print("=" * 75)

for i in range(N):
    print(
        f"ECG {i+1:02d} | "
        f"Eval={eval_preds[i].item()} | "
        f"Train={train_preds[i].item()} | "
        f"LogitDiff={logit_diff[i].max().item():.6f}"
    )


# ------------------------------------------------------------
# 7. Inspect BatchNorm layers
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("E. BATCHNORM RUNNING STATISTICS")
print("=" * 75)

bn_layers = []

for name, module in model.named_modules():
    if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
        bn_layers.append((name, module))

print(f"BatchNorm layers found: {len(bn_layers)}")

for name, bn in bn_layers:

    running_mean = bn.running_mean.detach().cpu()
    running_var = bn.running_var.detach().cpu()

    print(f"\n{name}")
    print(f"  features     : {bn.num_features}")
    print(f"  running mean : {running_mean.mean().item():.8f}")
    print(f"  running var  : {running_var.mean().item():.8f}")
    print(f"  num batches  : {bn.num_batches_tracked.item()}")


# ------------------------------------------------------------
# 8. Restore SAFE inference state
# ------------------------------------------------------------
model.eval()

print("\n" + "=" * 75)
print("STEP 14 COMPLETE")
print("=" * 75)

In [ ]:
import torch
import torch.nn as nn
import copy

print("=" * 75)
print("STEP 15 — BATCHNORM RUNNING-STATISTICS SENSITIVITY")
print("=" * 75)

model.eval()

# ------------------------------------------------------------
# Save original model state
# ------------------------------------------------------------
original_state = copy.deepcopy(model.state_dict())

# ------------------------------------------------------------
# Find ECG
# ------------------------------------------------------------
x = x_test.clone()

if x.ndim == 2:
    x = x.unsqueeze(0)

print(f"\nInput shape: {tuple(x.shape)}")


# ------------------------------------------------------------
# A. ORIGINAL RUNNING STATISTICS
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("A. ORIGINAL BATCHNORM STATISTICS")
print("=" * 75)

with torch.no_grad():
    original_logits = model(x)

    if isinstance(original_logits, (tuple, list)):
        original_logits = original_logits[0]

original_probs = torch.softmax(original_logits, dim=1)
original_pred = torch.argmax(original_logits, dim=1)

print("Logits       :", original_logits.cpu().numpy())
print("Probabilities:", original_probs.cpu().numpy())
print("Prediction   :", original_pred.item())


# ------------------------------------------------------------
# B. RESET ONLY BATCHNORM RUNNING STATISTICS
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("B. RESET BATCHNORM RUNNING STATISTICS")
print("=" * 75)

bn_count = 0

for module in model.modules():

    if isinstance(module, (nn.BatchNorm1d,
                           nn.BatchNorm2d,
                           nn.BatchNorm3d)):

        module.reset_running_stats()
        bn_count += 1

print(f"Reset {bn_count} BatchNorm layers.")

model.eval()

with torch.no_grad():
    reset_logits = model(x)

    if isinstance(reset_logits, (tuple, list)):
        reset_logits = reset_logits[0]

reset_probs = torch.softmax(reset_logits, dim=1)
reset_pred = torch.argmax(reset_logits, dim=1)

print("Logits       :", reset_logits.cpu().numpy())
print("Probabilities:", reset_probs.cpu().numpy())
print("Prediction   :", reset_pred.item())


# ------------------------------------------------------------
# C. COMPARISON
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("C. ORIGINAL vs RESET")
print("=" * 75)

diff = torch.abs(original_logits - reset_logits)

print(f"Maximum logit difference : {diff.max().item():.8f}")
print(f"Mean logit difference    : {diff.mean().item():.8f}")

print(f"\nOriginal prediction : {original_pred.item()}")
print(f"Reset prediction    : {reset_pred.item()}")

if original_pred.item() != reset_pred.item():
    print("\n⚠️ PREDICTION CHANGED")
    print("BatchNorm running statistics have a major effect.")
else:
    print("\nPrediction unchanged.")


# ------------------------------------------------------------
# D. RESTORE ORIGINAL MODEL
# ------------------------------------------------------------
model.load_state_dict(original_state)
model.eval()

print("\nOriginal model state restored.")

print("\n" + "=" * 75)
print("STEP 15 COMPLETE")
print("=" * 75)

In [ ]:
import torch
import torch.nn as nn
import copy

print("=" * 75)
print("STEP 16 — BATCHNORM RECALIBRATION EXPERIMENT")
print("=" * 75)

# ------------------------------------------------------------
# SAVE ORIGINAL MODEL
# ------------------------------------------------------------
original_state = copy.deepcopy(model.state_dict())

model.eval()

# ------------------------------------------------------------
# TEST ECG
# ------------------------------------------------------------
test_ecg = x_test.clone()

if test_ecg.ndim == 2:
    test_ecg = test_ecg.unsqueeze(0)

print(f"\nTest ECG shape: {tuple(test_ecg.shape)}")


# ------------------------------------------------------------
# ORIGINAL PREDICTION
# ------------------------------------------------------------
with torch.no_grad():

    original_output = model(test_ecg)

    if isinstance(original_output, (tuple, list)):
        original_output = original_output[0]

original_probs = torch.softmax(original_output, dim=1)
original_pred = torch.argmax(original_output, dim=1)

print("\nOriginal model:")
print("Logits       :", original_output.cpu().numpy())
print("Probabilities:", original_probs.cpu().numpy())
print("Prediction   :", original_pred.item())


# ------------------------------------------------------------
# FIND AVAILABLE ECG TENSORS
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("AVAILABLE ECG-LIKE TENSORS")
print("=" * 75)

candidate_tensors = {}

for name, value in globals().items():

    if isinstance(value, torch.Tensor):

        shape = tuple(value.shape)

        # Look for ECG-like tensors
        if len(shape) == 3 and shape[-2:] == (12, 1000):
            candidate_tensors[name] = value

            print(f"{name:20s} -> {shape}")


# ------------------------------------------------------------
# SELECT CALIBRATION DATA
# ------------------------------------------------------------
calibration_data = None

# Prefer a tensor containing multiple ECGs
for name, tensor in candidate_tensors.items():

    if tensor.shape[0] > 1:
        calibration_data = tensor
        print(f"\nUsing calibration tensor: {name}")
        break

# If no multi-ECG tensor exists, don't fake calibration
if calibration_data is None:

    print("\n⚠️ No multi-ECG calibration tensor found.")

    print("""
We need multiple representative ECGs before recalibrating
BatchNorm statistics.

The experiment will NOT modify the original model.
""")

    model.load_state_dict(original_state)
    model.eval()

else:

    print(f"Calibration data shape: {tuple(calibration_data.shape)}")


    # --------------------------------------------------------
    # RESET BN STATISTICS
    # --------------------------------------------------------
    bn_layers = []

    for module in model.modules():

        if isinstance(module,
                      (nn.BatchNorm1d,
                       nn.BatchNorm2d,
                       nn.BatchNorm3d)):

            module.reset_running_stats()
            bn_layers.append(module)

    print(f"\nReset {len(bn_layers)} BatchNorm layers.")


    # --------------------------------------------------------
    # RECALIBRATE BN
    # --------------------------------------------------------
    #
    # IMPORTANT:
    # We update ONLY BatchNorm running statistics.
    # No gradients.
    # No weight updates.
    #
    # --------------------------------------------------------

    model.train()

    with torch.no_grad():

        # Process calibration ECGs in small batches
        batch_size = min(8, calibration_data.shape[0])

        for start in range(0,
                           calibration_data.shape[0],
                           batch_size):

            batch = calibration_data[
                start:start + batch_size
            ]

            _ = model(batch)

    print("BatchNorm recalibration complete.")


    # --------------------------------------------------------
    # EVALUATION
    # --------------------------------------------------------
    model.eval()

    with torch.no_grad():

        recalibrated_output = model(test_ecg)

        if isinstance(recalibrated_output, (tuple, list)):
            recalibrated_output = recalibrated_output[0]

    recalibrated_probs = torch.softmax(
        recalibrated_output,
        dim=1
    )

    recalibrated_pred = torch.argmax(
        recalibrated_output,
        dim=1
    )


    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------
    print("\n" + "=" * 75)
    print("RECALIBRATED MODEL")
    print("=" * 75)

    print("Logits       :",
          recalibrated_output.cpu().numpy())

    print("Probabilities:",
          recalibrated_probs.cpu().numpy())

    print("Prediction   :",
          recalibrated_pred.item())


    # --------------------------------------------------------
    # COMPARISON
    # --------------------------------------------------------
    diff = torch.abs(
        original_output - recalibrated_output
    )

    print("\n" + "=" * 75)
    print("ORIGINAL vs RECALIBRATED")
    print("=" * 75)

    print(
        f"Maximum logit difference : "
        f"{diff.max().item():.8f}"
    )

    print(
        f"Mean logit difference    : "
        f"{diff.mean().item():.8f}"
    )

    print(
        f"\nOriginal prediction      : "
        f"{original_pred.item()}"
    )

    print(
        f"Recalibrated prediction  : "
        f"{recalibrated_pred.item()}"
    )


    if original_pred.item() != recalibrated_pred.item():

        print("\n⚠️ PREDICTION CHANGED")
        print(
            "BN recalibration changes the classification."
        )

    else:

        print("\nPrediction remained unchanged.")


    # --------------------------------------------------------
    # RESTORE ORIGINAL MODEL
    # --------------------------------------------------------
    model.load_state_dict(original_state)
    model.eval()

    print("\nOriginal model restored.")


print("\n" + "=" * 75)
print("STEP 16 COMPLETE")
print("=" * 75)

In [30]:
import torch
import torch.nn as nn

print("=" * 75)
print("STEP 17 — FEATURE REPRESENTATION & CLASSIFIER DIAGNOSTIC")
print("=" * 75)

model.eval()

# ------------------------------------------------------------
# 1. FIND THE FINAL CLASSIFIER LAYER
# ------------------------------------------------------------

print("\n1. CLASSIFIER STRUCTURE")
print("-" * 75)

for name, module in model.named_modules():

    if "classifier" in name:
        print(f"{name:30s} -> {module}")


# ------------------------------------------------------------
# 2. FIND FINAL LINEAR LAYER
# ------------------------------------------------------------

final_linear = None
final_name = None

for name, module in model.named_modules():

    if isinstance(module, nn.Linear):

        final_linear = module
        final_name = name

print("\nFinal Linear layer:")
print("Name :", final_name)
print("Shape:", tuple(final_linear.weight.shape))


# ------------------------------------------------------------
# 3. CAPTURE FEATURES BEFORE FINAL CLASSIFIER
# ------------------------------------------------------------

features = {}

def hook_fn(module, inputs, output):

    features["input"] = inputs[0].detach().clone()
    features["output"] = output.detach().clone()


hook = final_linear.register_forward_hook(hook_fn)


# ------------------------------------------------------------
# 4. FORWARD PASS
# ------------------------------------------------------------

with torch.no_grad():

    output = model(test_ecg)

    if isinstance(output, (tuple, list)):
        output = output[0]


hook.remove()


# ------------------------------------------------------------
# 5. FEATURE ANALYSIS
# ------------------------------------------------------------

classifier_input = features["input"]

print("\n2. CLASSIFIER INPUT FEATURES")
print("-" * 75)

print("Feature shape:",
      tuple(classifier_input.shape))

print("Feature mean :",
      classifier_input.mean().item())

print("Feature std  :",
      classifier_input.std().item())

print("Feature min  :",
      classifier_input.min().item())

print("Feature max  :",
      classifier_input.max().item())

print("Feature L2 norm:",
      torch.norm(classifier_input).item())


# ------------------------------------------------------------
# 6. CLASSIFIER WEIGHT ANALYSIS
# ------------------------------------------------------------

weights = final_linear.weight.detach()
bias = final_linear.bias.detach()

print("\n3. FINAL CLASSIFIER WEIGHTS")
print("-" * 75)

print("Weight shape:",
      tuple(weights.shape))

print("Weight mean:",
      weights.mean().item())

print("Weight std:",
      weights.std().item())

print("Weight min:",
      weights.min().item())

print("Weight max:",
      weights.max().item())

print("\nBias:")
print(bias.cpu().numpy())


# ------------------------------------------------------------
# 7. PER-CLASS WEIGHT NORMS
# ------------------------------------------------------------

print("\n4. PER-CLASS WEIGHT NORMS")
print("-" * 75)

for i in range(weights.shape[0]):

    norm = torch.norm(weights[i]).item()

    print(
        f"Class {i}: "
        f"weight L2 norm = {norm:.8f}"
    )


# ------------------------------------------------------------
# 8. CLASSIFIER CONTRIBUTION
# ------------------------------------------------------------

print("\n5. CLASSIFIER LOGIT ANALYSIS")
print("-" * 75)

manual_logits = (
    classifier_input @ weights.T
) + bias

print("Model logits:")
print(output.cpu().numpy())

print("\nManual logits:")
print(manual_logits.cpu().numpy())

print(
    "\nMaximum manual-vs-model difference:",
    torch.max(
        torch.abs(output - manual_logits)
    ).item()
)


# ------------------------------------------------------------
# 9. SOFTMAX ENTROPY
# ------------------------------------------------------------

probs = torch.softmax(output, dim=1)

entropy = -torch.sum(
    probs * torch.log(probs + 1e-12),
    dim=1
)

max_entropy = torch.log(
    torch.tensor(float(probs.shape[1]))
)

print("\n6. PREDICTION CONFIDENCE")
print("-" * 75)

print("Probabilities:")
print(probs.cpu().numpy())

print("\nEntropy:",
      entropy.item())

print("Maximum possible entropy for 5 classes:",
      max_entropy.item())

print(
    "Normalized entropy:",
    (entropy / max_entropy).item()
)


# ------------------------------------------------------------
# 10. TOP-1 vs TOP-2 MARGIN
# ------------------------------------------------------------

sorted_probs, indices = torch.sort(
    probs,
    descending=True
)

top1 = sorted_probs[0, 0].item()
top2 = sorted_probs[0, 1].item()

print("\n7. CLASS MARGIN")
print("-" * 75)

print("Top-1 class:",
      indices[0, 0].item())

print("Top-1 probability:",
      top1)

print("Top-2 class:",
      indices[0, 1].item())

print("Top-2 probability:",
      top2)

print("Probability margin:",
      top1 - top2)


print("\n" + "=" * 75)
print("STEP 17 COMPLETE")
print("=" * 75)

STEP 17 — FEATURE REPRESENTATION & CLASSIFIER DIAGNOSTIC

1. CLASSIFIER STRUCTURE
---------------------------------------------------------------------------
classifier                     -> Sequential(
  (0): Linear(in_features=512, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=128, out_features=5, bias=True)
)
classifier.0                   -> Linear(in_features=512, out_features=128, bias=True)
classifier.1                   -> ReLU()
classifier.2                   -> Dropout(p=0.3, inplace=False)
classifier.3                   -> Linear(in_features=128, out_features=5, bias=True)

Final Linear layer:
Name : classifier.3
Shape: (5, 128)


NameError: name 'test_ecg' is not defined

In [29]:
import torch
import torch.nn as nn

print("=" * 75)
print("STEP 18 — FEATURE COLLAPSE / FEATURE DIVERSITY TEST")
print("=" * 75)

model.eval()

# ------------------------------------------------------------
# 1. FIND FINAL LINEAR LAYER
# ------------------------------------------------------------

final_linear = None
final_name = None

for name, module in model.named_modules():

    if isinstance(module, nn.Linear):
        final_linear = module
        final_name = name

print("\n1. FINAL CLASSIFIER")
print("-" * 75)

print("Layer:", final_name)
print("Input features:", final_linear.in_features)
print("Output classes:", final_linear.out_features)


# ------------------------------------------------------------
# 2. CAPTURE CLASSIFIER INPUT FEATURES
# ------------------------------------------------------------

captured = {}

def hook_fn(module, inputs, output):

    captured["features"] = inputs[0].detach().clone()

hook = final_linear.register_forward_hook(hook_fn)


# ------------------------------------------------------------
# 3. SELECT CALIBRATION / ECG BATCH
# ------------------------------------------------------------

print("\n2. INPUT BATCH")
print("-" * 75)

# ecg_batch was discovered in Step 16
batch = ecg_batch

print("Batch shape:", tuple(batch.shape))


# ------------------------------------------------------------
# 4. FORWARD PASS
# ------------------------------------------------------------

with torch.no_grad():

    logits = model(batch)

    if isinstance(logits, (tuple, list)):
        logits = logits[0]

features = captured["features"]

hook.remove()


# ------------------------------------------------------------
# 5. FEATURE SHAPE
# ------------------------------------------------------------

print("\n3. FEATURE REPRESENTATION")
print("-" * 75)

print("Feature tensor shape:",
      tuple(features.shape))

print("Expected:",
      f"(batch_size, {final_linear.in_features})")


# ------------------------------------------------------------
# 6. PER-SAMPLE FEATURE NORMS
# ------------------------------------------------------------

feature_norms = torch.norm(
    features,
    dim=1
)

print("\n4. PER-SAMPLE FEATURE NORMS")
print("-" * 75)

for i, value in enumerate(feature_norms):

    print(
        f"ECG {i:02d}: "
        f"{value.item():.8f}"
    )

print("\nNorm statistics:")
print("Mean:", feature_norms.mean().item())
print("Std :", feature_norms.std().item())
print("Min :", feature_norms.min().item())
print("Max :", feature_norms.max().item())


# ------------------------------------------------------------
# 7. FEATURE VARIANCE ACROSS ECGs
# ------------------------------------------------------------

feature_mean = features.mean(dim=0)

feature_std = features.std(dim=0)

print("\n5. FEATURE VARIATION ACROSS ECGs")
print("-" * 75)

print("Mean absolute feature value:",
      features.abs().mean().item())

print("Mean feature std:",
      feature_std.mean().item())

print("Maximum feature std:",
      feature_std.max().item())

print("Minimum feature std:",
      feature_std.min().item())


# ------------------------------------------------------------
# 8. HOW SIMILAR ARE THE ECG REPRESENTATIONS?
# ------------------------------------------------------------

normalized_features = torch.nn.functional.normalize(
    features,
    p=2,
    dim=1
)

cosine_matrix = normalized_features @ normalized_features.T

# Remove diagonal
mask = ~torch.eye(
    cosine_matrix.shape[0],
    dtype=torch.bool,
    device=cosine_matrix.device
)

pairwise_cosine = cosine_matrix[mask]

print("\n6. FEATURE SIMILARITY")
print("-" * 75)

print("Mean pairwise cosine similarity:",
      pairwise_cosine.mean().item())

print("Minimum cosine similarity:",
      pairwise_cosine.min().item())

print("Maximum cosine similarity:",
      pairwise_cosine.max().item())


# ------------------------------------------------------------
# 9. LOGIT VARIATION ACROSS ECGs
# ------------------------------------------------------------

print("\n7. LOGIT VARIATION")
print("-" * 75)

print("Logits shape:",
      tuple(logits.shape))

print("Logit mean:",
      logits.mean().item())

print("Logit std:",
      logits.std().item())

print("Logit minimum:",
      logits.min().item())

print("Logit maximum:",
      logits.max().item())


# ------------------------------------------------------------
# 10. PREDICTIONS
# ------------------------------------------------------------

probabilities = torch.softmax(
    logits,
    dim=1
)

predictions = torch.argmax(
    probabilities,
    dim=1
)

print("\n8. PREDICTIONS")
print("-" * 75)

print("Predictions:",
      predictions.cpu().tolist())

print("\nMaximum probabilities:")

for i in range(len(predictions)):

    print(
        f"ECG {i:02d}: "
        f"class={predictions[i].item()} "
        f"confidence={probabilities[i].max().item():.6f}"
    )


# ------------------------------------------------------------
# 11. FEATURE COLLAPSE CHECK
# ------------------------------------------------------------

print("\n9. FEATURE COLLAPSE CHECK")
print("-" * 75)

if feature_norms.mean().item() < 0.5:

    print("⚠️ Feature magnitudes are small.")

else:

    print("Feature magnitudes are not extremely small.")


if pairwise_cosine.mean().item() > 0.95:

    print("⚠️ Features are highly similar across ECGs.")

else:

    print("Features show meaningful variation across ECGs.")


if feature_std.mean().item() < 0.005:

    print("⚠️ Very low feature variance detected.")

else:

    print("Feature variance is present.")


print("\n" + "=" * 75)
print("STEP 18 COMPLETE")
print("=" * 75)

STEP 18 — FEATURE COLLAPSE / FEATURE DIVERSITY TEST

1. FINAL CLASSIFIER
---------------------------------------------------------------------------
Layer: classifier.3
Input features: 128
Output classes: 5

2. INPUT BATCH
---------------------------------------------------------------------------


NameError: name 'ecg_batch' is not defined

In [28]:
import torch
import torch.nn as nn

print("=" * 75)
print("STEP 19 — REPRESENTATION FLOW / ACTIVATION COLLAPSE ANALYSIS")
print("=" * 75)

model.eval()

# ------------------------------------------------------------
# FIND THE ECG INPUT
# ------------------------------------------------------------

batch = ecg_batch

print("\nInput shape:", tuple(batch.shape))


# ------------------------------------------------------------
# REGISTER HOOKS
# ------------------------------------------------------------

activation_stats = {}

def make_hook(name):

    def hook(module, inputs, output):

        if isinstance(output, (tuple, list)):
            output = output[0]

        if not torch.is_tensor(output):
            return

        x = output.detach()

        activation_stats[name] = {
            "shape": tuple(x.shape),
            "mean": x.mean().item(),
            "std": x.std().item(),
            "abs_mean": x.abs().mean().item(),
            "min": x.min().item(),
            "max": x.max().item(),
            "zero_fraction": (
                (x == 0).float().mean().item()
            )
        }

    return hook


hooks = []


# ------------------------------------------------------------
# IMPORTANT LAYERS
# ------------------------------------------------------------

for name, module in model.named_modules():

    # We want major blocks rather than every tiny operation
    if (
        isinstance(module, nn.BatchNorm1d)
        or
        isinstance(module, nn.Conv1d)
        or
        isinstance(module, nn.Linear)
        or
        "stem" in name.lower()
        or
        "block" in name.lower()
    ):

        hooks.append(
            module.register_forward_hook(
                make_hook(name)
            )
        )


# ------------------------------------------------------------
# FORWARD PASS
# ------------------------------------------------------------

with torch.no_grad():

    output = model(batch)


# ------------------------------------------------------------
# REMOVE HOOKS
# ------------------------------------------------------------

for hook in hooks:
    hook.remove()


# ------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("ACTIVATION STATISTICS")
print("=" * 75)

for name, stats in activation_stats.items():

    print("\n" + "-" * 75)

    print("Layer:", name)

    print("Shape:",
          stats["shape"])

    print("Mean:",
          f'{stats["mean"]:.8f}')

    print("Std:",
          f'{stats["std"]:.8f}')

    print("Abs mean:",
          f'{stats["abs_mean"]:.8f}')

    print("Min:",
          f'{stats["min"]:.8f}')

    print("Max:",
          f'{stats["max"]:.8f}')

    print("Zero fraction:",
          f'{stats["zero_fraction"]:.6f}')


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 19 COMPLETE")
print("=" * 75)

STEP 19 — REPRESENTATION FLOW / ACTIVATION COLLAPSE ANALYSIS


NameError: name 'ecg_batch' is not defined

In [27]:
print(type(model))
print("Model is ready:", model is not None)

<class '__main__.ECGResNet'>
Model is ready: True


In [26]:
# ================================================================
# STEP 20 — LAYER-BY-LAYER REPRESENTATION DIVERSITY ANALYSIS
# ================================================================

import torch
import torch.nn as nn
import numpy as np

print("=" * 75)
print("STEP 20 — LAYER-BY-LAYER REPRESENTATION DIVERSITY ANALYSIS")
print("=" * 75)

model.eval()

# ------------------------------------------------
# 1. Find a suitable ECG batch
# ------------------------------------------------

candidate_tensors = {
    "ecg_batch": globals().get("ecg_batch"),
    "test_batch": globals().get("test_batch"),
    "x_batch": globals().get("x_batch"),
    "batch_tensor": globals().get("batch_tensor"),
    "x_test": globals().get("x_test"),
}

ecg_input = None

for name, tensor in candidate_tensors.items():
    if tensor is not None and isinstance(tensor, torch.Tensor):
        if tensor.ndim == 3 and tensor.shape[1] == 12:
            ecg_input = tensor
            print(f"Using tensor: {name}")
            print(f"Input shape: {tuple(tensor.shape)}")
            break

if ecg_input is None:
    raise RuntimeError(
        "Could not automatically find an ECG tensor with shape (N, 12, length)."
    )

# Use first 20 ECGs
ecg_input = ecg_input[:20].to(next(model.parameters()).device)


# ------------------------------------------------
# 2. Capture major layer outputs
# ------------------------------------------------

activations = {}

hooks = []

def make_hook(name):
    def hook(module, inp, output):
        if isinstance(output, torch.Tensor):
            activations[name] = output.detach().cpu()
    return hook


# Register hooks on major model components
for name, module in model.named_modules():

    # Only major computational blocks
    if (
        name in [
            "stem",
            "block1",
            "block2",
            "block3",
            "block4",
            "classifier",
        ]
        or name.endswith(".shortcut")
    ):
        hooks.append(
            module.register_forward_hook(make_hook(name))
        )


# ------------------------------------------------
# 3. Forward pass
# ------------------------------------------------

with torch.no_grad():
    _ = model(ecg_input)


# Remove hooks
for h in hooks:
    h.remove()


# ------------------------------------------------
# 4. Flatten each representation
# ------------------------------------------------

def flatten_representation(x):
    """
    Convert:
        (N, C, L)
    into:
        (N, C*L)

    Also handles:
        (N, features)
    """
    return x.reshape(x.shape[0], -1)


# ------------------------------------------------
# 5. Calculate inter-ECG similarity
# ------------------------------------------------

print("\n" + "=" * 75)
print("INTER-ECG REPRESENTATION SIMILARITY")
print("=" * 75)

results = []

for name, activation in activations.items():

    if activation.shape[0] != len(ecg_input):
        continue

    features = flatten_representation(activation).float()

    # Normalize each ECG representation
    normalized = torch.nn.functional.normalize(
        features,
        p=2,
        dim=1
    )

    # Pairwise cosine similarity
    similarity_matrix = normalized @ normalized.T

    # Remove diagonal
    mask = ~torch.eye(
        similarity_matrix.shape[0],
        dtype=torch.bool
    )

    pairwise = similarity_matrix[mask]

    mean_similarity = pairwise.mean().item()
    min_similarity = pairwise.min().item()
    max_similarity = pairwise.max().item()

    # Feature variation across ECGs
    feature_std = features.std(dim=0)
    mean_feature_std = feature_std.mean().item()
    max_feature_std = feature_std.max().item()

    results.append({
        "layer": name,
        "mean_cosine": mean_similarity,
        "min_cosine": min_similarity,
        "max_cosine": max_similarity,
        "mean_feature_std": mean_feature_std,
        "max_feature_std": max_feature_std,
    })


# ------------------------------------------------
# 6. Print results
# ------------------------------------------------

for r in results:

    print("\n" + "-" * 75)
    print(f"Layer: {r['layer']}")
    print("-" * 75)

    print(f"Mean cosine similarity : {r['mean_cosine']:.8f}")
    print(f"Minimum similarity     : {r['min_cosine']:.8f}")
    print(f"Maximum similarity     : {r['max_cosine']:.8f}")

    print(f"Mean feature std       : {r['mean_feature_std']:.8f}")
    print(f"Maximum feature std    : {r['max_feature_std']:.8f}")


# ------------------------------------------------
# 7. Interpretation
# ------------------------------------------------

print("\n" + "=" * 75)
print("STEP 20 INTERPRETATION")
print("=" * 75)

print("""
Interpretation:

Cosine similarity close to 1.0 means different ECGs
have very similar representations.

A healthy discriminative representation should show
meaningful variation between different ECG samples.

The most important question is:

WHERE does similarity suddenly increase?

If similarity is already extremely high at the INPUT,
the ECG samples themselves may be nearly identical.

If similarity becomes high inside a specific block,
that block may be destroying discriminative information.

If similarity remains reasonable until the final feature
layer and then becomes extremely high, the feature
projection/head may be responsible.

We will identify the first layer where representation
diversity collapses.
""")

print("=" * 75)
print("STEP 20 COMPLETE")
print("=" * 75)

STEP 20 — LAYER-BY-LAYER REPRESENTATION DIVERSITY ANALYSIS


RuntimeError: Could not automatically find an ECG tensor with shape (N, 12, length).

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
